# Lab: Building and Validating a Hansch QSAR with numpy
### PHS 312 - Drug Discovery, Design, and Development · Computational laboratory
**Lectures 29-30 (QSAR I and II) · about 75-90 minutes · work in pairs**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Spinner101/phs312-cadd-notebooks/blob/main/labs/PHS312_L29_L30_Lab_QSAR.ipynb)

You will build a classical Hansch equation for ten 3-substituted 4-anilino-6,7-dimethoxyquinazolines (EGFR inhibitors) from tabulated substituent constants, fit it by least squares with numpy, test it with leave-one-out cross-validation, compare it with a model built from whole-molecule RDKit descriptors, and see directly how r² can rise while q² collapses.

**How this lab works.** Run Step 0 (install) and the Set-up cell once. Then work down the steps in order. Code
cells that teach an API are visible: read them before you run them. Cells marked *(hidden)* are plumbing.
Every figure, table, checkpoint and calculation is collected automatically into a lab report that you
build at the end (**.docx** to edit and submit, **.html** with the equations as shown here).

*Protocol structure modelled on the UCL Open Docking Lab Handbook (Ho et al., J. Chem. Educ. 2021,
doi:10.1021/acs.jchemed.1c00289); code and text in this notebook are original or adapted from the
permissively licensed sources listed at the end.*

## Learning objectives

| # | By the end of this lab you should be able to | Bloom | Syllabus level | SILO |
|---|---|---|---|---|
| 1 | Assemble a congeneric QSAR data set with cited activities and check its descriptors for collinearity. | Apply | IV | 3 |
| 2 | Construct a Hansch equation by least squares and interpret the sign and size of each coefficient. | Apply | IV | 3 |
| 3 | Interpret q² from leave-one-out cross-validation and contrast it with r². | Analyze | III | 6 |
| 4 | Critique a QSAR for over-fitting and for predictions outside its applicability domain. | Evaluate | II | 6 |
| 5 | Report a QSAR following the OECD validation principles. | Create | I | 6 |

*Objective ids are L29_L30.n (for example L29_L30.1). Every part below states the objectives it addresses, and every
objective is practised by at least one step, figure or calculation and checked by a checkpoint or a card.
Syllabus levels follow the PHS 312 legend, in which Level I is Creating and Level VI is Remembering.*

---
### Before you start - what this notebook does
* It **downloads software** (from PyPI) and uses only data written into the notebook into this Colab virtual machine.
* Files it writes (structures, results tables, your report) stay in the Colab session until you download them.
* Nothing you type is sent anywhere. Do not type any personal or patient information.
* No drug doses or dosing regimens appear anywhere in this lab; all cases are about molecules and data.

In [ ]:
#@title Step 0 - Install the software (Colab, about 1-2 minutes) { display-mode: "form" }
# Installs only what is missing (safe to re-run). Exact versions are written into the lab report.
# Install recipe: PyPI wheels for RDKit / py3Dmol (all have wheels for Colab's Python 3.12 and 3.13).
import importlib.util, subprocess, sys, os, shutil, hashlib, urllib.request
NEED = ['rdkit', 'pandas']
PIP = {'rdkit': 'rdkit', 'pandas': 'pandas'}
missing = [PIP[m] for m in NEED if importlib.util.find_spec(m) is None]
if missing:
    print("pip install", " ".join(missing), "...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + missing, check=True)
for m in NEED:
    print("%-8s %s" % (m, "ready" if importlib.util.find_spec(m) else "MISSING - re-run this cell"))

In [ ]:
#@title Set-up - run this cell second { display-mode: "form" }
#@markdown Loads the calculation cards, the lab-report collector and this lab's helper code. Nothing is sent anywhere.
# ===== BEGIN calculator and lab-report kit (PHS 312 kit2) =====


# ---------------------------------------------------------------------------
# Calculator engines (no UI). Two families:
#   RPNEngine  - HP-15C / HP 35s RPN behaviour: 4-level stack X Y Z T, LAST X,
#                stack-lift rules, f/g (or left/right) shifted functions.
#   AlgEngine  - algebraic entry line with textbook precedence (TI-30XS,
#                TI-84 Plus CE, Casio fx-991EX/115ES, HP 35s ALG mode).
# Keys are canonical function ids (see FUNCS); the layout tables map each
# physical key + shift state to one of these ids.
# ---------------------------------------------------------------------------
import math

_FACT_MAX = 170


def _fact(x):
    if x < 0 or abs(x - round(x)) > 1e-12:
        return math.gamma(x + 1)                       # HP-15C x! gives Γ(x+1) for non-integers
    if x > _FACT_MAX:
        raise OverflowError("overflow")
    return float(math.factorial(int(round(x))))


def _perm(n, r):
    return math.perm(int(n), int(r))


def _comb(n, r):
    return math.comb(int(n), int(r))


def _hms(h):          # decimal hours -> H.MMSS
    sign = -1 if h < 0 else 1
    h = abs(h)
    H = int(h); m = (h - H) * 60; M = int(m); s = (m - M) * 60
    return sign * (H + M / 100 + s / 10000)


def _hr(hms):         # H.MMSS -> decimal hours
    sign = -1 if hms < 0 else 1
    x = abs(hms)
    H = int(x); MM = int(round((x - H) * 100, 8)); SS = ((x - H) * 100 - MM) * 100
    return sign * (H + MM / 60 + SS / 3600)


class AngleMixin:
    angle = "DEG"

    def to_rad(self, x):
        return {"DEG": math.radians(x), "RAD": x, "GRD": x * math.pi / 200}[self.angle]

    def from_rad(self, r):
        return {"DEG": math.degrees(r), "RAD": r, "GRD": r * 200 / math.pi}[self.angle]


def fmt_number(x, mode="FIX", digits=4, width=10):
    """HP-style display formatting (FIX/SCI/ENG/ALL)."""
    if isinstance(x, str):
        return x
    if x != x:
        return "Error"
    if math.isinf(x):
        return "Overflow"
    if mode == "ALL":
        s = "%.10g" % x
        return s
    if mode == "FIX":
        if x != 0 and (abs(x) >= 1e10 or abs(x) < 10 ** (-digits)):
            return "%.*e" % (digits, x)
        return "%.*f" % (digits, x)
    if mode == "SCI":
        return "%.*e" % (digits, x)
    if mode == "ENG":
        if x == 0:
            return "%.*fe+00" % (digits, 0.0)
        e = int(math.floor(math.log10(abs(x)) / 3) * 3)
        m = x / 10 ** e
        return "%.*fe%+03d" % (max(0, digits - (len(str(int(abs(m)))) - 1)), m, e)
    return str(x)


# =============================================================================
class RPNEngine(AngleMixin):
    """HP-15C-compatible RPN core. Unsupported programming keys report a message."""

    def __init__(self, model="HP-15C"):
        self.model = model
        self.X = self.Y = self.Z = self.T = 0.0
        self.LASTX = 0.0
        self.lift = True
        self.entry = None          # string being typed, or None
        self.exp_entry = None      # exponent digits being typed after EEX
        self.regs = {}
        self.pending = None        # 'STO' | 'RCL' | 'FIX' | 'SCI' | 'ENG' (+ optional op)
        self.pending_op = None
        self.disp_mode, self.disp_digits = "FIX", 4
        self.angle = "DEG"
        self.hyp = None            # None | 'HYP' | 'AHYP'
        self.message = ""
        self.stats_clear()

    # ---------------- stack primitives
    def _lift(self):
        self.T, self.Z, self.Y = self.Z, self.Y, self.X

    def _drop(self):
        self.Y, self.Z = self.Z, self.T

    def _finish_entry(self):
        if self.entry is not None:
            self.X = self._entry_value()
            self.entry = None
            self.exp_entry = None

    def _entry_value(self):
        s = self.entry or "0"
        if s in ("", "-", ".", "-."):
            s = s.replace(".", "") + "0"
        v = float(s if not s.endswith(".") else s[:-1] or "0")
        if self.exp_entry is not None:
            e = self.exp_entry or "0"
            if e in ("-", "+"):
                e = "0"
            v = v * 10 ** int(e)
        return v

    def stats_clear(self):
        self.S = dict(n=0.0, sx=0.0, sx2=0.0, sy=0.0, sy2=0.0, sxy=0.0)

    # ---------------- display
    def display(self):
        if self.message:
            return self.message
        if self.entry is not None:
            s = self.entry if self.entry else "0"
            if self.exp_entry is not None:
                e = self.exp_entry
                sign = "-" if e.startswith("-") else ""
                e = e.lstrip("-")
                s = "%s  %s%s" % (s, sign, e.rjust(2, "0") if e else "00")
            return s
        return fmt_number(self.X, self.disp_mode, self.disp_digits)

    def stack(self):
        x = self._entry_value() if self.entry is not None else self.X
        return dict(X=x, Y=self.Y, Z=self.Z, T=self.T, LASTX=self.LASTX)

    # ---------------- key dispatcher
    def press(self, key):
        self.message = ""
        if self.pending:
            return self._pending_key(key)
        if key in "0123456789" or key == ".":
            return self._digit(key)
        h = getattr(self, "k_" + key, None)
        if h is None:
            self._finish_entry()
            self.message = "%s: not simulated" % key
            return
        h()

    # ---------------- entry keys
    def _digit(self, d):
        if self.entry is None:
            if self.lift:
                self._lift()
            self.entry = ""
            self.exp_entry = None
        if self.exp_entry is not None:
            digits = self.exp_entry.lstrip("-")
            if d == ".":
                return
            digits = (digits + d)[-2:]
            self.exp_entry = ("-" if self.exp_entry.startswith("-") else "") + digits
            return
        if d == "." and "." in self.entry:
            return
        if len(self.entry.replace("-", "").replace(".", "")) >= 10 and d != ".":
            return
        self.entry += d
        self.lift = True

    def k_EEX(self):
        if self.entry is None:
            self._digit("1")
        if self.exp_entry is None:
            self.exp_entry = ""

    def k_CHS(self):
        if self.entry is not None:
            if self.exp_entry is not None:
                self.exp_entry = self.exp_entry[1:] if self.exp_entry.startswith("-") else "-" + self.exp_entry
            else:
                self.entry = self.entry[1:] if self.entry.startswith("-") else "-" + self.entry
        else:
            self.X = -self.X
            self.lift = True

    def k_BSP(self):
        if self.entry is not None:
            if self.exp_entry is not None:
                if self.exp_entry.lstrip("-"):
                    self.exp_entry = self.exp_entry[:-1]
                else:
                    self.exp_entry = None
                return
            self.entry = self.entry[:-1]
            if self.entry in ("", "-"):
                self.entry = None
                self.X = 0.0
                self.lift = False
            return
        self.k_CLX()

    def k_ENTER(self):
        self._finish_entry()
        self._lift()
        self.lift = False

    def k_CLX(self):
        self.entry = None
        self.exp_entry = None
        self.X = 0.0
        self.lift = False

    def k_CLSTK(self):
        self.entry = None
        self.X = self.Y = self.Z = self.T = 0.0
        self.lift = False

    # ---------------- generic helpers
    def _unary(self, f):
        self._finish_entry()
        try:
            r = f(self.X)
        except (ValueError, ZeroDivisionError, OverflowError) as e:
            self.message = "Error 0" if not isinstance(e, OverflowError) else "Overflow"
            return
        if isinstance(r, complex):
            self.message = "Error 0"
            return
        self.LASTX = self.X
        self.X = r
        self.lift = True

    def _binary(self, f):
        self._finish_entry()
        try:
            r = f(self.Y, self.X)
        except (ValueError, ZeroDivisionError, OverflowError) as e:
            self.message = "Error 0" if not isinstance(e, OverflowError) else "Overflow"
            return
        if isinstance(r, complex):
            self.message = "Error 0"
            return
        self.LASTX = self.X
        self.X = r
        self._drop()
        self.lift = True

    def _push(self, v):
        self._finish_entry()
        if self.lift:
            self._lift()
        self.X = v
        self.lift = True

    # ---------------- arithmetic
    def k_ADD(self): self._binary(lambda y, x: y + x)
    def k_SUB(self): self._binary(lambda y, x: y - x)
    def k_MUL(self): self._binary(lambda y, x: y * x)
    def k_DIV(self): self._binary(lambda y, x: y / x)
    def k_YX(self): self._binary(lambda y, x: y ** x)
    def k_XROOTY(self): self._binary(lambda y, x: y ** (1.0 / x))

    def k_PCT(self):        # % : X = Y*X/100, Y kept
        self._finish_entry()
        self.LASTX = self.X
        self.X = self.Y * self.X / 100.0
        self.lift = True

    def k_DPCT(self):       # Δ% : 100 (X - Y) / Y, Y kept
        self._finish_entry()
        try:
            r = 100.0 * (self.X - self.Y) / self.Y
        except ZeroDivisionError:
            self.message = "Error 0"; return
        self.LASTX = self.X
        self.X = r
        self.lift = True

    # ---------------- unary functions
    def k_SQRT(self): self._unary(lambda x: math.sqrt(x))
    def k_SQ(self): self._unary(lambda x: x * x)
    def k_EXP(self): self._unary(math.exp)
    def k_LN(self): self._unary(math.log)
    def k_10X(self): self._unary(lambda x: 10.0 ** x)
    def k_LOG(self): self._unary(math.log10)
    def k_INV(self): self._unary(lambda x: 1.0 / x)
    def k_ABS(self): self._unary(abs)
    def k_INT(self): self._unary(lambda x: float(math.trunc(x)))
    def k_FRAC(self): self._unary(lambda x: x - math.trunc(x))
    def k_FACT(self): self._unary(_fact)
    def k_RND(self): self._unary(lambda x: float(fmt_number(x, self.disp_mode, self.disp_digits).replace("e", "E")) if self.disp_mode != "ALL" else x)
    def k_HMS(self): self._unary(_hms)
    def k_HR(self): self._unary(_hr)
    def k_TORAD(self): self._unary(math.radians)
    def k_TODEG(self): self._unary(math.degrees)
    # HP 35s conversion keys (exact factors)
    def k_TOF(self): self._unary(lambda c: c * 9.0 / 5.0 + 32.0)
    def k_TOC(self): self._unary(lambda f: (f - 32.0) * 5.0 / 9.0)
    def k_TOLB(self): self._unary(lambda kg: kg / 0.45359237)
    def k_TOKG(self): self._unary(lambda lb: lb * 0.45359237)
    def k_TOMILE(self): self._unary(lambda km: km / 1.609344)
    def k_TOKM(self): self._unary(lambda mi: mi * 1.609344)
    def k_TOIN(self): self._unary(lambda cm: cm / 2.54)
    def k_TOCM(self): self._unary(lambda inch: inch * 2.54)
    def k_TOGAL(self): self._unary(lambda l: l / 3.785411784)
    def k_TOL(self): self._unary(lambda gal: gal * 3.785411784)

    def _trig(self, name):
        hyp = self.hyp
        self.hyp = None
        if hyp == "HYP":
            f = {"SIN": math.sinh, "COS": math.cosh, "TAN": math.tanh}[name]
            return self._unary(f)
        if hyp == "AHYP":
            f = {"SIN": math.asinh, "COS": math.acosh, "TAN": math.atanh}[name]
            return self._unary(f)
        f = {"SIN": math.sin, "COS": math.cos, "TAN": math.tan}[name]
        self._unary(lambda x: f(self.to_rad(x)))

    def k_SIN(self): self._trig("SIN")
    def k_COS(self): self._trig("COS")
    def k_TAN(self): self._trig("TAN")
    def k_ASIN(self):
        if self.hyp:
            self.hyp = "AHYP"; return self._trig("SIN")
        self._unary(lambda x: self.from_rad(math.asin(x)))
    def k_ACOS(self):
        if self.hyp:
            self.hyp = "AHYP"; return self._trig("COS")
        self._unary(lambda x: self.from_rad(math.acos(x)))
    def k_ATAN(self):
        if self.hyp:
            self.hyp = "AHYP"; return self._trig("TAN")
        self._unary(lambda x: self.from_rad(math.atan(x)))
    def k_HYP(self): self._finish_entry(); self.hyp = "HYP"
    def k_AHYP(self): self._finish_entry(); self.hyp = "AHYP"

    # ---------------- stack manipulation
    def k_SWAP(self):
        self._finish_entry()
        self.X, self.Y = self.Y, self.X
        self.lift = True

    def k_RDN(self):
        self._finish_entry()
        self.X, self.Y, self.Z, self.T = self.Y, self.Z, self.T, self.X
        self.lift = True

    def k_RUP(self):
        self._finish_entry()
        self.X, self.Y, self.Z, self.T = self.T, self.X, self.Y, self.Z
        self.lift = True

    def k_LASTX(self): self._push(self.LASTX)
    def k_PI(self): self._push(math.pi)
    def k_E(self): self._push(math.e)

    # ---------------- polar / rectangular (15C: X=x? -> f→R uses X=r, Y=θ)
    def k_TOP(self):     # rectangular (x in X, y in Y) -> polar: X = r, Y = θ
        self._finish_entry()
        x, y = self.X, self.Y
        self.LASTX = self.X
        self.X = math.hypot(x, y)
        self.Y = self.from_rad(math.atan2(y, x))
        self.lift = True

    def k_TOR(self):     # polar (r in X, θ in Y) -> rectangular: X = x, Y = y
        self._finish_entry()
        r, th = self.X, self.to_rad(self.Y)
        self.LASTX = self.X
        self.X, self.Y = r * math.cos(th), r * math.sin(th)
        self.lift = True

    def k_PERM(self): self._binary(_perm)
    def k_COMB(self): self._binary(_comb)

    # ---------------- statistics (Σ registers as on HP-15C)
    def k_SIGMAP(self, sign=1):
        self._finish_entry()
        x, y = self.X, self.Y
        S = self.S
        S["n"] += sign; S["sx"] += sign * x; S["sx2"] += sign * x * x
        S["sy"] += sign * y; S["sy2"] += sign * y * y; S["sxy"] += sign * x * y
        self.LASTX = x
        self.X = S["n"]
        self.lift = False

    def k_SIGMAM(self): self.k_SIGMAP(-1)

    def _need(self, k):
        if self.S["n"] < k:
            self.message = "Error 2"
            return False
        return True

    def k_MEAN(self):
        self._finish_entry()
        if not self._need(1): return
        n = self.S["n"]
        if self.lift: self._lift()
        self._lift()
        self.X, self.Y = self.S["sx"] / n, self.S["sy"] / n
        self.lift = True

    def k_SDEV(self):
        self._finish_entry()
        if not self._need(2): return
        n, S = self.S["n"], self.S
        sx = math.sqrt((n * S["sx2"] - S["sx"] ** 2) / (n * (n - 1)))
        sy = math.sqrt(max(0.0, (n * S["sy2"] - S["sy"] ** 2) / (n * (n - 1))))
        if self.lift: self._lift()
        self._lift()
        self.X, self.Y = sx, sy
        self.lift = True

    def _lr(self):
        n, S = self.S["n"], self.S
        M = n * S["sx2"] - S["sx"] ** 2
        N = n * S["sy2"] - S["sy"] ** 2
        P = n * S["sxy"] - S["sx"] * S["sy"]
        slope = P / M
        icpt = (M * S["sy"] - P * S["sx"]) / (n * M)
        r = P / math.sqrt(M * N) if N > 0 else float("nan")
        return slope, icpt, r

    def k_LR(self):         # X = intercept, Y = slope (HP-15C L.R.)
        self._finish_entry()
        if not self._need(2): return
        slope, icpt, _ = self._lr()
        if self.lift: self._lift()
        self._lift()
        self.X, self.Y = icpt, slope
        self.lift = True

    def k_YHATR(self):      # X = ŷ at x in X, Y = r
        self._finish_entry()
        if not self._need(2): return
        slope, icpt, r = self._lr()
        self.LASTX = self.X
        yhat = icpt + slope * self.X
        self._lift()
        self.X, self.Y = yhat, r
        self.lift = True

    def k_CLSIGMA(self):
        self.stats_clear()
        self.k_CLSTK()

    # ---------------- modes and prefixes
    def k_DEG(self): self.angle = "DEG"
    def k_RAD(self): self.angle = "RAD"
    def k_GRD(self): self.angle = "GRD"
    def k_FIX(self): self._finish_entry(); self.pending = "FIX"
    def k_SCI(self): self._finish_entry(); self.pending = "SCI"
    def k_ENG(self): self._finish_entry(); self.pending = "ENG"
    def k_ALL(self): self.disp_mode = "ALL"
    def k_STO(self): self._finish_entry(); self.pending = "STO"; self.pending_op = None
    def k_RCL(self): self._finish_entry(); self.pending = "RCL"; self.pending_op = None

    def _pending_key(self, key):
        p = self.pending
        if p in ("FIX", "SCI", "ENG"):
            if key in "0123456789":
                self.disp_mode, self.disp_digits = p, int(key)
            self.pending = None
            return
        if p in ("STO", "RCL"):
            if key in ("ADD", "SUB", "MUL", "DIV") and self.pending_op is None:
                self.pending_op = key
                return
            if key == ".":
                self.pending_op = (self.pending_op or "") + "."
                return
            if key in "0123456789":
                op = self.pending_op or ""
                dot = "." if op.endswith(".") else ""
                op = op.rstrip(".")
                name = dot + key
                self.pending, self.pending_op = None, None
                if p == "STO":
                    cur = self.regs.get(name, 0.0)
                    v = {"": self.X, "ADD": cur + self.X, "SUB": cur - self.X,
                         "MUL": cur * self.X, "DIV": (cur / self.X) if self.X else float("nan")}[op]
                    self.regs[name] = v
                    self.lift = True
                else:
                    v = self.regs.get(name, 0.0)
                    if op:
                        self.LASTX = self.X
                        self.X = {"ADD": self.X + v, "SUB": self.X - v, "MUL": self.X * v,
                                  "DIV": (self.X / v) if v else float("nan")}[op]
                        self.lift = True
                    else:
                        self._push(v)
                return
            self.pending, self.pending_op = None, None
            return
        self.pending = None


# =============================================================================
class AlgEngine(AngleMixin):
    """Algebraic entry-line calculator with textbook precedence.
    Keys append tokens; '=' (ENTER/EXE) evaluates. ANS holds the last result."""

    FUNCS1 = {"sin(": "sin", "cos(": "cos", "tan(": "tan", "sin⁻¹(": "asin", "cos⁻¹(": "acos", "tan⁻¹(": "atan",
              "sinh(": "sinh", "cosh(": "cosh", "tanh(": "tanh", "sinh⁻¹(": "asinh", "cosh⁻¹(": "acosh",
              "tanh⁻¹(": "atanh", "ln(": "ln", "log(": "log", "√(": "sqrt", "e^(": "exp", "10^(": "pow10",
              "abs(": "abs", "∛(": "cbrt"}

    def __init__(self, model="TI-30XS MultiView", implicit_mult=True, neg_power_first=True):
        self.model = model
        self.line = []            # list of tokens (strings)
        self.ans = 0.0
        self.result = None
        self.history = []
        self.vars = {}
        self.angle = "DEG"
        self.just_evaluated = False
        self.message = ""
        self.implicit = implicit_mult
        self.pending_sto = False
        self.disp_mode, self.disp_digits = "ALL", 10

    # ---------------- display helpers
    def entry_text(self):
        return "".join(self.line)

    def display(self):
        if self.message:
            return self.message
        return fmt_number(self.result, self.disp_mode, self.disp_digits) if self.result is not None else ""

    # ---------------- keys
    def press(self, key):
        self.message = ""
        if self.pending_sto:
            self.pending_sto = False
            if key.startswith("VAR_"):
                v = self._evaluate(store=False)
                if v is not None:
                    self.vars[key[4:]] = v
                    self.result = v
                    self.ans = v
                    self.history.append((list(self.line), v))
                    self.just_evaluated = True
                return
        if key in ("EQ", "ENTER", "EXE"):
            if self.line == [] and self.history:
                self.line = list(self.history[-1][0])
            v = self._evaluate()
            return
        if key in ("CLEAR", "AC"):
            if self.line:
                self.line = []
            else:
                self.result = None
            self.just_evaluated = False
            return
        if key in ("DEL", "BSP"):
            if self.line:
                self.line.pop()
            return
        if key == "STO":
            self.pending_sto = True
            return
        if key == "ENTRY":               # TI-84 2nd ENTRY: recall the last entry line
            if self.history:
                self.line = list(self.history[-1][0]); self.just_evaluated = False
            return
        if key == "DEG": self.angle = "DEG"; return
        if key == "RAD": self.angle = "RAD"; return
        if key == "GRD": self.angle = "GRD"; return
        tok = self._token_for(key)
        if tok is None:
            self.message = "%s: not simulated" % key
            return
        if self.just_evaluated:
            # an operator continues from Ans; anything else starts a new line
            if tok in ("+", "−", "×", "÷", "^", "²", "³", "⁻¹", "!", "E", "%"):
                self.line = ["Ans"]
            else:
                self.line = []
            self.just_evaluated = False
        self.line.append(tok)

    TOKENS = {"ADD": "+", "SUB": "−", "MUL": "×", "DIV": "÷", "NEG": "(−)", "LP": "(", "RP": ")",
              "YX": "^", "SQ": "²", "INV": "⁻¹", "SQRT": "√(", "SIN": "sin(", "COS": "cos(", "TAN": "tan(",
              "ASIN": "sin⁻¹(", "ACOS": "cos⁻¹(", "ATAN": "tan⁻¹(", "SINH": "sinh(", "COSH": "cosh(",
              "TANH": "tanh(", "ASINH": "sinh⁻¹(", "ACOSH": "cosh⁻¹(", "ATANH": "tanh⁻¹(",
              "LN": "ln(", "LOG": "log(", "EXP": "e^(", "10X": "10^(", "PI": "π", "E": "e", "EEX": "E",
              "ANS": "Ans", "FACT": "!", "PERM": " nPr ", "COMB": " nCr ", "ABS": "abs(", "COMMA": ",",
              "PCT": "%", "XROOTY": "ˣ√(", "CBRT": "∛(", "CUBE": "³"}

    def _token_for(self, key):
        if key in "0123456789" or key == ".":
            return key
        if key.startswith("VAR_"):
            return key[4:]
        return self.TOKENS.get(key)

    # ---------------- evaluation: tokenizer + recursive descent
    def _evaluate(self, store=True):
        text = self.entry_text()
        try:
            toks = self._lex(text)
            self._toks, self._i = toks, 0
            v = self._expr()
            if self._i != len(self._toks):
                raise SyntaxError("unexpected %r" % (self._toks[self._i],))
            if isinstance(v, complex) or v != v:
                raise ValueError("domain")
        except ZeroDivisionError:
            self.message = "DIVIDE BY 0"; return None
        except (ValueError, OverflowError):
            self.message = "DOMAIN" if True else ""; return None
        except (SyntaxError, IndexError, KeyError):
            self.message = "SYNTAX ERROR"; return None
        if store:
            self.history.append((list(self.line), v))
            self.ans = v
            self.result = v
            self.just_evaluated = True
        return v

    def _lex(self, s):
        out, i = [], 0
        words = sorted(list(self.FUNCS1) + ["Ans", " nPr ", " nCr ", "(−)", "⁻¹", "ˣ√("], key=len, reverse=True)
        while i < len(s):
            c = s[i]
            if c == " " and not any(s.startswith(w, i) for w in (" nPr ", " nCr ")):
                i += 1; continue
            m = next((w for w in words if s.startswith(w, i)), None)
            if m:
                out.append(("FN", self.FUNCS1[m]) if m in self.FUNCS1 else ("SYM", m.strip()))
                i += len(m); continue
            if c in "0123456789.":
                j = i
                while j < len(s) and s[j] in "0123456789.":
                    j += 1
                num = s[i:j]
                if num.count(".") > 1:
                    raise SyntaxError("number")
                out.append(("NUM", float(num if num != "." else "0")))
                i = j; continue
            if c in "+−×÷^²³!(),%E":
                out.append(("SYM", c)); i += 1; continue
            if c == "-":
                out.append(("SYM", "−")); i += 1; continue
            if c in "πe":
                out.append(("NUM", math.pi if c == "π" else math.e)); i += 1; continue
            if c.isalpha():
                out.append(("VAR", c)); i += 1; continue
            raise SyntaxError(c)
        return out

    def _peek(self):
        return self._toks[self._i] if self._i < len(self._toks) else (None, None)

    def _take(self):
        t = self._toks[self._i]; self._i += 1; return t

    def _expr(self):
        v = self._term()
        while self._peek() in (("SYM", "+"), ("SYM", "−")):
            op = self._take()[1]
            r = self._term()
            v = v + r if op == "+" else v - r
        return v

    def _starts_factor(self):
        k, t = self._peek()
        return k in ("NUM", "FN", "VAR") or (k == "SYM" and t in ("(", "(−)", "√")) or (k == "SYM" and t == "Ans")

    def _term(self):
        v = self._factor()
        while True:
            k, t = self._peek()
            if (k, t) in (("SYM", "×"), ("SYM", "÷")):
                self._take()
                r = self._factor()
                v = v * r if t == "×" else v / r
            elif (k, t) in (("SYM", "nPr"), ("SYM", "nCr")):
                self._take()
                r = self._factor()
                v = float(_perm(v, r) if t == "nPr" else _comb(v, r))
            elif self.implicit and self._starts_factor():
                v = v * self._factor()
            else:
                return v

    def _factor(self):
        k, t = self._peek()
        if (k, t) == ("SYM", "(−)") or (k, t) == ("SYM", "−"):
            self._take()
            return -self._factor()          # negation binds looser than ^ : (−)2² = −4
        return self._power()

    def _power(self):
        base = self._postfix()
        if self._peek() == ("SYM", "^"):
            self._take()
            expo = self._factor()            # right-associative, allows (−) in exponent
            return base ** expo
        return base

    def _postfix(self):
        v = self._primary()
        while True:
            k, t = self._peek()
            if (k, t) == ("SYM", "²"):
                self._take(); v = v * v
            elif (k, t) == ("SYM", "³"):
                self._take(); v = v * v * v
            elif (k, t) == ("SYM", "⁻¹"):
                self._take(); v = 1.0 / v
            elif (k, t) == ("SYM", "!"):
                self._take(); v = _fact(v)
            elif (k, t) == ("SYM", "ˣ√("):
                self._take()
                inner = self._expr()
                if self._peek() == ("SYM", ")"):
                    self._take()
                v = inner ** (1.0 / v)
            elif (k, t) == ("SYM", "%"):
                self._take(); v = v / 100.0
            elif (k, t) == ("SYM", "E"):
                self._take()
                sign = 1
                if self._peek() in (("SYM", "(−)"), ("SYM", "−")):
                    self._take(); sign = -1
                kk, ee = self._take()
                if kk != "NUM":
                    raise SyntaxError("EE")
                v = v * 10 ** (sign * ee)
            else:
                return v

    def _primary(self):
        k, t = self._take()
        if k == "NUM":
            return t
        if k == "VAR":
            return self.vars.get(t, 0.0)
        if (k, t) == ("SYM", "Ans"):
            return self.ans
        if (k, t) == ("SYM", "("):
            v = self._expr()
            if self._peek() == ("SYM", ")"):
                self._take()
            return v                          # closing parenthesis optional at the end, as on TI/Casio
        if (k, t) == ("SYM", "ˣ√("):
            raise SyntaxError("x-root needs index first")
        if k == "FN":
            v = self._expr()
            if self._peek() == ("SYM", ")"):
                self._take()
            return self._apply(t, v)
        raise SyntaxError(t)

    def _apply(self, fn, v):
        r = self.to_rad
        f = {"sin": lambda x: math.sin(r(x)), "cos": lambda x: math.cos(r(x)), "tan": lambda x: math.tan(r(x)),
             "asin": lambda x: self.from_rad(math.asin(x)), "acos": lambda x: self.from_rad(math.acos(x)),
             "atan": lambda x: self.from_rad(math.atan(x)), "sinh": math.sinh, "cosh": math.cosh,
             "tanh": math.tanh, "asinh": math.asinh, "acosh": math.acosh, "atanh": math.atanh,
             "ln": math.log, "log": math.log10, "sqrt": math.sqrt, "exp": math.exp,
             "pow10": lambda x: 10.0 ** x, "abs": abs, "cbrt": lambda x: math.copysign(abs(x) ** (1 / 3), x)}[fn]
        return f(v)


def run_keys(engine, keys):
    """Feed a space-separated key string to an engine; returns the engine."""
    for k in keys.split():
        engine.press(k)
    return engine


# ---------------------------------------------------------------------------
# Step-by-step calculation cards (equation -> case -> worked solution)
#
#   1. Equation: name, colour-coded LaTeX equation, table of variables and
#      constants (meaning, role, units, source).
#   2. Case: pharmacy / pharmaceutical-sciences narrative with each given
#      value highlighted in the same colour as its symbol.
#   3. Solution table: general equation | values from the case | unit check
#      (prefixes expanded, cancellations noted, conversions) | substitution |
#      arithmetic steps | answer | what the answer means.
# The same card renders static Markdown (build time) and an interactive
# solver (Colab), and logs every solve to the lab report.
# ---------------------------------------------------------------------------
import math
import re

PALETTE = ["#1d6fb8", "#c2410c", "#15803d", "#7c3aed", "#b91c1c", "#0e7490", "#a16207", "#be185d"]

# ---- units ---------------------------------------------------------------
PREFIX = {"p": (-12, "pico"), "n": (-9, "nano"), "µ": (-6, "micro"), "u": (-6, "micro"),
          "m": (-3, "milli"), "c": (-2, "centi"), "d": (-1, "deci"), "k": (3, "kilo"), "M": (6, "mega")}
BASE = {"g": "gram", "L": "litre", "mol": "mole", "M": "molar (mol/L)", "Da": "dalton", "s": "second",
        "J": "joule", "cal": "calorie", "eq": "equivalent", "m": "metre", "Pa": "pascal", "U": "enzyme unit",
        "kat": "katal", "Hz": "hertz", "V": "volt", "W": "watt"}
NO_PREFIX = {"min", "h", "day", "week", "yr", "%", "mmHg", "°C", "K", "bp", "kb", "Å", "ppm", "dL"}
SPECIAL = {"dL": (-1, "L"), "mmHg": None, "kb": None}


def parse_unit_token(tok):
    """'mg' -> (exp=-3, base='g', prefix='milli'); 'min' -> (0, 'min', '')."""
    t = tok.strip()
    p = 1
    m = re.match(r"^(.*?)(\^(-?\d+)|²|³)?$", t)
    core = m.group(1)
    if m.group(2):
        p = 2 if m.group(2) == "²" else 3 if m.group(2) == "³" else int(m.group(3))
    if core in NO_PREFIX and core not in SPECIAL:
        return 0, core, "", p
    if core == "dL":
        return -1, "L", "deci", p
    if core in BASE:
        return 0, core, "", p
    if len(core) > 1 and core[0] in PREFIX and core[1:] in BASE:
        e, name = PREFIX[core[0]]
        return e, core[1:], name, p
    return 0, core, "", p


def unit_factor(unit):
    """Return (power of ten, base-unit string) for a compound unit like 'mg/dL'."""
    if not unit:
        return 0, ""
    num, _, den = unit.partition("/")
    e_tot, parts_n, parts_d = 0, [], []
    for tok in re.split(r"[·*\s]+", num.strip()):
        if tok:
            e, b, _, p = parse_unit_token(tok)
            e_tot += e * p; parts_n.append(b + ("^%d" % p if p != 1 else ""))
    for tok in re.split(r"[·*/\s]+", den.strip()):
        if tok:
            e, b, _, p = parse_unit_token(tok)
            e_tot -= e * p; parts_d.append(b + ("^%d" % p if p != 1 else ""))
    base = "·".join(parts_n) + ("/" + "·".join(parts_d) if parts_d else "")
    return e_tot, base


def _plain(x, sig=4):
    """sig significant figures without exponent notation (1e-3 <= |x| < 1e5)."""
    d = int(math.floor(math.log10(abs(x))))
    v = round(x, sig - 1 - d)
    dec = max(0, sig - 1 - d)
    t = ("%.*f" % (dec, v))
    if "." in t:
        t = t.rstrip("0").rstrip(".")
    return t


def sci(x, sig=4):
    """LaTeX scientific notation with sig figs: 1.23 \\times 10^{-4}."""
    if x == 0:
        return "0"
    if isinstance(x, int) and abs(x) < 1e5 or (abs(x) >= 1e-3 and abs(x) < 1e5):
        return _plain(x, sig)
    e = int(math.floor(math.log10(abs(x))))
    m = x / 10 ** e
    return r"%s \times 10^{%d}" % (("%.*g" % (sig, m)), e)


def fmt(x, sig=4):
    if isinstance(x, str):
        return x
    if x == 0:
        return "0"
    if abs(x) >= 1e5 or abs(x) < 1e-3:
        return sci(x, sig)
    return _plain(x, sig)


def utex(u):
    """Unit string to LaTeX (upright)."""
    if not u:
        return ""
    u = u.replace("µ", r"\mu ").replace("·", r"\cdot ").replace("%", r"\%")
    u = re.sub(r"\^(-?\d+)", r"^{\1}", u)
    u = u.strip().replace(" ", r"\ ")
    return r"\,\mathrm{%s}" % u


def expand_prefix(value, unit):
    """LaTeX showing the prefix written out, e.g. 5 mg/dL = 5 × 10⁻³ g / 10⁻¹ L."""
    e, base = unit_factor(unit)
    if e == 0:
        return None
    return r"%s%s = %s \times 10^{%d}\,\mathrm{%s} = %s%s" % (
        fmt(value), utex(unit), fmt(value), e, base.replace("µ", r"\mu "), sci(value * 10 ** e), utex(base))


def _katex_safe(s):
    """KaTeX (Colab) rejects a superscript on a spacing command: \\,^{\\circ} -> \\,{}^{\\circ}."""
    return re.sub(r"(\\[,;:! ]|\\quad|\\qquad)\s*\^", r"\1{}^", s) if isinstance(s, str) else s


# ---- card model -------------------------------------------------------------
CARDS = {}


class Var:
    def __init__(self, key, sym, name, unit="", role="input", source="", value=None, color=None, const=False):
        self.key, self.sym, self.name, self.unit, self.role = key, sym, name, unit, role
        self.source, self.value, self.color, self.const = source, value, color, const

    def tex(self):
        return r"{\color{%s}{%s}}" % (self.color, self.sym)

    def hl(self, value=None, unit=None):
        v = self.value if value is None else value
        u = self.unit if unit is None else unit
        return r"$\mathbf{{\color{%s}{%s%s}}}$" % (self.color, fmt(v), utex(u))


class Card:
    """A worked calculation. `solve(card, vals)` must return a Solution."""

    def __init__(self, id, title, equation, variables, case, solve, constants=(), conversions=(),
                 result_name="", result_unit="", source="", inputs=None):
        self.id, self.title, self.equation = id, title, equation
        self.vars = list(variables) + list(constants)
        for k, v in enumerate(self.vars):
            if v.color is None:
                v.color = "#4b5563" if v.const else PALETTE[k % len(PALETTE)]
        self.V = {v.key: v for v in self.vars}
        self.case, self.solve_fn = case, solve
        self.conversions = list(conversions)
        self.result_name, self.result_unit, self.source = result_name, result_unit, source
        self.inputs = inputs or [v.key for v in self.vars if v.role == "input" and not v.const]
        CARDS[id] = self

    # ---- LaTeX helpers
    def eq_tex(self, template=None):
        t = template or self.equation
        return re.sub(r"\{\{(\w+)\}\}", lambda m: self.V[m.group(1)].tex(), t)

    def defaults(self):
        return {v.key: v.value for v in self.vars if v.value is not None}

    def solve(self, **vals):
        d = self.defaults()
        d.update(vals)
        return self.solve_fn(self, d)

    # ---- static markdown
    def md_equation(self):
        return _katex_safe(self._md_equation())

    def _md_equation(self):
        rows = ["| Symbol | Meaning | Role in the calculation | Units | Source / value |", "|---|---|---|---|---|"]
        for v in self.vars:
            role = "constant" if v.const else v.role
            src = v.source
            if v.const and v.value is not None:
                src = "%s = %s %s; %s" % ("value", fmt(v.value), v.unit, v.source)
            rows.append("| $%s$ | %s | %s | %s | %s |" % (v.tex(), v.name, role, (v.unit or "—").replace("|", "/"), src))
        return ("**1. %s**\n\n$$%s$$\n\n" % (self.title, self.eq_tex())) + "\n".join(rows) + (
            "\n\n*Source of the relationship:* %s" % self.source if self.source else "")

    def md_case(self, vals=None):
        return _katex_safe(self._md_case(vals))

    def _md_case(self, vals=None):
        d = self.defaults()
        if vals:
            d.update(vals)
        txt = re.sub(r"\[\[(\w+)\]\]", lambda m: self.V[m.group(1)].hl(d.get(m.group(1))), self.case)
        return "**2. Practice problem (case)**\n\n" + txt

    def markdown(self, vals=None, collapse=True):
        return _katex_safe(self._markdown(vals, collapse))

    def _markdown(self, vals=None, collapse=True):
        sol = self.solve(**(vals or {}))
        body = sol.markdown()
        s3 = "**3. Step-by-step solution**\n\n" + body
        if collapse:
            s3 = "<details><summary><b>▸ 3. Step-by-step solution (try it first)</b></summary>\n\n" + body + "\n</details>"
        return self.md_equation() + "\n\n" + self.md_case(vals) + "\n\n" + s3


class Solution:
    def __init__(self, card, vals):
        self.card, self.vals, self.rows, self.result, self.unit, self.meaning = card, vals, [], None, "", ""
        self.extra_conversions = []

    # rows are (label, latex-or-markdown, kind)
    def general(self, template=None):
        self.rows.append(("General equation", "$%s$" % self.card.eq_tex(template), "tex"))
        return self

    def given(self):
        parts = []
        for k in self.card.inputs:
            v = self.card.V[k]
            parts.append("$%s = %s%s$" % (v.tex(), fmt(self.vals[k]), utex(v.unit)))
        consts = [v for v in self.card.vars if v.const]
        for v in consts:
            parts.append("$%s = %s%s$ (constant)" % (v.tex(), fmt(self.vals.get(v.key, v.value)), utex(v.unit)))
        self.rows.append(("Values from the case", "; ".join(parts), "md"))
        return self

    def unit_check(self, note=""):
        """Expand every prefix; say whether units cancel or carry through."""
        bits = []
        for v in self.card.vars:
            if v.key not in self.vals or not v.unit:
                continue
            ex = expand_prefix(self.vals[v.key], v.unit)
            if ex:
                bits.append("$%s$" % ex)
        units = sorted({v.unit for v in self.card.vars if v.key in self.vals and v.unit})
        txt = ("Prefixes written out: " + "; ".join(bits) + ". ") if bits else "No unit prefixes to expand. "
        txt += note or ("Units present: " + ", ".join(units) + ". " if units else "")
        self.rows.append(("Unit check", txt, "md"))
        return self

    def sub(self, latex, label="Substitute"):
        self.rows.append((label, "$%s$" % latex, "tex"))
        return self

    def step(self, label, latex):
        self.rows.append((label, "$%s$" % latex, "tex"))
        return self

    def answer(self, value, unit="", meaning="", label=None):
        self.result, self.unit, self.meaning = value, unit, meaning
        self.rows.append((label or "Answer", r"$\boxed{%s = %s%s}$" % (
            self.card.result_name or "x", fmt(value), utex(unit)), "tex"))
        if meaning:
            self.rows.append(("What it means", meaning, "md"))
        return self

    def convert(self, text):
        self.extra_conversions.append(text)
        return self

    def markdown(self):
        return _katex_safe(self._markdown())

    def _markdown(self):
        out = ["| Step | Work |", "|---|---|"]
        for k, (lab, s, kind) in enumerate(self.rows, 1):
            out.append("| %d. %s | %s |" % (k, lab, s.replace("|", r"\vert ")))
        conv = self.card.conversions + self.extra_conversions
        if conv:
            out.append("\n**Common conversions for this calculation:** " + "; ".join(conv))
        return "\n".join(out)

    def plain_lines(self):
        """[(label, latex_or_text)] for the report renderer."""
        return [(lab, _katex_safe(s)) for lab, s, _ in self.rows]


# ---- interactive solver (Colab) ------------------------------------------
def solver_widget(card, report=None, calculator=None):
    """Input boxes for the card's inputs + 'Solve step by step' + optional calculator."""
    try:
        import ipywidgets as W
        from IPython.display import display, Markdown, clear_output
    except Exception:                                     # plain Python (validator)
        sol = card.solve()
        print(sol.markdown())
        if report is not None:
            report.add_calc(card, sol)
        return sol
    boxes = {}
    for k in card.inputs:
        v = card.V[k]
        boxes[k] = W.FloatText(value=float(v.value if v.value is not None else 0.0),
                               description="%s (%s)" % (v.name[:28], v.unit or "–"),
                               style={"description_width": "230px"}, layout=W.Layout(width="420px"))
    btn = W.Button(description="Solve step by step", button_style="success", icon="list-ol",
                   layout=W.Layout(width="200px"))
    cbtn = W.Button(description="Open calculator", icon="calculator", layout=W.Layout(width="200px"))
    out, cout = W.Output(), W.Output()

    def go(_=None):
        with out:
            clear_output()
            try:
                sol = card.solve(**{k: b.value for k, b in boxes.items()})
            except (ValueError, ZeroDivisionError) as e:
                print("Check the inputs:", e)
                return
            display(Markdown(card.md_case({k: b.value for k, b in boxes.items()}) + "\n\n" + sol.markdown()))
            if report is not None:
                report.add_calc(card, sol)

    def calc(_=None):
        with cout:
            clear_output()
            if calculator is not None:
                calculator()
    btn.on_click(go)
    cbtn.on_click(calc)
    display(W.VBox(list(boxes.values()) + [W.HBox([btn, cbtn]), out, cout]))
    go()


# ---------------------------------------------------------------------------
# Lab report collector: every figure shown, every step-by-step calculation
# and every self-check answer is logged in this browser session; at the end
# the student downloads a formatted report (.html and .docx). Nothing is
# sent anywhere; files are created only when the student asks.
# ---------------------------------------------------------------------------
import io, re, base64, zipfile, datetime, html as _html

_LATEX_FIX = [(r"\\textcolor\{[^}]*\}", ""), (r"\\color\{[^}]*\}", ""),
              (r"\\text\{", r"\\mathrm{"), (r"\\boxed\{", "{"), (r"\\quad", r"\ \ "), (r"\\%", r"\%"),
              (r"\\vert", "|"), (r"\\mathbf\{", r"\\mathbf{")]


def latex_png(tex, fontsize=12, dpi=170):
    """Render one LaTeX line to PNG bytes with matplotlib mathtext (colours stripped)."""
    import matplotlib
    import matplotlib.pyplot as plt
    t = tex
    for a, b in _LATEX_FIX:
        t = re.sub(a, b, t)
    t = t.replace(r"\,", r"\ ")
    fig = plt.figure(figsize=(0.01, 0.01))
    try:
        fig.text(0, 0, "$%s$" % t, fontsize=fontsize)
        buf = io.BytesIO()
        fig.savefig(buf, format="png", dpi=dpi, bbox_inches="tight", pad_inches=0.04, transparent=False,
                    facecolor="white")
        return buf.getvalue()
    except Exception:
        return None
    finally:
        plt.close(fig)


def _split_math(s):
    """Split '... $a$ ... $b$' into [('text', ..), ('math', ..)]."""
    out, pos = [], 0
    for m in re.finditer(r"\$([^$]+)\$", s):
        if m.start() > pos:
            out.append(("text", s[pos:m.start()]))
        out.append(("math", m.group(1)))
        pos = m.end()
    if pos < len(s):
        out.append(("text", s[pos:]))
    return out


class LabReport:
    def __init__(self, course, session, title):
        self.course, self.session, self.title = course, session, title
        self.entries = []          # dicts: kind, section, title, ...
        self.section = ""
        self.student = ""
        self.started = datetime.datetime.now()

    # ---- collection
    def set_section(self, label):
        self.section = label

    def add_figure(self, fig, title=None):
        buf = io.BytesIO()
        try:
            fig.savefig(buf, format="png", dpi=110, bbox_inches="tight", facecolor="white")
        except Exception:
            return
        t = title
        if not t:
            st = getattr(fig, "_suptitle", None)
            t = st.get_text() if st is not None else ""
            if not t:
                for ax in fig.axes:
                    tt = ax.get_title() or ax.get_title(loc="left") or ax.get_title(loc="right")
                    if tt:
                        t = tt; break
            if not t and fig.texts:
                t = fig.texts[0].get_text()
        self.entries.append(dict(kind="figure", section=self.section, title=t or "Figure", png=buf.getvalue()))

    def add_calc(self, card, sol):
        self.entries.append(dict(kind="calc", section=self.section, title=card.title, card_id=card.id,
                                 case=card.md_case(sol.vals), rows=sol.plain_lines(),
                                 result=sol.result, unit=sol.unit, meaning=sol.meaning))

    def add_answer(self, qid, stem, choice, correct, feedback=""):
        self.entries.append(dict(kind="answer", section=self.section, qid=qid, stem=stem, choice=choice,
                                 correct=correct, feedback=feedback))

    def add_note(self, title, text):
        self.entries.append(dict(kind="note", section=self.section, title=title, text=text))

    def counts(self):
        c = {}
        for e in self.entries:
            c[e["kind"]] = c.get(e["kind"], 0) + 1
        return c

    # ---- HTML
    def to_html(self, summary_md=""):
        css = ("body{font-family:Georgia,serif;max-width:860px;margin:2em auto;color:#1f2430;line-height:1.5}"
               "h1{font-size:1.6em}h2{border-bottom:2px solid #1d6fb8;padding-bottom:4px;margin-top:1.6em}"
               "table{border-collapse:collapse;width:100%;margin:.6em 0}td,th{border:1px solid #cbd5e1;padding:6px;"
               "vertical-align:top;font-size:.95em}th{background:#eef4fb}.fig{text-align:center;margin:1em 0}"
               ".fig img{max-width:100%}.cap{font-size:.9em;color:#4b5563}.ok{color:#15803d}.no{color:#b91c1c}"
               ".meta{color:#4b5563}")
        h = ['<!doctype html><html><head><meta charset="utf-8"><title>%s</title><style>%s</style>'
             '<script src="https://cdnjs.cloudflare.com/ajax/libs/mathjax/2.7.9/MathJax.js?config=TeX-AMS_HTML"></script>'
             '<script type="text/x-mathjax-config">MathJax.Hub.Config({tex2jax:{inlineMath:[["$","$"]]}});</script>'
             '</head><body>' % (_html.escape(self.title), css)]
        h.append("<h1>%s · %s</h1><p class='meta'>%s<br>Student: %s<br>Generated %s</p>" % (
            _html.escape(self.course), _html.escape(self.title), _html.escape(self.session),
            _html.escape(self.student or "________________"), datetime.datetime.now().strftime("%Y-%m-%d %H:%M")))
        sec = None
        nfig = ncalc = 0
        for e in self.entries:
            if e["section"] != sec:
                sec = e["section"]
                h.append("<h2>%s</h2>" % _html.escape(sec or "Work"))
            if e["kind"] == "figure":
                nfig += 1
                h.append("<div class='fig'><img src='data:image/png;base64,%s'><div class='cap'>Figure %d. %s</div></div>"
                         % (base64.b64encode(e["png"]).decode(), nfig, _html.escape(e["title"])))
            elif e["kind"] == "calc":
                ncalc += 1
                h.append("<h3>Calculation %d. %s</h3><p>%s</p><table><tr><th>Step</th><th>Work</th></tr>" % (
                    ncalc, _html.escape(e["title"]), e["case"].replace("**2. Practice problem (case)**", "").strip()))
                for k, (lab, s) in enumerate(e["rows"], 1):
                    h.append("<tr><td>%d. %s</td><td>%s</td></tr>" % (k, _html.escape(lab), s))
                h.append("</table>")
            elif e["kind"] == "answer":
                h.append("<p><b>%s.</b> %s<br>Answer: %s <span class='%s'>(%s)</span></p>" % (
                    _html.escape(e["qid"]), _html.escape(e["stem"]), _html.escape(e["choice"]),
                    "ok" if e["correct"] else "no", "correct" if e["correct"] else "review"))
            elif e["kind"] == "note":
                h.append("<h3>%s</h3><p>%s</p>" % (_html.escape(e["title"]), _html.escape(e["text"])))
        if summary_md:
            h.append("<h2>Summary</h2><pre>%s</pre>" % _html.escape(summary_md))
        h.append("</body></html>")
        return "\n".join(h)

    # ---- DOCX (minimal WordprocessingML writer; equations as images)
    def to_docx(self):
        D = _Docx()
        D.heading("%s · %s" % (self.course, self.title), 1)
        D.para("%s    Student: %s    Generated %s" % (self.session, self.student or "________________",
                                                      datetime.datetime.now().strftime("%Y-%m-%d %H:%M")), size=18)
        sec = None
        nfig = ncalc = 0
        for e in self.entries:
            if e["section"] != sec:
                sec = e["section"]
                D.heading(sec or "Work", 2)
            if e["kind"] == "figure":
                nfig += 1
                D.image(e["png"], width_in=6.0)
                D.para("Figure %d. %s" % (nfig, e["title"]), size=18, italic=True)
            elif e["kind"] == "calc":
                ncalc += 1
                D.heading("Calculation %d. %s" % (ncalc, e["title"]), 3)
                D.rich(e["case"].replace("**2. Practice problem (case)**", "").strip())
                rows = []
                for k, (lab, s) in enumerate(e["rows"], 1):
                    rows.append(("%d. %s" % (k, lab), s))
                D.table(rows)
            elif e["kind"] == "answer":
                D.para("%s. %s — answer %s (%s)" % (e["qid"], e["stem"], e["choice"],
                                                     "correct" if e["correct"] else "review"), size=20)
            elif e["kind"] == "note":
                D.heading(e["title"], 3)
                D.para(e["text"])
        return D.build()

    # ---- delivery
    def download(self, basename=None):
        base = basename or ("%s_%s_lab_report" % (self.course.replace(" ", ""), self.session.split()[0]))
        html_bytes = self.to_html().encode("utf-8")
        docx_bytes = self.to_docx()
        with open(base + ".html", "wb") as f:
            f.write(html_bytes)
        with open(base + ".docx", "wb") as f:
            f.write(docx_bytes)
        try:
            from google.colab import files
            files.download(base + ".docx")
            files.download(base + ".html")
            return "Downloading %s.docx and %s.html" % (base, base)
        except Exception:
            return "Saved %s.docx and %s.html in the notebook's folder" % (base, base)


# ---------------------------------------------------------------------------
_W = 'xmlns:w="http://schemas.openxmlformats.org/wordprocessingml/2006/main" ' \
     'xmlns:r="http://schemas.openxmlformats.org/officeDocument/2006/relationships" ' \
     'xmlns:wp="http://schemas.openxmlformats.org/drawingml/2006/wordprocessingDrawing" ' \
     'xmlns:a="http://schemas.openxmlformats.org/drawingml/2006/main" ' \
     'xmlns:pic="http://schemas.openxmlformats.org/drawingml/2006/picture"'


def _x(s):
    return _html.escape(s, quote=False)


class _Docx:
    def __init__(self):
        self.body, self.media = [], []

    def _run(self, text, bold=False, italic=False, size=None, color=None):
        rpr = ""
        if bold: rpr += "<w:b/>"
        if italic: rpr += "<w:i/>"
        if color: rpr += '<w:color w:val="%s"/>' % color.lstrip("#")
        if size: rpr += '<w:sz w:val="%d"/>' % size
        return '<w:r>%s<w:t xml:space="preserve">%s</w:t></w:r>' % ("<w:rPr>%s</w:rPr>" % rpr if rpr else "", _x(text))

    def heading(self, text, level=1):
        size = {1: 36, 2: 28, 3: 24}.get(level, 22)
        self.body.append('<w:p><w:pPr><w:spacing w:before="240" w:after="80"/></w:pPr>%s</w:p>'
                         % self._run(text, bold=True, size=size, color="1d3557" if level < 3 else None))

    def para(self, text, size=22, italic=False):
        self.body.append("<w:p>%s</w:p>" % self._run(text, size=size, italic=italic))

    def _img_run(self, png, width_in):
        from PIL import Image
        im = Image.open(io.BytesIO(png))
        w, h = im.size
        cx = int(width_in * 914400)
        cy = int(cx * h / w)
        self.media.append(png)
        rid = "rIdImg%d" % len(self.media)
        n = len(self.media)
        return ('<w:r><w:drawing><wp:inline><wp:extent cx="%d" cy="%d"/><wp:docPr id="%d" name="img%d"/>'
                '<a:graphic><a:graphicData uri="http://schemas.openxmlformats.org/drawingml/2006/picture">'
                '<pic:pic><pic:nvPicPr><pic:cNvPr id="%d" name="img%d.png"/><pic:cNvPicPr/></pic:nvPicPr>'
                '<pic:blipFill><a:blip r:embed="%s"/><a:stretch><a:fillRect/></a:stretch></pic:blipFill>'
                '<pic:spPr><a:xfrm><a:off x="0" y="0"/><a:ext cx="%d" cy="%d"/></a:xfrm>'
                '<a:prstGeom prst="rect"><a:avLst/></a:prstGeom></pic:spPr></pic:pic></a:graphicData></a:graphic>'
                '</wp:inline></w:drawing></w:r>' % (cx, cy, n, n, n, n, rid, cx, cy))

    def image(self, png, width_in=6.0):
        self.body.append('<w:p><w:pPr><w:jc w:val="center"/></w:pPr>%s</w:p>' % self._img_run(png, width_in))

    def _rich_runs(self, s, size=22):
        runs = []
        for kind, part in _split_math(s):
            if kind == "text":
                txt = re.sub(r"\*\*(.*?)\*\*", r"\1", part)
                runs.append(self._run(txt, size=size))
            else:
                png = latex_png(part, fontsize=11)
                if png:
                    from PIL import Image
                    w, h = Image.open(io.BytesIO(png)).size
                    runs.append(self._img_run(png, min(6.0, w / 170.0)))
                else:
                    runs.append(self._run(part, size=size, italic=True))
        return "".join(runs)

    def rich(self, s):
        self.body.append("<w:p>%s</w:p>" % self._rich_runs(s))

    def table(self, rows):
        trs = []
        for a, b in rows:
            trs.append('<w:tr><w:tc><w:tcPr><w:tcW w:w="2400" w:type="dxa"/></w:tcPr><w:p>%s</w:p></w:tc>'
                       '<w:tc><w:tcPr><w:tcW w:w="7000" w:type="dxa"/></w:tcPr><w:p>%s</w:p></w:tc></w:tr>'
                       % (self._run(a, bold=True, size=20), self._rich_runs(b, size=20)))
        borders = "".join('<w:%s w:val="single" w:sz="4" w:space="0" w:color="CBD5E1"/>' % e
                          for e in ("top", "left", "bottom", "right", "insideH", "insideV"))
        self.body.append('<w:tbl><w:tblPr><w:tblW w:w="9400" w:type="dxa"/><w:tblBorders>%s</w:tblBorders>'
                         '</w:tblPr><w:tblGrid><w:gridCol w:w="2400"/><w:gridCol w:w="7000"/></w:tblGrid>%s</w:tbl>'
                         '<w:p/>' % (borders, "".join(trs)))

    def build(self):
        doc = ('<?xml version="1.0" encoding="UTF-8" standalone="yes"?><w:document %s><w:body>%s'
               '<w:sectPr><w:pgSz w:w="12240" w:h="15840"/><w:pgMar w:top="1080" w:right="1080" w:bottom="1080" '
               'w:left="1080" w:header="720" w:footer="720" w:gutter="0"/></w:sectPr></w:body></w:document>'
               % (_W, "".join(self.body)))
        rels = ['<Relationship Id="rIdImg%d" Type="http://schemas.openxmlformats.org/officeDocument/2006/relationships/image" '
                'Target="media/image%d.png"/>' % (k, k) for k in range(1, len(self.media) + 1)]
        buf = io.BytesIO()
        with zipfile.ZipFile(buf, "w", zipfile.ZIP_DEFLATED) as z:
            z.writestr("[Content_Types].xml",
                       '<?xml version="1.0" encoding="UTF-8" standalone="yes"?>'
                       '<Types xmlns="http://schemas.openxmlformats.org/package/2006/content-types">'
                       '<Default Extension="rels" ContentType="application/vnd.openxmlformats-package.relationships+xml"/>'
                       '<Default Extension="xml" ContentType="application/xml"/>'
                       '<Default Extension="png" ContentType="image/png"/>'
                       '<Override PartName="/word/document.xml" '
                       'ContentType="application/vnd.openxmlformats-officedocument.wordprocessingml.document.main+xml"/>'
                       '</Types>')
            z.writestr("_rels/.rels",
                       '<?xml version="1.0" encoding="UTF-8" standalone="yes"?>'
                       '<Relationships xmlns="http://schemas.openxmlformats.org/package/2006/relationships">'
                       '<Relationship Id="rId1" Type="http://schemas.openxmlformats.org/officeDocument/2006/relationships/officeDocument" '
                       'Target="word/document.xml"/></Relationships>')
            z.writestr("word/_rels/document.xml.rels",
                       '<?xml version="1.0" encoding="UTF-8" standalone="yes"?>'
                       '<Relationships xmlns="http://schemas.openxmlformats.org/package/2006/relationships">%s'
                       '</Relationships>' % "".join(rels))
            z.writestr("word/document.xml", doc)
            for k, png in enumerate(self.media, 1):
                z.writestr("word/media/image%d.png" % k, png)
        return buf.getvalue()


# ---------------------------------------------------------------------------
# Pathway figure kit (BioRender-inspired, matplotlib only).
# Muted palette, rounded compartments, lipid-bilayer membranes, short labels,
# arrows = conversion/activation, T-bars = inhibition, dashed = indirect,
# hexagon badges = drugs. Coordinates are in a 0-100 x 0-60 canvas.
# ---------------------------------------------------------------------------
import math
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, Circle, RegularPolygon, FancyArrowPatch, Ellipse, Rectangle
from matplotlib.lines import Line2D

PW = dict(teal="#2a9d8f", blue="#3a6ea5", coral="#e76f51", gold="#e9b44c", green="#5a9e6f",
          purple="#7b6fb0", grey="#6b7280", ink="#1f2937", paper="#ffffff",
          cyto="#f3f8fb", mito="#fdf3ea", nucleus="#f1eef8", lumen="#eef7f1", er="#fbf1f4",
          blood="#fdeeee", extracell="#f7f7f2")
KIND = {  # node fill, edge, text colour, shape
    "metabolite": (PW["paper"], PW["blue"], PW["ink"], "round"),
    "enzyme": ("#e3f1ef", PW["teal"], PW["ink"], "pill"),
    "transporter": ("#fbe9e3", PW["coral"], PW["ink"], "pill"),
    "receptor": ("#efeaf7", PW["purple"], PW["ink"], "pill"),
    "gene": ("#fdf5e1", PW["gold"], PW["ink"], "square"),
    "cofactor": (PW["paper"], PW["grey"], PW["grey"], "round"),
    "hormone": ("#e8f3ea", PW["green"], PW["ink"], "round"),
}


class Pathway:
    def __init__(self, title="", w=10.5, h=6.0, xlim=(0, 100), ylim=(0, 60)):
        self.fig, self.ax = plt.subplots(figsize=(w, h))
        self.ax.set_xlim(*xlim); self.ax.set_ylim(*ylim); self.ax.set_aspect("equal"); self.ax.axis("off")
        self.fig.patch.set_facecolor("white")
        if title:
            self.ax.set_title(title, fontsize=13, color=PW["ink"], loc="left", fontweight="bold", pad=10)
        self.nodes = {}
        self._legend = set()

    # ---- containers
    def compartment(self, x, y, w, h, label="", kind="cyto", membrane=True, label_pos="tl"):
        fill = PW.get(kind, kind)
        self.ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0,rounding_size=%.1f" % min(4, h / 4),
                                         fc=fill, ec="#c9d3dd" if not membrane else "#d9a441", lw=0.8 if not membrane else 3.2,
                                         zorder=0))
        if membrane:   # inner line gives a bilayer look
            self.ax.add_patch(FancyBboxPatch((x + 0.6, y + 0.6), w - 1.2, h - 1.2,
                                             boxstyle="round,pad=0,rounding_size=%.1f" % max(0.5, min(4, h / 4) - 0.6),
                                             fc="none", ec="#f2d48f", lw=1.4, zorder=0))
        if label:
            tx, ha = (x + 1.6, "left") if "l" in label_pos else (x + w - 1.6, "right")
            ty, va = (y + h - 1.4, "top") if "t" in label_pos else (y + 1.4, "bottom")
            self.ax.text(tx, ty, label, ha=ha, va=va, fontsize=9, color=PW["grey"], style="italic", zorder=1)

    def membrane(self, x0, y0, x1, y1, label=""):
        """Straight lipid bilayer (two rows of heads) between two points."""
        n = max(4, int(math.hypot(x1 - x0, y1 - y0) / 1.3))
        dx, dy = (x1 - x0), (y1 - y0)
        L = math.hypot(dx, dy); nx, ny = -dy / L, dx / L
        for k in range(n + 1):
            px, py = x0 + dx * k / n, y0 + dy * k / n
            for s in (-1, 1):
                self.ax.add_patch(Circle((px + s * 0.9 * nx, py + s * 0.9 * ny), 0.42, fc="#e9b44c", ec="#c9902a", lw=0.4, zorder=1))
            self.ax.plot([px - 0.6 * nx, px + 0.6 * nx], [py - 0.6 * ny, py + 0.6 * ny], color="#d8c08a", lw=0.6, zorder=0.9)
        if label:
            self.ax.text(x1, y1 + 1.2, label, fontsize=8, color=PW["grey"], ha="right", style="italic")

    # ---- nodes
    def node(self, key, x, y, label, kind="metabolite", w=None, fs=9.5):
        fc, ec, tc, shape = KIND[kind]
        w = w or max(7.0, 1.15 * len(max(label.split("\n"), key=len)) + 2.6)
        h = 3.6 + 2.2 * label.count("\n")
        rs = {"round": h / 2, "pill": h / 2, "square": 0.6}[shape]
        self.ax.add_patch(FancyBboxPatch((x - w / 2, y - h / 2), w, h, boxstyle="round,pad=0,rounding_size=%.2f" % rs,
                                         fc=fc, ec=ec, lw=1.6 if kind != "metabolite" else 1.2, zorder=3))
        self.ax.text(x, y, label, ha="center", va="center", fontsize=fs, color=tc, zorder=4,
                     fontweight="bold" if kind in ("enzyme", "transporter", "receptor") else "normal",
                     style="italic" if kind == "gene" else "normal")
        self.nodes[key] = (x, y, w, h)
        self._legend.add(kind)
        return key

    def drug(self, key, x, y, label, r=2.6):
        self.ax.add_patch(RegularPolygon((x, y), 6, radius=r, orientation=math.pi / 6, fc=PW["coral"], ec="#a8432b",
                                         lw=1.2, zorder=5))
        self.ax.text(x, y - r - 0.9, label, ha="center", va="top", fontsize=8.6, color="#a8432b", fontweight="bold", zorder=5)
        self.ax.text(x, y, "Rx", ha="center", va="center", fontsize=7.5, color="white", fontweight="bold", zorder=6)
        self.nodes[key] = (x, y, 2 * r, 2 * r)
        self._legend.add("drug")
        return key

    def _edge_pt(self, key, toward):
        x, y, w, h = self.nodes[key]
        tx, ty = toward
        dx, dy = tx - x, ty - y
        if dx == 0 and dy == 0:
            return x, y
        sx = (w / 2 + 0.5) / abs(dx) if dx else 1e9
        sy = (h / 2 + 0.5) / abs(dy) if dy else 1e9
        s = min(sx, sy)
        return x + dx * s, y + dy * s

    # ---- edges
    def arrow(self, a, b, kind="convert", label="", rad=0.0, color=None, lw=1.7, label_side=1):
        pa = self.nodes[a][:2] if isinstance(a, str) else a
        pb = self.nodes[b][:2] if isinstance(b, str) else b
        p0 = self._edge_pt(a, pb) if isinstance(a, str) else pa
        p1 = self._edge_pt(b, pa) if isinstance(b, str) else pb
        col = color or {"convert": PW["ink"], "activate": PW["green"], "inhibit": PW["coral"],
                        "indirect": PW["grey"], "transport": PW["blue"]}[kind]
        ls = "--" if kind == "indirect" else "-"
        style = "-[" if kind == "inhibit" else "-|>"
        self.ax.add_patch(FancyArrowPatch(p0, p1, arrowstyle=style if kind != "inhibit" else "-",
                                          mutation_scale=13, color=col, lw=lw, ls=ls,
                                          connectionstyle="arc3,rad=%.2f" % rad, zorder=2, shrinkA=0, shrinkB=0))
        if kind == "inhibit":   # T-bar
            dx, dy = p1[0] - p0[0], p1[1] - p0[1]
            L = math.hypot(dx, dy) or 1
            nx, ny = -dy / L * 1.5, dx / L * 1.5
            self.ax.plot([p1[0] - nx, p1[0] + nx], [p1[1] - ny, p1[1] + ny], color=col, lw=2.6, zorder=2,
                         solid_capstyle="round")
        if label:
            mx, my = (p0[0] + p1[0]) / 2, (p0[1] + p1[1]) / 2
            dx, dy = p1[0] - p0[0], p1[1] - p0[1]
            L = math.hypot(dx, dy) or 1
            ox, oy = -dy / L * 1.6 * label_side, dx / L * 1.6 * label_side
            if rad:
                ox += -dy * rad * 0.5; oy += dx * rad * 0.5
            self.ax.text(mx + ox, my + oy, label, fontsize=8.2, color=col, ha="center", va="center", zorder=6,
                         bbox=dict(boxstyle="round,pad=0.15", fc="white", ec="none", alpha=0.85))
        self._legend.add(kind)

    def catalyze(self, enzyme, a, b, label_side=1):
        """Metabolite a -> b with enzyme drawn beside the arrow and a short dashed tie to its midpoint."""
        self.arrow(a, b, "convert")
        xa, ya = self.nodes[a][:2]; xb, yb = self.nodes[b][:2]
        xe, ye = self.nodes[enzyme][:2]
        self.ax.add_patch(FancyArrowPatch((xe, ye), ((xa + xb) / 2, (ya + yb) / 2), arrowstyle="-", color=PW["teal"],
                                          lw=1.0, ls=(0, (2, 2)), zorder=1, shrinkA=10, shrinkB=2))

    def note(self, x, y, text, fs=8.4, color=None, ha="left"):
        import textwrap
        xmax = self.ax.get_xlim()[1]
        room = (xmax - x) if ha == "left" else (x if ha == "right" else 2 * min(x, xmax - x))
        width = max(12, int(room * 1.55 * 8.4 / fs))
        text = "\n".join(textwrap.fill(line, width) for line in text.split("\n"))
        self.ax.text(x, y, text, fontsize=fs, color=color or PW["grey"], ha=ha, va="top", zorder=6)

    def legend(self, loc="below"):
        items = []
        names = {"convert": ("conversion", PW["ink"], "-"), "activate": ("activates", PW["green"], "-"),
                 "inhibit": ("inhibits (T-bar)", PW["coral"], "-"), "indirect": ("indirect / multistep", PW["grey"], "--"),
                 "transport": ("transport", PW["blue"], "-")}
        for k in ("convert", "activate", "inhibit", "indirect", "transport"):
            if k in self._legend:
                n, c, ls = names[k]
                items.append(Line2D([0], [0], color=c, lw=2, ls=ls, label=n))
        for k in ("enzyme", "transporter", "receptor", "gene", "hormone"):
            if k in self._legend:
                fc, ec, _, _ = KIND[k]
                items.append(Rectangle((0, 0), 1, 1, fc=fc, ec=ec, lw=1.4, label=k))
        if "drug" in self._legend:
            items.append(Line2D([0], [0], marker="h", color="w", markerfacecolor=PW["coral"], markersize=11, label="drug"))
        if items and loc == "below":
            self.ax.legend(handles=items, loc="upper center", bbox_to_anchor=(0.5, -0.01), fontsize=7.8,
                           frameon=False, ncol=min(6, len(items)))
        elif items:
            self.ax.legend(handles=items, loc=loc, fontsize=7.8, frameon=True, framealpha=0.95, edgecolor="#d1d5db",
                           ncol=2 if len(items) > 4 else 1)

    def show(self, report=None):
        self.fig.tight_layout()
        plt.show()
        return self.fig


def demo_orlistat():
    """Example: dietary TAG digestion, absorption and the orlistat block."""
    P = Pathway("Fat digestion in the small-intestine lumen and the orlistat block", 10.5, 6.2)
    P.compartment(2, 30, 62, 27, "intestinal lumen", "lumen", membrane=False)
    P.compartment(66, 4, 32, 53, "enterocyte", "cyto")
    P.compartment(2, 3, 62, 24, "pancreas → duodenum", "extracell", membrane=False, label_pos="bl")
    P.node("tag", 10, 48, "TAG", "metabolite")
    P.node("bile", 10, 38, "bile salts", "cofactor")
    P.node("mic", 32, 48, "emulsion\ndroplet", "metabolite")
    P.node("mag", 48, 48, "2-MAG +\n2 FFA", "metabolite")
    P.node("pl", 32, 14, "pancreatic\nlipase", "enzyme")
    P.node("col", 50, 14, "colipase", "enzyme")
    P.node("cd36", 66, 36, "FATP4 /\nCD36", "transporter")
    P.node("rtag", 82, 34, "TAG", "metabolite")
    P.node("cm", 82, 16, "chylomicron", "metabolite")
    P.drug("orl", 18, 22, "orlistat")
    P.arrow("tag", "mic", "convert")
    P.arrow("bile", "mic", "activate", "emulsifies", rad=-0.2, label_side=-1)
    P.arrow("mic", "mag", "convert", "lipolysis")
    P.arrow("pl", "mic", "activate", rad=0.15)
    P.arrow("col", "pl", "activate", "anchors", label_side=-1)
    P.arrow("orl", "pl", "inhibit", "covalent Ser", label_side=1)
    P.arrow("mag", "cd36", "transport")
    P.arrow("cd36", "rtag", "indirect", "re-esterified", label_side=-1)
    P.arrow("rtag", "cm", "convert", "+ apoB-48")
    P.note(3, 2.2, "Unabsorbed TAG passes to stool (steatorrhea): measured as fecal fat.", fs=8)
    P.legend()
    return P.show()


import json as _json
CALC_LAYOUTS = _json.loads('{"HP-15C": {"sources": ["https://finseth.com/hpdata/hp15c.php", "https://www.manualslib.com/manual/558055/Hp-Hp-15c.html", "https://manualshive.com/en/manual/hp/15c/156479?page=10"], "shift_keys": {"f": {"color": "#E3A63A", "label_position": "above the key, printed on the faceplate (gold)"}, "g": {"color": "#5AA7D6", "label_position": "on the slanted front face of the key, below the main legend (blue)"}}, "grid": {"rows": 4, "cols": 10}, "keys": [{"row": 0, "col": 0, "rowspan": 1, "colspan": 1, "main": "√x", "shift1": "A", "shift2": "x²", "alpha": "", "key_color": "#1E1E1E", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"row": 0, "col": 1, "rowspan": 1, "colspan": 1, "main": "eˣ", "shift1": "B", "shift2": "LN", "alpha": "", "key_color": "#1E1E1E", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"row": 0, "col": 2, "rowspan": 1, "colspan": 1, "main": "10ˣ", "shift1": "C", "shift2": "LOG", "alpha": "", "key_color": "#1E1E1E", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"row": 0, "col": 3, "rowspan": 1, "colspan": 1, "main": "yˣ", "shift1": "D", "shift2": "%", "alpha": "", "key_color": "#1E1E1E", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"row": 0, "col": 4, "rowspan": 1, "colspan": 1, "main": "1/x", "shift1": "E", "shift2": "Δ%", "alpha": "", "key_color": "#1E1E1E", "text_color": "#FFFFFF", "uncertain": false, "note": "Blue legend is Δ% (percent change); Finseth renders it \'%CHG\'."}, {"row": 0, "col": 5, "rowspan": 1, "colspan": 1, "main": "CHS", "shift1": "MATRIX", "shift2": "ABS", "alpha": "", "key_color": "#1E1E1E", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"row": 0, "col": 6, "rowspan": 1, "colspan": 1, "main": "7", "shift1": "FIX", "shift2": "DEG", "alpha": "", "key_color": "#1E1E1E", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"row": 0, "col": 7, "rowspan": 1, "colspan": 1, "main": "8", "shift1": "SCI", "shift2": "RAD", "alpha": "", "key_color": "#1E1E1E", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"row": 0, "col": 8, "rowspan": 1, "colspan": 1, "main": "9", "shift1": "ENG", "shift2": "GRD", "alpha": "", "key_color": "#1E1E1E", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"row": 0, "col": 9, "rowspan": 1, "colspan": 1, "main": "÷", "shift1": "SOLVE", "shift2": "x≤y", "alpha": "", "key_color": "#1E1E1E", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"row": 1, "col": 0, "rowspan": 1, "colspan": 1, "main": "SST", "shift1": "LBL", "shift2": "BST", "alpha": "", "key_color": "#1E1E1E", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"row": 1, "col": 1, "rowspan": 1, "colspan": 1, "main": "GTO", "shift1": "HYP", "shift2": "HYP⁻¹", "alpha": "", "key_color": "#1E1E1E", "text_color": "#FFFFFF", "uncertain": false, "note": "Blue legend is HYP⁻¹ (inverse hyperbolic); Finseth\'s text shows it as a second \'HYP\' because the superscript is lost."}, {"row": 1, "col": 2, "rowspan": 1, "colspan": 1, "main": "SIN", "shift1": "DIM", "shift2": "SIN⁻¹", "alpha": "", "key_color": "#1E1E1E", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"row": 1, "col": 3, "rowspan": 1, "colspan": 1, "main": "COS", "shift1": "(i)", "shift2": "COS⁻¹", "alpha": "", "key_color": "#1E1E1E", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"row": 1, "col": 4, "rowspan": 1, "colspan": 1, "main": "TAN", "shift1": "I", "shift2": "TAN⁻¹", "alpha": "", "key_color": "#1E1E1E", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"row": 1, "col": 5, "rowspan": 1, "colspan": 1, "main": "EEX", "shift1": "RESULT", "shift2": "π", "alpha": "", "key_color": "#1E1E1E", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"row": 1, "col": 6, "rowspan": 1, "colspan": 1, "main": "4", "shift1": "x≷", "shift2": "SF", "alpha": "", "key_color": "#1E1E1E", "text_color": "#FFFFFF", "uncertain": false, "note": "Gold legend is x≷ (exchange X with a register), often written x<>."}, {"row": 1, "col": 7, "rowspan": 1, "colspan": 1, "main": "5", "shift1": "DSE", "shift2": "CF", "alpha": "", "key_color": "#1E1E1E", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"row": 1, "col": 8, "rowspan": 1, "colspan": 1, "main": "6", "shift1": "ISG", "shift2": "F?", "alpha": "", "key_color": "#1E1E1E", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"row": 1, "col": 9, "rowspan": 1, "colspan": 1, "main": "×", "shift1": "∫ᵧˣ", "shift2": "x=0", "alpha": "", "key_color": "#1E1E1E", "text_color": "#FFFFFF", "uncertain": false, "note": "Gold legend is the integral sign ∫ with limits y and x (∫ᵧˣ)."}, {"row": 2, "col": 0, "rowspan": 1, "colspan": 1, "main": "R/S", "shift1": "PSE", "shift2": "P/R", "alpha": "", "key_color": "#1E1E1E", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"row": 2, "col": 1, "rowspan": 1, "colspan": 1, "main": "GSB", "shift1": "Σ", "shift2": "RTN", "alpha": "", "key_color": "#1E1E1E", "text_color": "#FFFFFF", "uncertain": false, "note": "Gold legend sits under the gold CLEAR bracket, so the full f-function is CLEAR Σ."}, {"row": 2, "col": 2, "rowspan": 1, "colspan": 1, "main": "R↓", "shift1": "PRGM", "shift2": "R↑", "alpha": "", "key_color": "#1E1E1E", "text_color": "#FFFFFF", "uncertain": false, "note": "Gold legend sits under the gold CLEAR bracket, so the full f-function is CLEAR PRGM."}, {"row": 2, "col": 3, "rowspan": 1, "colspan": 1, "main": "x≷y", "shift1": "REG", "shift2": "RND", "alpha": "", "key_color": "#1E1E1E", "text_color": "#FFFFFF", "uncertain": false, "note": "Gold legend sits under the gold CLEAR bracket, so the full f-function is CLEAR REG."}, {"row": 2, "col": 4, "rowspan": 1, "colspan": 1, "main": "←", "shift1": "PREFIX", "shift2": "CLx", "alpha": "", "key_color": "#1E1E1E", "text_color": "#FFFFFF", "uncertain": false, "note": "Gold legend sits under the gold CLEAR bracket, so the full f-function is CLEAR PREFIX."}, {"row": 2, "col": 5, "rowspan": 2, "colspan": 1, "main": "ENTER", "shift1": "RAN #", "shift2": "LSTx", "alpha": "", "key_color": "#1E1E1E", "text_color": "#FFFFFF", "uncertain": false, "note": "Tall ENTER: occupies column 5 of rows 2 and 3."}, {"row": 2, "col": 6, "rowspan": 1, "colspan": 1, "main": "1", "shift1": "→R", "shift2": "→P", "alpha": "", "key_color": "#1E1E1E", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"row": 2, "col": 7, "rowspan": 1, "colspan": 1, "main": "2", "shift1": "→H.MS", "shift2": "→H", "alpha": "", "key_color": "#1E1E1E", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"row": 2, "col": 8, "rowspan": 1, "colspan": 1, "main": "3", "shift1": "→RAD", "shift2": "→DEG", "alpha": "", "key_color": "#1E1E1E", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"row": 2, "col": 9, "rowspan": 1, "colspan": 1, "main": "−", "shift1": "Re≷Im", "shift2": "TEST", "alpha": "", "key_color": "#1E1E1E", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"row": 3, "col": 0, "rowspan": 1, "colspan": 1, "main": "ON", "shift1": "", "shift2": "", "alpha": "", "key_color": "#1E1E1E", "text_color": "#FFFFFF", "uncertain": false, "note": "No shifted functions. Held with other keys at power-on for self-tests / decimal-point swap."}, {"row": 3, "col": 1, "rowspan": 1, "colspan": 1, "main": "f", "shift1": "", "shift2": "", "alpha": "", "key_color": "#E3A63A", "text_color": "#1E1E1E", "uncertain": false, "note": "Gold prefix key; activates the gold legends printed above the keys."}, {"row": 3, "col": 2, "rowspan": 1, "colspan": 1, "main": "g", "shift1": "", "shift2": "", "alpha": "", "key_color": "#5AA7D6", "text_color": "#1E1E1E", "uncertain": false, "note": "Blue prefix key; activates the blue legends printed on the slanted front face of the keys."}, {"row": 3, "col": 3, "rowspan": 1, "colspan": 1, "main": "STO", "shift1": "FRAC", "shift2": "INT", "alpha": "", "key_color": "#1E1E1E", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"row": 3, "col": 4, "rowspan": 1, "colspan": 1, "main": "RCL", "shift1": "USER", "shift2": "MEM", "alpha": "", "key_color": "#1E1E1E", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"row": 3, "col": 6, "rowspan": 1, "colspan": 1, "main": "0", "shift1": "x!", "shift2": "x̄", "alpha": "", "key_color": "#1E1E1E", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"row": 3, "col": 7, "rowspan": 1, "colspan": 1, "main": ".", "shift1": "ŷ,r", "shift2": "s", "alpha": "", "key_color": "#1E1E1E", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"row": 3, "col": 8, "rowspan": 1, "colspan": 1, "main": "Σ+", "shift1": "L.R.", "shift2": "Σ−", "alpha": "", "key_color": "#1E1E1E", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"row": 3, "col": 9, "rowspan": 1, "colspan": 1, "main": "+", "shift1": "Py,x", "shift2": "Cy,x", "alpha": "", "key_color": "#1E1E1E", "text_color": "#FFFFFF", "uncertain": false, "note": ""}], "notes": "Landscape (Voyager) layout: 4 rows x 10 columns, 39 physical keys. ENTER is the tall key in column 5 spanning rows 2-3, so row 3 has no separate key at column 5. Bottom-left of row 3 holds ON, f (gold key) and g (blue key). Main legends are white on black keys; f-legends are gold on the faceplate above each key; g-legends are blue on the key\'s front face. A gold \'CLEAR\' bracket spans above GSB, R↓, x≷y and ← (CLEAR Σ, CLEAR PRGM, CLEAR REG, CLEAR PREFIX). The HP 15C Collector\'s Edition (2023) keeps the identical layout and legends. Hex colours are approximations of the printed/moulded colours, not official values."}, "HP 35s": {"sources": ["https://finseth.com/hpdata/hp35s.php", "https://stefanv.com/calculators/hp35s.html", "https://hpmuseum.org/forum/post-7043.html", "https://www.hpmuseum.org/forum/post-39775.html", "https://literature.hpcalc.org/official/hp35s-qsg-en.pdf", "https://www.manualslib.com/manual/257003/Hp-35s.html?page=18"], "shift_keys": {"left_shift": {"color": "#F2B33D", "label_position": "above the key, printed on the faceplate (yellow)"}, "right_shift": {"color": "#3D8BD4", "label_position": "on the bottom/front face of the key (blue)"}, "alpha": {"color": "#E0457B", "label_position": "small red/pink letter at the bottom-right corner of the key (used after STO/RCL/XEQ/GTO, no ALPHA key)"}}, "sections": [{"name": "top", "rows": 2, "cols": 4, "placement": "upper-left block"}, {"name": "cursor", "rows": 3, "cols": 3, "placement": "right of \'top\', 4 keys in a diamond (only (0,1),(1,0),(1,2),(2,1) used)"}, {"name": "upper", "rows": 2, "cols": 6, "placement": "full width below top/cursor"}, {"name": "lower", "rows": 4, "cols": 5, "placement": "full width, bottom"}], "grid": {"rows": 10, "cols": 6, "note": "overall; use \'sections\' for exact positions"}, "keys": [{"section": "top", "row": 0, "col": 0, "rowspan": 1, "colspan": 1, "main": "R/S", "shift1": "FN=", "shift2": "PRGM", "alpha": "A", "key_color": "#232323", "text_color": "#FFFFFF", "uncertain": false, "note": "Alpha letter from Finseth\'s alpha block only (single source)."}, {"section": "top", "row": 0, "col": 1, "rowspan": 1, "colspan": 1, "main": "GTO", "shift1": "ISG", "shift2": "DSE", "alpha": "B", "key_color": "#232323", "text_color": "#FFFFFF", "uncertain": false, "note": "Alpha letter from Finseth\'s alpha block only (single source)."}, {"section": "top", "row": 0, "col": 2, "rowspan": 1, "colspan": 1, "main": "XEQ", "shift1": "RTN", "shift2": "LBL", "alpha": "C", "key_color": "#232323", "text_color": "#FFFFFF", "uncertain": false, "note": "Alpha letter from Finseth\'s alpha block only (single source)."}, {"section": "top", "row": 0, "col": 3, "rowspan": 1, "colspan": 1, "main": "MODE", "shift1": "x?y", "shift2": "x?0", "alpha": "D", "key_color": "#232323", "text_color": "#FFFFFF", "uncertain": false, "note": "Alpha letter from Finseth\'s alpha block only (single source). MODE is a menu key."}, {"section": "top", "row": 1, "col": 0, "rowspan": 1, "colspan": 1, "main": "RCL", "shift1": "x≷", "shift2": "STO", "alpha": "", "key_color": "#232323", "text_color": "#FFFFFF", "uncertain": false, "note": "STO is the blue (right-shifted) function of RCL - confirmed by Stefan Vorkoetter\'s review."}, {"section": "top", "row": 1, "col": 1, "rowspan": 1, "colspan": 1, "main": "R↓", "shift1": "VIEW", "shift2": "R↑", "alpha": "E", "key_color": "#232323", "text_color": "#FFFFFF", "uncertain": false, "note": "Alpha letter from Finseth\'s alpha block only (single source)."}, {"section": "top", "row": 1, "col": 2, "rowspan": 1, "colspan": 1, "main": "x≷y", "shift1": "INPUT", "shift2": "PSE", "alpha": "F", "key_color": "#232323", "text_color": "#FFFFFF", "uncertain": false, "note": "Alpha letter from Finseth\'s alpha block only (single source)."}, {"section": "top", "row": 1, "col": 3, "rowspan": 1, "colspan": 1, "main": "i", "shift1": "ARG", "shift2": "θ", "alpha": "G", "key_color": "#232323", "text_color": "#FFFFFF", "uncertain": false, "note": "Alpha letter from Finseth\'s alpha block only (single source)."}, {"section": "cursor", "row": 0, "col": 1, "rowspan": 1, "colspan": 1, "main": "▲", "shift1": "FLAGS", "shift2": "", "alpha": "", "key_color": "#232323", "text_color": "#FFFFFF", "uncertain": false, "note": "Cursor key; only a yellow (left-shift) function. Cursor pad sits to the right of the \'top\' section, centred vertically on its two rows."}, {"section": "cursor", "row": 1, "col": 0, "rowspan": 1, "colspan": 1, "main": "◀", "shift1": "DISPLAY", "shift2": "", "alpha": "", "key_color": "#232323", "text_color": "#FFFFFF", "uncertain": false, "note": "Cursor key; only a yellow (left-shift) function. Cursor pad sits to the right of the \'top\' section, centred vertically on its two rows."}, {"section": "cursor", "row": 1, "col": 2, "rowspan": 1, "colspan": 1, "main": "▶", "shift1": "CONST", "shift2": "", "alpha": "", "key_color": "#232323", "text_color": "#FFFFFF", "uncertain": false, "note": "Cursor key; only a yellow (left-shift) function. Cursor pad sits to the right of the \'top\' section, centred vertically on its two rows."}, {"section": "cursor", "row": 2, "col": 1, "rowspan": 1, "colspan": 1, "main": "▼", "shift1": "MEM", "shift2": "", "alpha": "", "key_color": "#232323", "text_color": "#FFFFFF", "uncertain": false, "note": "Cursor key; only a yellow (left-shift) function. Cursor pad sits to the right of the \'top\' section, centred vertically on its two rows."}, {"section": "upper", "row": 0, "col": 0, "rowspan": 1, "colspan": 1, "main": "SIN", "shift1": "HYP", "shift2": "ASIN", "alpha": "H", "key_color": "#232323", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "upper", "row": 0, "col": 1, "rowspan": 1, "colspan": 1, "main": "COS", "shift1": "π", "shift2": "ACOS", "alpha": "I", "key_color": "#232323", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "upper", "row": 0, "col": 2, "rowspan": 1, "colspan": 1, "main": "TAN", "shift1": "INTG", "shift2": "ATAN", "alpha": "J", "key_color": "#232323", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "upper", "row": 0, "col": 3, "rowspan": 1, "colspan": 1, "main": "√x", "shift1": "ˣ√y", "shift2": "x²", "alpha": "K", "key_color": "#232323", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "upper", "row": 0, "col": 4, "rowspan": 1, "colspan": 1, "main": "yˣ", "shift1": "LOG", "shift2": "LN", "alpha": "L", "key_color": "#232323", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "upper", "row": 0, "col": 5, "rowspan": 1, "colspan": 1, "main": "1/x", "shift1": "10ˣ", "shift2": "eˣ", "alpha": "M", "key_color": "#232323", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "upper", "row": 1, "col": 0, "rowspan": 1, "colspan": 2, "main": "ENTER", "shift1": "SHOW", "shift2": "LASTx", "alpha": "", "key_color": "#232323", "text_color": "#FFFFFF", "uncertain": false, "note": "Double-width ENTER spanning columns 0-1."}, {"section": "upper", "row": 1, "col": 2, "rowspan": 1, "colspan": 1, "main": "+/−", "shift1": "=", "shift2": "ABS", "alpha": "N", "key_color": "#232323", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "upper", "row": 1, "col": 3, "rowspan": 1, "colspan": 1, "main": "E", "shift1": "←ENG", "shift2": "RND", "alpha": "O", "key_color": "#232323", "text_color": "#FFFFFF", "uncertain": false, "note": "E = EEX (enter exponent)."}, {"section": "upper", "row": 1, "col": 4, "rowspan": 1, "colspan": 1, "main": "( )", "shift1": "ENG→", "shift2": "[ ]", "alpha": "P", "key_color": "#232323", "text_color": "#FFFFFF", "uncertain": false, "note": "Parentheses key; blue shift gives vector brackets [ ] (Dieter, hpmuseum post-7043; Vorkoetter review)."}, {"section": "upper", "row": 1, "col": 5, "rowspan": 1, "colspan": 1, "main": "←", "shift1": "UNDO", "shift2": "CLEAR", "alpha": "", "key_color": "#232323", "text_color": "#FFFFFF", "uncertain": false, "note": "Backspace. CLEAR is a menu."}, {"section": "lower", "row": 0, "col": 0, "rowspan": 1, "colspan": 1, "main": "EQN", "shift1": "∫FN", "shift2": "SOLVE", "alpha": "Q", "key_color": "#232323", "text_color": "#FFFFFF", "uncertain": true, "note": "Finseth encodes the yellow legend as \'\\\\GI\'; read as the integrate function ∫FN (pairs with FN= on R/S). Verify against a photo."}, {"section": "lower", "row": 0, "col": 1, "rowspan": 1, "colspan": 1, "main": "7", "shift1": "→°F", "shift2": "→°C", "alpha": "R", "key_color": "#232323", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "lower", "row": 0, "col": 2, "rowspan": 1, "colspan": 1, "main": "8", "shift1": "HMS→", "shift2": "→HMS", "alpha": "S", "key_color": "#232323", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "lower", "row": 0, "col": 3, "rowspan": 1, "colspan": 1, "main": "9", "shift1": "→RAD", "shift2": "→DEG", "alpha": "T", "key_color": "#232323", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "lower", "row": 0, "col": 4, "rowspan": 1, "colspan": 1, "main": "÷", "shift1": "%CHG", "shift2": "%", "alpha": "", "key_color": "#232323", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "lower", "row": 1, "col": 0, "rowspan": 1, "colspan": 1, "main": "◀ (left shift)", "shift1": "", "shift2": "", "alpha": "", "key_color": "#F2B33D", "text_color": "#1E1E1E", "uncertain": false, "note": "Yellow/gold left-shift key (key cap printed with a bent arrow)."}, {"section": "lower", "row": 1, "col": 1, "rowspan": 1, "colspan": 1, "main": "4", "shift1": "→lb", "shift2": "→kg", "alpha": "U", "key_color": "#232323", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "lower", "row": 1, "col": 2, "rowspan": 1, "colspan": 1, "main": "5", "shift1": "→MILE", "shift2": "→KM", "alpha": "V", "key_color": "#232323", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "lower", "row": 1, "col": 3, "rowspan": 1, "colspan": 1, "main": "6", "shift1": "→in", "shift2": "→cm", "alpha": "W", "key_color": "#232323", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "lower", "row": 1, "col": 4, "rowspan": 1, "colspan": 1, "main": "×", "shift1": "nCr", "shift2": "nPr", "alpha": "", "key_color": "#232323", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "lower", "row": 2, "col": 0, "rowspan": 1, "colspan": 1, "main": "▶ (right shift)", "shift1": "", "shift2": "", "alpha": "", "key_color": "#3D8BD4", "text_color": "#1E1E1E", "uncertain": false, "note": "Blue right-shift key (key cap printed with a bent arrow)."}, {"section": "lower", "row": 2, "col": 1, "rowspan": 1, "colspan": 1, "main": "1", "shift1": "LOGIC", "shift2": "BASE", "alpha": "X", "key_color": "#232323", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "lower", "row": 2, "col": 2, "rowspan": 1, "colspan": 1, "main": "2", "shift1": "→gal", "shift2": "→l", "alpha": "Y", "key_color": "#232323", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "lower", "row": 2, "col": 3, "rowspan": 1, "colspan": 1, "main": "3", "shift1": "SEED", "shift2": "RAND", "alpha": "Z", "key_color": "#232323", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "lower", "row": 2, "col": 4, "rowspan": 1, "colspan": 1, "main": "−", "shift1": "L.R.", "shift2": "SUMS", "alpha": "", "key_color": "#232323", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "lower", "row": 3, "col": 0, "rowspan": 1, "colspan": 1, "main": "C", "shift1": "OFF", "shift2": "ON", "alpha": "", "key_color": "#232323", "text_color": "#FFFFFF", "uncertain": false, "note": "Clear/cancel key; turns calculator on. \'ON\' is printed in blue on the bottom (front) edge; OFF in yellow above."}, {"section": "lower", "row": 3, "col": 1, "rowspan": 1, "colspan": 1, "main": "0", "shift1": ",", "shift2": "SPACE", "alpha": "(I)", "key_color": "#232323", "text_color": "#FFFFFF", "uncertain": true, "note": "Yellow \',\' confirmed (hpmuseum post-7043: comma on yellow-shifted zero). Red \'(I)\' read from Finseth alpha line \'(i)\'; manual says indirect registers (I)/(J) are on specific keys but key glyphs are lost in text."}, {"section": "lower", "row": 3, "col": 2, "rowspan": 1, "colspan": 1, "main": ".", "shift1": "/c", "shift2": "FDISP", "alpha": "(J)", "key_color": "#232323", "text_color": "#FFFFFF", "uncertain": true, "note": "Yellow \'/c\' (max fraction denominator) confirmed by hpmuseum post-39775 (\'left shift /c\'). \'a b/c\' reminder printed in red below the key. Red \'(J)\' read from Finseth alpha line; exact red legend arrangement uncertain."}, {"section": "lower", "row": 3, "col": 3, "rowspan": 1, "colspan": 1, "main": "Σ+", "shift1": "Σ−", "shift2": "!", "alpha": "", "key_color": "#232323", "text_color": "#FFFFFF", "uncertain": false, "note": "Blue \'!\' is factorial (x!)."}, {"section": "lower", "row": 3, "col": 4, "rowspan": 1, "colspan": 1, "main": "+", "shift1": "x̄,ȳ", "shift2": "s,σ", "alpha": "", "key_color": "#232323", "text_color": "#FFFFFF", "uncertain": false, "note": "Both shifted legends are menus (x̄,ȳ means; s,σ standard deviations)."}], "notes": "43 keys counted here (8 top + 4 cursor + 11 upper + 20 lower). shift1 = yellow left-shift, shift2 = blue right-shift, alpha = red letter. The yellow shift key is in the first column of the 2nd row of the lower block (left of 4) and the blue shift key below it (left of 1); the C/ON key is bottom-left. R/S is top-left (not bottom-right as on classic HPs). ENTER is double width. 16 keys open menus. Finseth\'s text map is the primary positional source; HP\'s own QSG/manual confirm the colour scheme (yellow above, blue on key bottom, letters bottom-right, cursor keys yellow-only) but their key glyphs do not survive text extraction. Colours are approximations."}, "TI-30XS MultiView": {"sources": ["https://education.ti.com/html/eguides/scientifics/PDFs/EN/TI30XS-MultiView-Guide_EN.pdf", "https://education.ti.com/html/eguides/scientifics/TI-30XS-MultiView/EN/Content/EG_SmartView/M_KeyBoard/M_KeyBoard.HTML", "https://cdn.geogebra.org/resource/dckja6sx/Hxe3aChHaf42lER1/material-dckja6sx.pdf", "https://tech.aph.org/Orion%20TI-30XS%20Documents/xs_doc.htm", "https://www.rskey.org/~mwsebastian/reviews/ti30xsmv.htm", "https://education.ti.com/en/customer-support/knowledge-base/scientific-elem-calculators/product-usage/20309"], "shift_keys": {"2nd": {"color": "#3BAA4B", "label_position": "above the key (green text)"}}, "sections": [{"name": "top", "rows": 2, "cols": 3, "placement": "upper-left, under the display"}, {"name": "arrows", "rows": 3, "cols": 3, "placement": "4-way rocker in the top-right corner beside \'top\' (occupies the space of columns 3-4 of a 5-column grid)"}, {"name": "main", "rows": 7, "cols": 5, "placement": "full width below"}], "grid": {"rows": 9, "cols": 5, "note": "overall; \'top\' + \'arrows\' form the first two rows"}, "keys": [{"section": "top", "row": 0, "col": 0, "rowspan": 1, "colspan": 1, "main": "2nd", "shift1": "", "shift2": "", "alpha": "", "key_color": "#3BAA4B", "text_color": "#FFFFFF", "uncertain": false, "note": "Green 2nd key; 2nd legends are printed in green above keys."}, {"section": "top", "row": 0, "col": 1, "rowspan": 1, "colspan": 1, "main": "mode", "shift1": "quit", "shift2": "", "alpha": "", "key_color": "#2E2E2E", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "top", "row": 0, "col": 2, "rowspan": 1, "colspan": 1, "main": "delete", "shift1": "insert", "shift2": "", "alpha": "", "key_color": "#2E2E2E", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "top", "row": 1, "col": 0, "rowspan": 1, "colspan": 1, "main": "log", "shift1": "10ˣ", "shift2": "", "alpha": "", "key_color": "#2E2E2E", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "top", "row": 1, "col": 1, "rowspan": 1, "colspan": 1, "main": "prb", "shift1": "angle", "shift2": "", "alpha": "", "key_color": "#2E2E2E", "text_color": "#FFFFFF", "uncertain": true, "note": "Key position derived (see notes). 2nd legend (angle) from memory; guidebook confirms 2nd[angle] and 2nd[stat] exist but text extraction loses which key."}, {"section": "top", "row": 1, "col": 2, "rowspan": 1, "colspan": 1, "main": "data", "shift1": "stat", "shift2": "", "alpha": "", "key_color": "#2E2E2E", "text_color": "#FFFFFF", "uncertain": true, "note": "Key position derived (see notes). 2nd legend (stat) from memory; guidebook confirms 2nd[angle] and 2nd[stat] exist but text extraction loses which key."}, {"section": "arrows", "row": 0, "col": 1, "rowspan": 1, "colspan": 1, "main": "▲", "shift1": "", "shift2": "", "alpha": "", "key_color": "#9B9B9B", "text_color": "#1E1E1E", "uncertain": false, "note": "Part of 4-way rocker."}, {"section": "arrows", "row": 1, "col": 0, "rowspan": 1, "colspan": 1, "main": "◀", "shift1": "", "shift2": "", "alpha": "", "key_color": "#9B9B9B", "text_color": "#1E1E1E", "uncertain": false, "note": "Part of 4-way rocker. 2nd ◀ jumps to start of entry (no printed legend)"}, {"section": "arrows", "row": 1, "col": 2, "rowspan": 1, "colspan": 1, "main": "▶", "shift1": "", "shift2": "", "alpha": "", "key_color": "#9B9B9B", "text_color": "#1E1E1E", "uncertain": false, "note": "Part of 4-way rocker. 2nd ▶ jumps to end of entry (no printed legend)"}, {"section": "arrows", "row": 2, "col": 1, "rowspan": 1, "colspan": 1, "main": "▼", "shift1": "", "shift2": "", "alpha": "", "key_color": "#9B9B9B", "text_color": "#1E1E1E", "uncertain": false, "note": "Part of 4-way rocker."}, {"section": "main", "row": 0, "col": 0, "rowspan": 1, "colspan": 1, "main": "ln", "shift1": "eˣ", "shift2": "", "alpha": "", "key_color": "#2E2E2E", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "main", "row": 0, "col": 1, "rowspan": 1, "colspan": 1, "main": "n/d", "shift1": "U n/d", "shift2": "", "alpha": "", "key_color": "#2E2E2E", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "main", "row": 0, "col": 2, "rowspan": 1, "colspan": 1, "main": "×10ⁿ", "shift1": "n/d◀▶U n/d", "shift2": "", "alpha": "", "key_color": "#2E2E2E", "text_color": "#FFFFFF", "uncertain": false, "note": "2nd n/d◀▶U n/d confirmed by GED reference sheet (2nd + this key toggles fraction/mixed)."}, {"section": "main", "row": 0, "col": 3, "rowspan": 1, "colspan": 1, "main": "table", "shift1": "f◀▶d", "shift2": "", "alpha": "", "key_color": "#2E2E2E", "text_color": "#FFFFFF", "uncertain": false, "note": "2nd f◀▶d confirmed by GED reference sheet."}, {"section": "main", "row": 0, "col": 4, "rowspan": 1, "colspan": 1, "main": "clear", "shift1": "", "shift2": "", "alpha": "", "key_color": "#2E2E2E", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "main", "row": 1, "col": 0, "rowspan": 1, "colspan": 1, "main": "π", "shift1": "hyp", "shift2": "", "alpha": "", "key_color": "#2E2E2E", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "main", "row": 1, "col": 1, "rowspan": 1, "colspan": 1, "main": "sin", "shift1": "sin⁻¹", "shift2": "", "alpha": "", "key_color": "#2E2E2E", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "main", "row": 1, "col": 2, "rowspan": 1, "colspan": 1, "main": "cos", "shift1": "cos⁻¹", "shift2": "", "alpha": "", "key_color": "#2E2E2E", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "main", "row": 1, "col": 3, "rowspan": 1, "colspan": 1, "main": "tan", "shift1": "tan⁻¹", "shift2": "", "alpha": "", "key_color": "#2E2E2E", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "main", "row": 1, "col": 4, "rowspan": 1, "colspan": 1, "main": "÷", "shift1": "K", "shift2": "", "alpha": "", "key_color": "#2E2E2E", "text_color": "#FFFFFF", "uncertain": true, "note": "2nd legend K (constant) placed from memory; guidebook confirms 2nd[K] exists."}, {"section": "main", "row": 2, "col": 0, "rowspan": 1, "colspan": 1, "main": "^", "shift1": "ˣ√", "shift2": "", "alpha": "", "key_color": "#2E2E2E", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "main", "row": 2, "col": 1, "rowspan": 1, "colspan": 1, "main": "x⁻¹", "shift1": "", "shift2": "", "alpha": "", "key_color": "#2E2E2E", "text_color": "#FFFFFF", "uncertain": true, "note": "Position inferred from TI-SmartView key-letter sequence (only remaining key for slot). 2nd legend unknown."}, {"section": "main", "row": 2, "col": 2, "rowspan": 1, "colspan": 1, "main": "(", "shift1": "%", "shift2": "", "alpha": "", "key_color": "#2E2E2E", "text_color": "#FFFFFF", "uncertain": true, "note": "2nd legend (%) placed from memory; TI KB 20309 confirms 2nd[%] and 2nd[▶%] exist but not which key."}, {"section": "main", "row": 2, "col": 3, "rowspan": 1, "colspan": 1, "main": ")", "shift1": "▶%", "shift2": "", "alpha": "", "key_color": "#2E2E2E", "text_color": "#FFFFFF", "uncertain": true, "note": "2nd legend (▶%) placed from memory; TI KB 20309 confirms 2nd[%] and 2nd[▶%] exist but not which key."}, {"section": "main", "row": 2, "col": 4, "rowspan": 1, "colspan": 1, "main": "×", "shift1": "", "shift2": "", "alpha": "", "key_color": "#2E2E2E", "text_color": "#FFFFFF", "uncertain": true, "note": "2nd legend unknown/unverified (left blank rather than guessed)."}, {"section": "main", "row": 3, "col": 0, "rowspan": 1, "colspan": 1, "main": "x²", "shift1": "√", "shift2": "", "alpha": "", "key_color": "#2E2E2E", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "main", "row": 3, "col": 1, "rowspan": 1, "colspan": 1, "main": "7", "shift1": "", "shift2": "", "alpha": "", "key_color": "#ECECEC", "text_color": "#1E1E1E", "uncertain": true, "note": "2nd legend unknown/unverified (left blank rather than guessed)."}, {"section": "main", "row": 3, "col": 2, "rowspan": 1, "colspan": 1, "main": "8", "shift1": "", "shift2": "", "alpha": "", "key_color": "#ECECEC", "text_color": "#1E1E1E", "uncertain": true, "note": "2nd legend unknown/unverified (left blank rather than guessed)."}, {"section": "main", "row": 3, "col": 3, "rowspan": 1, "colspan": 1, "main": "9", "shift1": "", "shift2": "", "alpha": "", "key_color": "#ECECEC", "text_color": "#1E1E1E", "uncertain": true, "note": "2nd legend unknown/unverified (left blank rather than guessed)."}, {"section": "main", "row": 3, "col": 4, "rowspan": 1, "colspan": 1, "main": "−", "shift1": "", "shift2": "", "alpha": "", "key_color": "#2E2E2E", "text_color": "#FFFFFF", "uncertain": true, "note": "2nd legend unknown/unverified (left blank rather than guessed)."}, {"section": "main", "row": 4, "col": 0, "rowspan": 1, "colspan": 1, "main": "x y z t a b c", "shift1": "clear var", "shift2": "", "alpha": "", "key_color": "#2E2E2E", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "main", "row": 4, "col": 1, "rowspan": 1, "colspan": 1, "main": "4", "shift1": "", "shift2": "", "alpha": "", "key_color": "#ECECEC", "text_color": "#1E1E1E", "uncertain": true, "note": "2nd legend unknown/unverified (left blank rather than guessed)."}, {"section": "main", "row": 4, "col": 2, "rowspan": 1, "colspan": 1, "main": "5", "shift1": "", "shift2": "", "alpha": "", "key_color": "#ECECEC", "text_color": "#1E1E1E", "uncertain": true, "note": "2nd legend unknown/unverified (left blank rather than guessed)."}, {"section": "main", "row": 4, "col": 3, "rowspan": 1, "colspan": 1, "main": "6", "shift1": "", "shift2": "", "alpha": "", "key_color": "#ECECEC", "text_color": "#1E1E1E", "uncertain": true, "note": "2nd legend unknown/unverified (left blank rather than guessed)."}, {"section": "main", "row": 4, "col": 4, "rowspan": 1, "colspan": 1, "main": "+", "shift1": "", "shift2": "", "alpha": "", "key_color": "#2E2E2E", "text_color": "#FFFFFF", "uncertain": true, "note": "2nd legend unknown/unverified (left blank rather than guessed)."}, {"section": "main", "row": 5, "col": 0, "rowspan": 1, "colspan": 1, "main": "sto→", "shift1": "recall", "shift2": "", "alpha": "", "key_color": "#2E2E2E", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "main", "row": 5, "col": 1, "rowspan": 1, "colspan": 1, "main": "1", "shift1": "", "shift2": "", "alpha": "", "key_color": "#ECECEC", "text_color": "#1E1E1E", "uncertain": true, "note": "2nd legend unknown/unverified (left blank rather than guessed)."}, {"section": "main", "row": 5, "col": 2, "rowspan": 1, "colspan": 1, "main": "2", "shift1": "", "shift2": "", "alpha": "", "key_color": "#ECECEC", "text_color": "#1E1E1E", "uncertain": true, "note": "2nd legend unknown/unverified (left blank rather than guessed)."}, {"section": "main", "row": 5, "col": 3, "rowspan": 1, "colspan": 1, "main": "3", "shift1": "", "shift2": "", "alpha": "", "key_color": "#ECECEC", "text_color": "#1E1E1E", "uncertain": true, "note": "2nd legend unknown/unverified (left blank rather than guessed)."}, {"section": "main", "row": 5, "col": 4, "rowspan": 1, "colspan": 1, "main": "◀▶", "shift1": "", "shift2": "", "alpha": "", "key_color": "#2E2E2E", "text_color": "#FFFFFF", "uncertain": true, "note": "Answer-toggle key (switches fraction/decimal/π display). Position directly above enter derived from SmartView key-letter sequence; 2nd legend unknown."}, {"section": "main", "row": 6, "col": 0, "rowspan": 1, "colspan": 1, "main": "on", "shift1": "off", "shift2": "", "alpha": "", "key_color": "#2E2E2E", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "main", "row": 6, "col": 1, "rowspan": 1, "colspan": 1, "main": "0", "shift1": "reset", "shift2": "", "alpha": "", "key_color": "#ECECEC", "text_color": "#1E1E1E", "uncertain": false, "note": "2nd reset confirmed by GED sheet (\'2nd 0 2\' = reset all)."}, {"section": "main", "row": 6, "col": 2, "rowspan": 1, "colspan": 1, "main": ".", "shift1": "", "shift2": "", "alpha": "", "key_color": "#ECECEC", "text_color": "#1E1E1E", "uncertain": true, "note": "2nd legend unknown/unverified (left blank rather than guessed)."}, {"section": "main", "row": 6, "col": 3, "rowspan": 1, "colspan": 1, "main": "(−)", "shift1": "Ans", "shift2": "", "alpha": "", "key_color": "#ECECEC", "text_color": "#1E1E1E", "uncertain": true, "note": "2nd Ans from memory; Ans is a confirmed 2nd function."}, {"section": "main", "row": 6, "col": 4, "rowspan": 1, "colspan": 1, "main": "enter", "shift1": "", "shift2": "", "alpha": "", "key_color": "#2E2E2E", "text_color": "#FFFFFF", "uncertain": true, "note": "2nd legend unknown/unverified (left blank rather than guessed)."}], "notes": "45 keys including the 4 rocker positions (rskey.org review; APH guide: 41 keys in a 9x5 arrangement plus a 4-key directional pad top-right; ON bottom-left, 2nd top-left). No ALPHA key; variables come from the \'x y z t a b c\' key. POSITION METHOD: TI\'s own guidebooks render keys in a symbol font that does not survive text extraction, and the keypad figures are images. Positions were reconstructed from the TI-SmartView computer-key shortcuts (mode=[W] stated by TI; a GED reference sheet using the same shortcut letters gives q=2nd, w=mode, e=delete, r/u=log/ln, t=prb, y=data, i=n/d, p=×10ⁿ, a=table, s=clear, d=π, f/g/h=sin/cos/tan, j=^, l=x², c=toggle, n=enter). These letters run in keyboard order (q w e r t y u i [o skipped] p a s d f g h j k l z x c v b n) across the non-digit/non-operator keys row by row, which pins every key into a consistent 9x5 layout. Treat as strong inference, not a direct reading of a keypad image. Colours approximate (dark function keys, light number keys, green 2nd key; varies by case colour)."}, "TI-84 Plus CE": {"sources": ["https://education.ti.com/html/eguides/graphing/84CE-Online-Calculator/EN/Content/EG_GSGuide/M_Map/KB_Key_Map.HTML", "https://learn.cemetech.net/index.php/TI-BASIC:Key_Codes", "https://learn.cemetech.net/index.php/TI-BASIC:Getkey", "https://education.ti.com/en/product-resources/guidebooks/ti-84-plus-ce-python"], "shift_keys": {"2nd": {"color": "#3FA4DC", "label_position": "above the key, left side (blue)"}, "alpha": {"color": "#62B446", "label_position": "above the key, right side (green)"}}, "sections": [{"name": "top_row", "rows": 1, "cols": 5, "placement": "directly under the screen"}, {"name": "upper_left", "rows": 2, "cols": 3, "placement": "left 3 columns under top_row"}, {"name": "arrows", "rows": 3, "cols": 3, "placement": "right of upper_left (diamond; only (0,1),(1,0),(1,2),(2,1) used)"}, {"name": "main", "rows": 7, "cols": 5, "placement": "full width below"}], "grid": {"rows": 10, "cols": 5, "note": "getKey rows 1-10; use \'sections\' for exact positions"}, "keys": [{"section": "top_row", "row": 0, "col": 0, "rowspan": 1, "colspan": 1, "main": "y=", "shift1": "stat plot", "shift2": "", "alpha": "f1", "key_color": "#2B2B2B", "text_color": "#FFFFFF", "uncertain": false, "note": "Graph keys; 2nd legends blue, alpha (f1-f5 menu softkeys) green."}, {"section": "top_row", "row": 0, "col": 1, "rowspan": 1, "colspan": 1, "main": "window", "shift1": "tblset", "shift2": "", "alpha": "f2", "key_color": "#2B2B2B", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "top_row", "row": 0, "col": 2, "rowspan": 1, "colspan": 1, "main": "zoom", "shift1": "format", "shift2": "", "alpha": "f3", "key_color": "#2B2B2B", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "top_row", "row": 0, "col": 3, "rowspan": 1, "colspan": 1, "main": "trace", "shift1": "calc", "shift2": "", "alpha": "f4", "key_color": "#2B2B2B", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "top_row", "row": 0, "col": 4, "rowspan": 1, "colspan": 1, "main": "graph", "shift1": "table", "shift2": "", "alpha": "f5", "key_color": "#2B2B2B", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "upper_left", "row": 0, "col": 0, "rowspan": 1, "colspan": 1, "main": "2nd", "shift1": "", "shift2": "", "alpha": "", "key_color": "#3FA4DC", "text_color": "#FFFFFF", "uncertain": false, "note": "Blue 2nd key (getKey 21)."}, {"section": "upper_left", "row": 0, "col": 1, "rowspan": 1, "colspan": 1, "main": "mode", "shift1": "quit", "shift2": "", "alpha": "", "key_color": "#2B2B2B", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "upper_left", "row": 0, "col": 2, "rowspan": 1, "colspan": 1, "main": "del", "shift1": "ins", "shift2": "", "alpha": "", "key_color": "#2B2B2B", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "upper_left", "row": 1, "col": 0, "rowspan": 1, "colspan": 1, "main": "alpha", "shift1": "A-lock", "shift2": "", "alpha": "", "key_color": "#62B446", "text_color": "#FFFFFF", "uncertain": false, "note": "Green alpha key (getKey 31)."}, {"section": "upper_left", "row": 1, "col": 1, "rowspan": 1, "colspan": 1, "main": "X,T,θ,n", "shift1": "link", "shift2": "", "alpha": "", "key_color": "#2B2B2B", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "upper_left", "row": 1, "col": 2, "rowspan": 1, "colspan": 1, "main": "stat", "shift1": "list", "shift2": "", "alpha": "", "key_color": "#2B2B2B", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "arrows", "row": 0, "col": 1, "rowspan": 1, "colspan": 1, "main": "▲", "shift1": "", "shift2": "", "alpha": "", "key_color": "#A6A6A6", "text_color": "#1E1E1E", "uncertain": false, "note": "Arrow pad to the right of \'upper_left\' (getKey 25). 2nd ▲/▼ adjust screen brightness on the CE (no printed legend)."}, {"section": "arrows", "row": 1, "col": 0, "rowspan": 1, "colspan": 1, "main": "◀", "shift1": "", "shift2": "", "alpha": "", "key_color": "#A6A6A6", "text_color": "#1E1E1E", "uncertain": false, "note": "Arrow pad to the right of \'upper_left\' (getKey 24). 2nd ▲/▼ adjust screen brightness on the CE (no printed legend)."}, {"section": "arrows", "row": 1, "col": 2, "rowspan": 1, "colspan": 1, "main": "▶", "shift1": "", "shift2": "", "alpha": "", "key_color": "#A6A6A6", "text_color": "#1E1E1E", "uncertain": false, "note": "Arrow pad to the right of \'upper_left\' (getKey 26). 2nd ▲/▼ adjust screen brightness on the CE (no printed legend)."}, {"section": "arrows", "row": 2, "col": 1, "rowspan": 1, "colspan": 1, "main": "▼", "shift1": "", "shift2": "", "alpha": "", "key_color": "#A6A6A6", "text_color": "#1E1E1E", "uncertain": false, "note": "Arrow pad to the right of \'upper_left\' (getKey 34). 2nd ▲/▼ adjust screen brightness on the CE (no printed legend)."}, {"section": "main", "row": 0, "col": 0, "rowspan": 1, "colspan": 1, "main": "math", "shift1": "test", "shift2": "", "alpha": "A", "key_color": "#2B2B2B", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "main", "row": 0, "col": 1, "rowspan": 1, "colspan": 1, "main": "apps", "shift1": "angle", "shift2": "", "alpha": "B", "key_color": "#2B2B2B", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "main", "row": 0, "col": 2, "rowspan": 1, "colspan": 1, "main": "prgm", "shift1": "draw", "shift2": "", "alpha": "C", "key_color": "#2B2B2B", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "main", "row": 0, "col": 3, "rowspan": 1, "colspan": 1, "main": "vars", "shift1": "distr", "shift2": "", "alpha": "", "key_color": "#2B2B2B", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "main", "row": 0, "col": 4, "rowspan": 1, "colspan": 1, "main": "clear", "shift1": "", "shift2": "", "alpha": "", "key_color": "#2B2B2B", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "main", "row": 1, "col": 0, "rowspan": 1, "colspan": 1, "main": "x⁻¹", "shift1": "matrix", "shift2": "", "alpha": "D", "key_color": "#2B2B2B", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "main", "row": 1, "col": 1, "rowspan": 1, "colspan": 1, "main": "sin", "shift1": "sin⁻¹", "shift2": "", "alpha": "E", "key_color": "#2B2B2B", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "main", "row": 1, "col": 2, "rowspan": 1, "colspan": 1, "main": "cos", "shift1": "cos⁻¹", "shift2": "", "alpha": "F", "key_color": "#2B2B2B", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "main", "row": 1, "col": 3, "rowspan": 1, "colspan": 1, "main": "tan", "shift1": "tan⁻¹", "shift2": "", "alpha": "G", "key_color": "#2B2B2B", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "main", "row": 1, "col": 4, "rowspan": 1, "colspan": 1, "main": "^", "shift1": "π", "shift2": "", "alpha": "H", "key_color": "#2B2B2B", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "main", "row": 2, "col": 0, "rowspan": 1, "colspan": 1, "main": "x²", "shift1": "√", "shift2": "", "alpha": "I", "key_color": "#2B2B2B", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "main", "row": 2, "col": 1, "rowspan": 1, "colspan": 1, "main": ",", "shift1": "EE", "shift2": "", "alpha": "J", "key_color": "#2B2B2B", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "main", "row": 2, "col": 2, "rowspan": 1, "colspan": 1, "main": "(", "shift1": "{", "shift2": "", "alpha": "K", "key_color": "#2B2B2B", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "main", "row": 2, "col": 3, "rowspan": 1, "colspan": 1, "main": ")", "shift1": "}", "shift2": "", "alpha": "L", "key_color": "#2B2B2B", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "main", "row": 2, "col": 4, "rowspan": 1, "colspan": 1, "main": "÷", "shift1": "e", "shift2": "", "alpha": "M", "key_color": "#2B2B2B", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "main", "row": 3, "col": 0, "rowspan": 1, "colspan": 1, "main": "log", "shift1": "10ˣ", "shift2": "", "alpha": "N", "key_color": "#2B2B2B", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "main", "row": 3, "col": 1, "rowspan": 1, "colspan": 1, "main": "7", "shift1": "u", "shift2": "", "alpha": "O", "key_color": "#E9E9E9", "text_color": "#1E1E1E", "uncertain": false, "note": ""}, {"section": "main", "row": 3, "col": 2, "rowspan": 1, "colspan": 1, "main": "8", "shift1": "v", "shift2": "", "alpha": "P", "key_color": "#E9E9E9", "text_color": "#1E1E1E", "uncertain": false, "note": ""}, {"section": "main", "row": 3, "col": 3, "rowspan": 1, "colspan": 1, "main": "9", "shift1": "w", "shift2": "", "alpha": "Q", "key_color": "#E9E9E9", "text_color": "#1E1E1E", "uncertain": false, "note": ""}, {"section": "main", "row": 3, "col": 4, "rowspan": 1, "colspan": 1, "main": "×", "shift1": "[", "shift2": "", "alpha": "R", "key_color": "#2B2B2B", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "main", "row": 4, "col": 0, "rowspan": 1, "colspan": 1, "main": "ln", "shift1": "eˣ", "shift2": "", "alpha": "S", "key_color": "#2B2B2B", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "main", "row": 4, "col": 1, "rowspan": 1, "colspan": 1, "main": "4", "shift1": "L4", "shift2": "", "alpha": "T", "key_color": "#E9E9E9", "text_color": "#1E1E1E", "uncertain": false, "note": ""}, {"section": "main", "row": 4, "col": 2, "rowspan": 1, "colspan": 1, "main": "5", "shift1": "L5", "shift2": "", "alpha": "U", "key_color": "#E9E9E9", "text_color": "#1E1E1E", "uncertain": false, "note": ""}, {"section": "main", "row": 4, "col": 3, "rowspan": 1, "colspan": 1, "main": "6", "shift1": "L6", "shift2": "", "alpha": "V", "key_color": "#E9E9E9", "text_color": "#1E1E1E", "uncertain": false, "note": ""}, {"section": "main", "row": 4, "col": 4, "rowspan": 1, "colspan": 1, "main": "−", "shift1": "]", "shift2": "", "alpha": "W", "key_color": "#2B2B2B", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "main", "row": 5, "col": 0, "rowspan": 1, "colspan": 1, "main": "sto→", "shift1": "rcl", "shift2": "", "alpha": "X", "key_color": "#2B2B2B", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "main", "row": 5, "col": 1, "rowspan": 1, "colspan": 1, "main": "1", "shift1": "L1", "shift2": "", "alpha": "Y", "key_color": "#E9E9E9", "text_color": "#1E1E1E", "uncertain": false, "note": ""}, {"section": "main", "row": 5, "col": 2, "rowspan": 1, "colspan": 1, "main": "2", "shift1": "L2", "shift2": "", "alpha": "Z", "key_color": "#E9E9E9", "text_color": "#1E1E1E", "uncertain": false, "note": ""}, {"section": "main", "row": 5, "col": 3, "rowspan": 1, "colspan": 1, "main": "3", "shift1": "L3", "shift2": "", "alpha": "θ", "key_color": "#E9E9E9", "text_color": "#1E1E1E", "uncertain": false, "note": ""}, {"section": "main", "row": 5, "col": 4, "rowspan": 1, "colspan": 1, "main": "+", "shift1": "mem", "shift2": "", "alpha": "\\"", "key_color": "#2B2B2B", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "main", "row": 6, "col": 0, "rowspan": 1, "colspan": 1, "main": "on", "shift1": "off", "shift2": "", "alpha": "", "key_color": "#2B2B2B", "text_color": "#FFFFFF", "uncertain": false, "note": "ON has no getKey code."}, {"section": "main", "row": 6, "col": 1, "rowspan": 1, "colspan": 1, "main": "0", "shift1": "catalog", "shift2": "", "alpha": "␣", "key_color": "#E9E9E9", "text_color": "#1E1E1E", "uncertain": false, "note": "Alpha legend is a space character (␣)."}, {"section": "main", "row": 6, "col": 2, "rowspan": 1, "colspan": 1, "main": ".", "shift1": "i", "shift2": "", "alpha": ":", "key_color": "#E9E9E9", "text_color": "#1E1E1E", "uncertain": false, "note": ""}, {"section": "main", "row": 6, "col": 3, "rowspan": 1, "colspan": 1, "main": "(−)", "shift1": "ans", "shift2": "", "alpha": "?", "key_color": "#E9E9E9", "text_color": "#1E1E1E", "uncertain": false, "note": ""}, {"section": "main", "row": 6, "col": 4, "rowspan": 1, "colspan": 1, "main": "enter", "shift1": "entry", "shift2": "", "alpha": "solve", "key_color": "#2B2B2B", "text_color": "#FFFFFF", "uncertain": false, "note": ""}], "notes": "50 keys (5 + 6 + 4 arrows + 35). Positions cross-checked two ways: TI\'s online-calculator key map lists keys in row order, and the getKey code grid (row*10+col: 11-15, 21-23 + arrows 24/25/26/34, 31-33, 41-45 ... 101-105) fixes rows/columns. Legend text from TI\'s standard keyboard (identical to TI-84 Plus except CE uses f1-f5 alpha legends on the top row). Key/body colours vary by case colour; hex values are approximate (dark function keys, light number keys, blue 2nd, green alpha, grey arrows)."}, "Casio fx-991EX": {"sources": ["https://www.casio.com/content/dam/casio/global/support/manuals/calculators/pdf/004-en/f/fx-570_991EX_EN.pdf", "https://de.ifixit.com/Document/DynfKy4jK4lucArw/QUICK-START-GUIDE-fx-991EX.pdf", "https://www.manualslib.com/manual/1260605/Casio-Fx-991ex.html", "https://pundit.pratt.duke.edu/wiki/Calculator_Tips/Classwiz", "https://hpmuseum.org/forum/thread-4442-page-3.html", "https://www.casio.com/content/dam/casio/global/support/manuals/calculators/pdf/004-en/f/fx-115ESPLUS_991ESPLUSC_EN.pdf", "https://support.casio.com/global/en/calc/manual/fx-570ESPLUS_991ESPLUS_en/before_using_calculator/getting_started.html"], "shift_keys": {"SHIFT": {"color": "#F2C230", "label_position": "above the key, left (yellow)"}, "ALPHA": {"color": "#E0453A", "label_position": "above the key, right (red)"}, "complex_mode": {"color": "#9B59B6", "label_position": "purple legends/brackets (Complex mode)"}, "base_n": {"color": "#3D7CC9", "label_position": "blue legends/brackets (Base-N: DEC HEX BIN OCT)"}}, "sections": [{"name": "top", "rows": 2, "cols": 6, "placement": "under the display; columns 2-3 are occupied by the cursor pad"}, {"name": "cursor", "rows": 3, "cols": 3, "placement": "round pad centred in \'top\' columns 2-3"}, {"name": "function", "rows": 3, "cols": 6, "placement": "6-column scientific block"}, {"name": "numeric", "rows": 4, "cols": 5, "placement": "5-column number block (larger keys)"}], "grid": {"rows": 9, "cols": 6, "note": "overall; use \'sections\'"}, "keys": [{"section": "top", "row": 0, "col": 0, "rowspan": 1, "colspan": 1, "main": "SHIFT", "shift1": "", "shift2": "", "alpha": "", "key_color": "#2A2D33", "text_color": "#F2C230", "uncertain": false, "note": "SHIFT key: dark key with yellow \'SHIFT\' text."}, {"section": "top", "row": 0, "col": 1, "rowspan": 1, "colspan": 1, "main": "ALPHA", "shift1": "", "shift2": "", "alpha": "", "key_color": "#2A2D33", "text_color": "#E0453A", "uncertain": false, "note": "ALPHA key: dark key with red \'ALPHA\' text."}, {"section": "top", "row": 0, "col": 4, "rowspan": 1, "colspan": 1, "main": "MENU", "shift1": "SETUP", "shift2": "", "alpha": "", "key_color": "#2A2D33", "text_color": "#FFFFFF", "uncertain": false, "note": "SHIFT MENU = SETUP (confirmed in manual and Duke ClassWiz notes)."}, {"section": "top", "row": 0, "col": 5, "rowspan": 1, "colspan": 1, "main": "ON", "shift1": "", "shift2": "", "alpha": "", "key_color": "#2A2D33", "text_color": "#FFFFFF", "uncertain": false, "note": "ON is top-right on ClassWiz."}, {"section": "top", "row": 1, "col": 0, "rowspan": 1, "colspan": 1, "main": "OPTN", "shift1": "QR", "shift2": "", "alpha": "", "key_color": "#2A2D33", "text_color": "#FFFFFF", "uncertain": false, "note": "SHIFT OPTN = QR (QR-code) per quick-start-guide keyboard OCR."}, {"section": "top", "row": 1, "col": 1, "rowspan": 1, "colspan": 1, "main": "CALC", "shift1": "SOLVE", "shift2": "", "alpha": "=", "key_color": "#2A2D33", "text_color": "#FFFFFF", "uncertain": false, "note": "ALPHA CALC gives the \'=\' used in equations."}, {"section": "top", "row": 1, "col": 4, "rowspan": 1, "colspan": 1, "main": "∫▫□", "shift1": "d/dx▫", "shift2": "", "alpha": ":", "key_color": "#2A2D33", "text_color": "#FFFFFF", "uncertain": true, "note": "ALPHA \':\' (multi-statement) assumed from ES PLUS, not confirmed for EX."}, {"section": "top", "row": 1, "col": 5, "rowspan": 1, "colspan": 1, "main": "x", "shift1": "∑▫□", "shift2": "", "alpha": "", "key_color": "#2A2D33", "text_color": "#FFFFFF", "uncertain": true, "note": "Dedicated variable-x key. SHIFT legend Σ placed from memory; manual confirms Σ( exists but not its key."}, {"section": "cursor", "row": 0, "col": 1, "rowspan": 1, "colspan": 1, "main": "▲", "shift1": "", "shift2": "", "alpha": "", "key_color": "#3A3E45", "text_color": "#FFFFFF", "uncertain": false, "note": "Round 4-way cursor pad in the centre of the \'top\' block (occupies top-block columns 2-3, both rows)."}, {"section": "cursor", "row": 1, "col": 0, "rowspan": 1, "colspan": 1, "main": "◀", "shift1": "", "shift2": "", "alpha": "", "key_color": "#3A3E45", "text_color": "#FFFFFF", "uncertain": false, "note": "Round 4-way cursor pad in the centre of the \'top\' block (occupies top-block columns 2-3, both rows)."}, {"section": "cursor", "row": 1, "col": 2, "rowspan": 1, "colspan": 1, "main": "▶", "shift1": "", "shift2": "", "alpha": "", "key_color": "#3A3E45", "text_color": "#FFFFFF", "uncertain": false, "note": "Round 4-way cursor pad in the centre of the \'top\' block (occupies top-block columns 2-3, both rows)."}, {"section": "cursor", "row": 2, "col": 1, "rowspan": 1, "colspan": 1, "main": "▼", "shift1": "", "shift2": "", "alpha": "", "key_color": "#3A3E45", "text_color": "#FFFFFF", "uncertain": false, "note": "Round 4-way cursor pad in the centre of the \'top\' block (occupies top-block columns 2-3, both rows)."}, {"section": "function", "row": 0, "col": 0, "rowspan": 1, "colspan": 1, "main": "■/□", "shift1": "■■/□", "shift2": "", "alpha": "", "key_color": "#2A2D33", "text_color": "#FFFFFF", "uncertain": false, "note": "SHIFT = mixed-fraction template."}, {"section": "function", "row": 0, "col": 1, "rowspan": 1, "colspan": 1, "main": "√■", "shift1": "∛■", "shift2": "", "alpha": "", "key_color": "#2A2D33", "text_color": "#FFFFFF", "uncertain": false, "note": "No ALPHA legend (hpmuseum forum)."}, {"section": "function", "row": 0, "col": 2, "rowspan": 1, "colspan": 1, "main": "x²", "shift1": "x³", "shift2": "", "alpha": "", "key_color": "#2A2D33", "text_color": "#FFFFFF", "uncertain": false, "note": "Blue Base-N legend \'DEC\' printed beside x³ (OCR of QSG keyboard)."}, {"section": "function", "row": 0, "col": 3, "rowspan": 1, "colspan": 1, "main": "x^■", "shift1": "■√□", "shift2": "", "alpha": "", "key_color": "#2A2D33", "text_color": "#FFFFFF", "uncertain": false, "note": "Blue Base-N legend \'HEX\'."}, {"section": "function", "row": 0, "col": 4, "rowspan": 1, "colspan": 1, "main": "log▫□", "shift1": "10^■", "shift2": "", "alpha": "", "key_color": "#2A2D33", "text_color": "#FFFFFF", "uncertain": false, "note": "Blue Base-N legend \'BIN\'."}, {"section": "function", "row": 0, "col": 5, "rowspan": 1, "colspan": 1, "main": "ln", "shift1": "e^■", "shift2": "", "alpha": "", "key_color": "#2A2D33", "text_color": "#FFFFFF", "uncertain": false, "note": "Blue Base-N legend \'OCT\'."}, {"section": "function", "row": 1, "col": 0, "rowspan": 1, "colspan": 1, "main": "(−)", "shift1": "log", "shift2": "", "alpha": "A", "key_color": "#2A2D33", "text_color": "#FFFFFF", "uncertain": false, "note": "SHIFT (−) = log( (log with base argument), from QSG keyboard OCR (\'log A\') and manual \'(log)\'. Differs from ES PLUS, where SHIFT (−) = ∠."}, {"section": "function", "row": 1, "col": 1, "rowspan": 1, "colspan": 1, "main": "°\'\\"", "shift1": "FACT", "shift2": "", "alpha": "B", "key_color": "#2A2D33", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "function", "row": 1, "col": 2, "rowspan": 1, "colspan": 1, "main": "x⁻¹", "shift1": "x!", "shift2": "", "alpha": "C", "key_color": "#2A2D33", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "function", "row": 1, "col": 3, "rowspan": 1, "colspan": 1, "main": "sin", "shift1": "sin⁻¹", "shift2": "", "alpha": "D", "key_color": "#2A2D33", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "function", "row": 1, "col": 4, "rowspan": 1, "colspan": 1, "main": "cos", "shift1": "cos⁻¹", "shift2": "", "alpha": "E", "key_color": "#2A2D33", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "function", "row": 1, "col": 5, "rowspan": 1, "colspan": 1, "main": "tan", "shift1": "tan⁻¹", "shift2": "", "alpha": "F", "key_color": "#2A2D33", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "function", "row": 2, "col": 0, "rowspan": 1, "colspan": 1, "main": "STO", "shift1": "RECALL", "shift2": "", "alpha": "", "key_color": "#2A2D33", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "function", "row": 2, "col": 1, "rowspan": 1, "colspan": 1, "main": "ENG", "shift1": "∠", "shift2": "", "alpha": "i", "key_color": "#2A2D33", "text_color": "#FFFFFF", "uncertain": true, "note": "SHIFT ENG = ∠ per Duke ClassWiz notes (\'angle symbol (SHIFT ENG)\'). Imaginary unit i is printed in purple (Complex mode) on/above this key - exact placement (ALPHA vs purple bracket) uncertain."}, {"section": "function", "row": 2, "col": 2, "rowspan": 1, "colspan": 1, "main": "(", "shift1": "Abs", "shift2": "", "alpha": "", "key_color": "#2A2D33", "text_color": "#FFFFFF", "uncertain": true, "note": "SHIFT ( = Abs read from QSG keyboard OCR order \'RECALL Abs ...\'; single source."}, {"section": "function", "row": 2, "col": 3, "rowspan": 1, "colspan": 1, "main": ")", "shift1": ",", "shift2": "", "alpha": "x", "key_color": "#2A2D33", "text_color": "#FFFFFF", "uncertain": true, "note": "SHIFT ) = \',\' confirmed (Casio UK QSG \'SHIFT ) (,)\'). ALPHA legend x assumed from ES PLUS."}, {"section": "function", "row": 2, "col": 4, "rowspan": 1, "colspan": 1, "main": "S⇔D", "shift1": "a b/c⇔d/c", "shift2": "", "alpha": "y", "key_color": "#2A2D33", "text_color": "#FFFFFF", "uncertain": true, "note": "SHIFT legend assumed from ES PLUS; ALPHA y assumed."}, {"section": "function", "row": 2, "col": 5, "rowspan": 1, "colspan": 1, "main": "M+", "shift1": "M−", "shift2": "", "alpha": "M", "key_color": "#2A2D33", "text_color": "#FFFFFF", "uncertain": false, "note": "SHIFT M+ = M−, ALPHA M (QSG OCR \'M- M\')."}, {"section": "numeric", "row": 0, "col": 0, "rowspan": 1, "colspan": 1, "main": "7", "shift1": "CONST", "shift2": "", "alpha": "", "key_color": "#5B6068", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "numeric", "row": 0, "col": 1, "rowspan": 1, "colspan": 1, "main": "8", "shift1": "CONV", "shift2": "", "alpha": "", "key_color": "#5B6068", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "numeric", "row": 0, "col": 2, "rowspan": 1, "colspan": 1, "main": "9", "shift1": "RESET", "shift2": "", "alpha": "", "key_color": "#5B6068", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "numeric", "row": 0, "col": 3, "rowspan": 1, "colspan": 1, "main": "DEL", "shift1": "INS", "shift2": "", "alpha": "UNDO", "key_color": "#2A2D33", "text_color": "#FFFFFF", "uncertain": true, "note": "SHIFT DEL = INS; ALPHA DEL = UNDO read from OCR \'INS UNDO OFF\'."}, {"section": "numeric", "row": 0, "col": 4, "rowspan": 1, "colspan": 1, "main": "AC", "shift1": "OFF", "shift2": "", "alpha": "", "key_color": "#2A2D33", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "numeric", "row": 1, "col": 0, "rowspan": 1, "colspan": 1, "main": "4", "shift1": "", "shift2": "", "alpha": "", "key_color": "#5B6068", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "numeric", "row": 1, "col": 1, "rowspan": 1, "colspan": 1, "main": "5", "shift1": "", "shift2": "", "alpha": "", "key_color": "#5B6068", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "numeric", "row": 1, "col": 2, "rowspan": 1, "colspan": 1, "main": "6", "shift1": "", "shift2": "", "alpha": "", "key_color": "#5B6068", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "numeric", "row": 1, "col": 3, "rowspan": 1, "colspan": 1, "main": "×", "shift1": "nPr", "shift2": "", "alpha": "", "key_color": "#2A2D33", "text_color": "#FFFFFF", "uncertain": true, "note": "nPr/nCr exist (manual) and are SHIFT ×/÷ on ES PLUS, but the EX quick-start OCR shows no legend line above this row."}, {"section": "numeric", "row": 1, "col": 4, "rowspan": 1, "colspan": 1, "main": "÷", "shift1": "nCr", "shift2": "", "alpha": "", "key_color": "#2A2D33", "text_color": "#FFFFFF", "uncertain": true, "note": "See ×."}, {"section": "numeric", "row": 2, "col": 0, "rowspan": 1, "colspan": 1, "main": "1", "shift1": "", "shift2": "", "alpha": "", "key_color": "#5B6068", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "numeric", "row": 2, "col": 1, "rowspan": 1, "colspan": 1, "main": "2", "shift1": "", "shift2": "", "alpha": "", "key_color": "#5B6068", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "numeric", "row": 2, "col": 2, "rowspan": 1, "colspan": 1, "main": "3", "shift1": "", "shift2": "", "alpha": "", "key_color": "#5B6068", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "numeric", "row": 2, "col": 3, "rowspan": 1, "colspan": 1, "main": "+", "shift1": "Pol", "shift2": "", "alpha": "", "key_color": "#2A2D33", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "numeric", "row": 2, "col": 4, "rowspan": 1, "colspan": 1, "main": "−", "shift1": "Rec", "shift2": "", "alpha": "", "key_color": "#2A2D33", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "numeric", "row": 3, "col": 0, "rowspan": 1, "colspan": 1, "main": "0", "shift1": "Rnd", "shift2": "", "alpha": "", "key_color": "#5B6068", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "numeric", "row": 3, "col": 1, "rowspan": 1, "colspan": 1, "main": ".", "shift1": "Ran#", "shift2": "", "alpha": "RanInt", "key_color": "#5B6068", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "numeric", "row": 3, "col": 2, "rowspan": 1, "colspan": 1, "main": "×10ˣ", "shift1": "π", "shift2": "", "alpha": "e", "key_color": "#2A2D33", "text_color": "#FFFFFF", "uncertain": false, "note": "SHIFT ×10ˣ = π confirmed (Casio UK QSG). ALPHA = e."}, {"section": "numeric", "row": 3, "col": 3, "rowspan": 1, "colspan": 1, "main": "Ans", "shift1": "%", "shift2": "", "alpha": "PreAns", "key_color": "#2A2D33", "text_color": "#FFFFFF", "uncertain": true, "note": "SHIFT Ans = % read from QSG OCR (π % ≈ sequence). ALPHA legend PreAns from memory; not found in manual."}, {"section": "numeric", "row": 3, "col": 4, "rowspan": 1, "colspan": 1, "main": "=", "shift1": "≈", "shift2": "", "alpha": "", "key_color": "#2A2D33", "text_color": "#FFFFFF", "uncertain": false, "note": "SHIFT = gives decimal result (≈)."}], "notes": "50 keys counted (8 top + 4 cursor directions + 18 function + 20 numeric). Casio manuals and quick-start guides draw keys as font glyphs/images, so legends were assembled from: the EN user\'s guide list of alternate-function names, an OCR of the quick-start-guide keyboard figure (iFixit copy), Casio UK QSG text, and Duke\'s ClassWiz notes; remaining items carried over from the ES PLUS layout are flagged uncertain. Digit keys 1-6 have no shifted legends on the EX (Matrix/Vector/Stat etc. moved to MENU apps and OPTN). Colours approximate.", "fx-115ES_PLUS_differences": {"uncertain": true, "summary": "The US fx-115ES PLUS is the previous (ES PLUS) generation and its keyboard differs materially: MODE (SHIFT=SETUP) instead of MENU; no OPTN key and no dedicated x key; row under SHIFT/ALPHA is CALC (SHIFT SOLVE, ALPHA =), ∫ (SHIFT d/dx, ALPHA :) on the left and x⁻¹ (SHIFT x!), log▫□ (SHIFT Σ) on the right of the cursor pad; next row ■/□, √■, x², x^■, log, ln; then (−) (SHIFT ∠, ALPHA A), °\'\\" (SHIFT ←, ALPHA B), hyp (SHIFT Abs, ALPHA C), sin/cos/tan (ALPHA D/E/F); then RCL (SHIFT STO), ENG (SHIFT ←, purple i), ( (SHIFT %), ) (SHIFT \',\', ALPHA X), S⇔D (ALPHA Y), M+ (SHIFT M−, ALPHA M). Number block: SHIFT 1 STAT, 2 CMPLX, 3 DISTR, 4 MATRIX, 5 VECTOR, 7 CONST, 8 CONV, 9 CLR, × nPr, ÷ nCr, + Pol, − Rec, 0 Rnd, . Ran# (ALPHA RanInt#), ×10ˣ π (ALPHA e), Ans DRG▸, DEL INS, AC OFF. Manual confirms (STO), (CLR), (DRG▸), (←), (Abs), (FACT), (∠) as shifted functions; exact key placement is from memory of the ES PLUS keyboard and should be checked against a photo before use."}}}')

# ---------------------------------------------------------------------------
# Calculator faceplates. Each model's keys sit at the same row/column as on
# the physical device, with the shifted legends printed where the device
# prints them. Pressing a key sends the legend that is active (main, first
# shift, second shift or alpha) to the simulated engine; legends that the
# engine does not cover (programming, graphing menus, cursor editing) show
# "not simulated" instead of a wrong result.
# ---------------------------------------------------------------------------
import re as _re_ui

# legend (normalised) -> engine key.  Normalisation: strip spaces, unify minus
# signs and superscripts.  RPN and algebraic legends share most names.
_LEG = {
    # digits / entry
    **{d: d for d in "0123456789"}, ".": ".", "·": ".",
    "enter": "ENTER", "=": "EQ", "exe": "EXE", "enter↑": "ENTER", "e": "E",
    "chs": "CHS", "+/-": "CHS", "(-)": "NEG", "eex": "EEX", "×10ˣ": "EEX", "x10ˣ": "EEX", "ee": "EEX",
    "×10^x": "EEX", "x10^x": "EEX", "×10x": "EEX",
    "←": "BSP", "bsp": "BSP", "del": "DEL", "clx": "CLX", "clear": "CLEAR", "ac": "AC", "c": "CLEAR",
    "clstk": "CLSTK", "clrstk": "CLSTK", "clear stk": "CLSTK",
    # arithmetic
    "+": "ADD", "-": "SUB", "×": "MUL", "*": "MUL", "÷": "DIV", "/": "DIV",
    "(": "LP", ")": "RP", ",": "COMMA",
    # powers and roots
    "yˣ": "YX", "y^x": "YX", "^": "YX", "xʸ": "YX", "x^y": "YX", "x■": "YX", "x^□": "YX", "□^□": "YX",
    "√x": "SQRT", "√": "SQRT", "√(": "SQRT", "√■": "SQRT", "√□": "SQRT",
    "x²": "SQ", "x^2": "SQ", "x■²": "SQ", "□²": "SQ",
    "1/x": "INV", "x⁻¹": "INV", "x^-1": "INV", "□⁻¹": "INV",
    "ˣ√y": "XROOTY", "x√y": "XROOTY", "ˣ√": "XROOTY", "x√": "XROOTY", "ˣ√■": "XROOTY", "■√□": "XROOTY",
    "□√□": "XROOTY", "ˣ√(": "XROOTY", "∛": "CBRT", "∛■": "CBRT", "∛□": "CBRT", "3√": "CBRT",
    # exp / log
    "eˣ": "EXP", "e^x": "EXP", "e■": "EXP", "e^□": "EXP", "ln": "LN", "10ˣ": "10X", "10^x": "10X",
    "10■": "10X", "10^□": "10X", "log": "LOG",
    # trig
    "sin": "SIN", "cos": "COS", "tan": "TAN", "sin⁻¹": "ASIN", "cos⁻¹": "ACOS", "tan⁻¹": "ATAN",
    "sin^-1": "ASIN", "cos^-1": "ACOS", "tan^-1": "ATAN", "asin": "ASIN", "acos": "ACOS", "atan": "ATAN",
    "hyp": "HYP", "hyp⁻¹": "AHYP", "hyp^-1": "AHYP",
    "deg": "DEG", "rad": "RAD", "grd": "GRD", "→rad": "TORAD", "→deg": "TODEG", "→r": "TOR", "→p": "TOP",
    "→h.ms": "HMS", "→h": "HR", "→hms": "HMS", "→hr": "HR",
    # misc functions
    "%": "PCT", "δ%": "DPCT", "∆%": "DPCT", "abs": "ABS", "|x|": "ABS", "abs(": "ABS", "int": "INT", "ip": "INT",
    "frac": "FRAC", "fp": "FRAC", "x!": "FACT", "!": "FACT", "rnd": "RND",
    "py,x": "PERM", "cy,x": "COMB", "npr": "PERM", "ncr": "COMB",
    "π": "PI", "pi": "PI", "ans": "ANS",
    # stack
    "x≷y": "SWAP", "x<>y": "SWAP", "x⇄y": "SWAP", "r↓": "RDN", "r↑": "RUP", "lstx": "LASTX", "lastx": "LASTX",
    # statistics
    "σ+": "SIGMAP", "σ-": "SIGMAM", "x̄": "MEAN", "s": "SDEV", "l.r.": "LR", "ŷ,r": "YHATR", "clσ": "CLSIGMA",
    "clear σ": "CLSIGMA",
    # display / memory
    "fix": "FIX", "sci": "SCI", "eng": "ENG", "all": "ALL", "sto": "STO", "rcl": "RCL", "sto→": "STO",
    "sto▸": "STO", "sto►": "STO",
    # model-specific spellings (from calc_layouts.json)
    "E": "EEX", "×10ⁿ": "EEX", "x10ⁿ": "EEX", "%chg": "DPCT", "x̄,ȳ": "MEAN", "s,σ": "SDEV", "hms→": "HR", "→hms": "HMS",
    "→°f": "TOF", "→°c": "TOC", "→lb": "TOLB", "→kg": "TOKG", "→mile": "TOMILE", "→km": "TOKM", "→in": "TOIN",
    "→cm": "TOCM", "→gal": "TOGAL", "→l": "TOL", "x^■": "YX", "10^■": "10X", "e^■": "EXP", "x³": "CUBE",
    "■/□": "DIV", "entry": "ENTRY", "delete": "DEL", "ans": "ANS", "√": "SQRT", "ip": "INT", "fp": "FRAC",
}
_SHIFT_NAMES = {"◀(leftshift)": "shift1", "▶(rightshift)": "shift2","f": "shift1", "g": "shift2", "2nd": "shift1", "shift": "shift1", "alpha": "alpha",
                "◢": "shift1", "◣": "shift2", "left-shift": "shift1", "right-shift": "shift2",
                "leftshift": "shift1", "rightshift": "shift2", "⬏": "shift1", "⬎": "shift2"}
_ALG_VARS = set("ABCDEFGHIJKLMNOPQRSTUVWXYZθxyzt")


def _norm(s):
    s = (s or "").strip()
    s = s.replace("−", "-").replace("–", "-").replace("Σ", "σ").replace("x̅", "x̄")
    s = _re_ui.sub(r"\s+", "", s)
    return s.lower() if len(s) > 1 else s


def legend_to_key(legend, rpn):
    if not legend:
        return None
    n = _norm(legend)
    if n in _LEG:
        k = _LEG[n]
        if rpn and k == "NEG":
            return "CHS"
        if not rpn and k == "CHS":
            return "NEG"
        if not rpn and k == "CLX":
            return "CLEAR"
        if rpn and k in ("CLEAR", "AC"):
            return "CLX"
        if rpn and k in ("CUBE", "ENTRY", "DEL", "ANS"):
            return {"DEL": "BSP"}.get(k)
        if not rpn and k.startswith("TO") and k not in ("TOP", "TOR"):
            return None
        if rpn and k == "EQ":
            return None
        if rpn and k == "LP":
            return None
        return k
    if not rpn and len(legend.strip()) == 1 and legend.strip() in _ALG_VARS:
        return "VAR_" + legend.strip()
    return None


class Faceplate:
    """UI-independent model of one calculator: key grid + shift state + engine."""

    RPN_MODELS = ("HP-15C", "HP 35s")

    def __init__(self, model, layout):
        self.model, self.layout = model, layout
        self.rpn = model in self.RPN_MODELS
        self.engine = RPNEngine(model) if self.rpn else AlgEngine(model, implicit_mult=True)
        if model.startswith("Casio"):
            self.engine.disp_mode = "ALL"
        self.state = None            # None | 'shift1' | 'shift2' | 'alpha'
        self.state_used = None
        self.log = []
        self.keys = layout["keys"]

    def shift_of(self, k):
        for lab in (k.get("main"), k.get("note")):
            n = _norm(lab)
            if n in _SHIFT_NAMES:
                return _SHIFT_NAMES[n]
        return None

    def active_legend(self, k):
        if self.state and k.get(self.state):
            return k[self.state]
        return k.get("main", "")

    def press(self, k):
        """Press key dict k. Returns the legend that acted."""
        sh = self.shift_of(k)
        if sh:
            self.state = None if self.state == sh else sh
            self.log.append(k.get("main"))
            return k.get("main")
        legend = self.active_legend(k)
        self.state_used = self.state if (self.state and k.get(self.state)) else None
        self.log.append(("%s " % {"shift1": self.layout.get("_s1", "2nd"), "shift2": self.layout.get("_s2", "g"),
                                  "alpha": "ALPHA"}[self.state] if self.state else "") + (k.get("main") or ""))
        self.state = None
        n = _norm(legend)
        if self.state_used == "alpha" and not self.rpn and len(legend.strip()) == 1 and legend.strip().isalpha():
            self.engine.press("VAR_" + legend.strip()); return legend
        if n in ("on", "on/c", "on/off", "c/on"):
            self.engine.__init__(self.model) if self.rpn else self.engine.__init__(self.model, True)
            return legend
        key = legend_to_key(legend, self.rpn)
        if key is None:
            self.engine.message = "%s: not simulated" % legend
            return legend
        self.engine.press(key)
        return legend

    def press_legend(self, legend, state=None):
        """Find the key whose legend (in the given state) matches and press it (used by tests)."""
        for k in self.keys:
            if (k.get(state) if state else k.get("main")) == legend:
                if state:
                    sk = next(x for x in self.keys if self.shift_of(x) == state)
                    self.press(sk)
                return self.press(k)
        raise KeyError("%s has no key %r (%s)" % (self.model, legend, state or "main"))

    def screen(self):
        e = self.engine
        if self.rpn:
            st = e.stack()
            return dict(main=e.display(), stack=[("T", st["T"]), ("Z", st["Z"]), ("Y", st["Y"])],
                        ann=" ".join(x for x in (self.state and {"shift1": "f", "shift2": "g"}[self.state],
                                                 e.angle if e.angle != "DEG" else "") if x))
        return dict(main=e.display(), entry=e.entry_text(), ann=" ".join(
            x for x in ({"shift1": "2nd", "shift2": "", "alpha": "ALPHA"}.get(self.state or "", ""), e.angle) if x))

    def coverage(self):
        """(mapped, total, unmapped legends) over every legend on the faceplate."""
        tot, ok, miss = 0, 0, []
        for k in self.keys:
            if self.shift_of(k):
                continue
            for st in ("main", "shift1", "shift2", "alpha"):
                lab = k.get(st)
                if not lab or (st == "alpha" and not self.rpn and False):
                    continue
                tot += 1
                if legend_to_key(lab, self.rpn) or _norm(lab) in ("on", "on/c", "on/off", "c/on"):
                    ok += 1
                else:
                    miss.append(lab)
        return ok, tot, miss


# ---------------------------------------------------------------------------
# How the JSON sections sit relative to each other on the physical device:
# each block is one grid; a section is placed at (row offset, col offset, col remap).
ARRANGE = {
    "HP 35s": [[("top", 0, 0), ("cursor", 0, 4)], [("upper", 0, 0)], [("lower", 0, 0)]],
    "TI-30XS MultiView": [[("top", 0, 0), ("arrows", 0, 3)], [("main", 0, 0)]],
    "TI-84 Plus CE": [[("top_row", 0, 0)], [("upper_left", 0, 0), ("arrows", 0, 3)], [("main", 0, 0)]],
    "Casio fx-991EX": [[("top", 0, 0, {4: 5, 5: 6}), ("cursor", 0, 2)], [("function", 0, 0)], [("numeric", 0, 0)]],
}


def blocks(model, layout):
    """[(rows, cols, [(key, row, col)])] — one grid per block, keys at device positions."""
    secs = {s["name"]: s for s in layout.get("sections", [])}
    if not secs:
        g = layout["grid"]
        return [(g["rows"], g["cols"], [(k, k["row"], k["col"]) for k in layout["keys"]])]
    arr = ARRANGE.get(model) or [[(n, 0, 0)] for n in secs]
    out = []
    for blk in arr:
        items, R, Cn = [], 0, 0
        for ent in blk:
            name, ro, co = ent[:3]
            remap = ent[3] if len(ent) > 3 else {}
            for k in layout["keys"]:
                if k.get("section") == name:
                    c = remap.get(k["col"], k["col"]) + co
                    r = k["row"] + ro
                    items.append((k, r, c))
                    R = max(R, r + k.get("rowspan", 1)); Cn = max(Cn, c + k.get("colspan", 1))
        out.append((R, Cn, items))
    return out


CALC_LAYOUTS = globals().get("CALC_LAYOUTS", {})
_PLATES = {}
# Course names for the simulated calculators; each follows the key layout of the model in the key
# (trademarks of their owners, named only to identify the layout).
CALC_NAMES = {"HP-15C": "SBJ-15C", "HP 35s": "SBJ-35S", "TI-30XS MultiView": "SBJ-30XS", "TI-84 Plus CE": "SBJ-84+",
              "Casio fx-991EX": "SBJ-991X"}


def calc_name(model):
    return CALC_NAMES.get(model, model)


def _key_html(k, model, layout):
    sk = layout.get("shift_keys", {})
    pick = lambda names, d: next((sk[n]["color"] for n in names if n in sk), d)
    s1c = pick(("f", "left_shift", "2nd", "SHIFT"), "#d4a017")
    s2c = pick(("g", "right_shift"), "#3b82c4")
    ac = pick(("alpha", "ALPHA"), "#16a34a")
    top = []
    if k.get("shift1"):
        top.append('<span style="color:%s">%s</span>' % (s1c, k["shift1"]))
    if k.get("alpha"):
        top.append('<span style="color:%s;float:right">%s</span>' % (ac, k["alpha"]))
    bottom = ('<span style="color:%s">%s</span>' % (s2c, k["shift2"])) if k.get("shift2") else ""
    return ("<div style='font:9px/11px sans-serif;height:11px;white-space:nowrap;overflow:hidden'>%s</div>" % " ".join(top),
            "<div style='font:9px/11px sans-serif;height:11px;white-space:nowrap;overflow:hidden;text-align:center'>%s</div>" % bottom)


def calculator_widget(model):
    import ipywidgets as W
    layout = CALC_LAYOUTS[model]
    fp = _PLATES.get(model) or Faceplate(model, layout)
    _PLATES[model] = fp
    lcd = W.HTML()
    keylog = W.HTML()

    def refresh():
        s = fp.screen()
        if fp.rpn:
            stack = "".join("<div style='color:#6b7280;font-size:11px'>%s: %s</div>" % (n, fmt_number(v, "ALL", 10))
                            for n, v in s["stack"])
            body = "%s<div style='font-size:22px;text-align:right'>%s</div>" % (stack, s["main"])
        else:
            body = ("<div style='font-size:15px;min-height:20px;word-break:break-all'>%s</div>"
                    "<div style='font-size:22px;text-align:right'>%s</div>" % (s["entry"] or "&nbsp;", s["main"] or "&nbsp;"))
        lcd.value = ("<div style='background:#c9d6b8;color:#111;font-family:monospace;padding:6px 10px;border-radius:6px;"
                     "width:300px;border:2px solid #555'><div style='font-size:10px;height:12px'>%s</div>%s</div>"
                     % (s["ann"], body))
        keylog.value = "<span style='font:11px monospace;color:#555'>Keys: %s</span>" % " ".join(
            str(x) for x in fp.log[-24:])

    boxes = []
    for R, Cn, items in blocks(model, layout):
        children = []
        for k, r, c in items:
            top, bot = _key_html(k, model, layout)
            b = W.Button(description=k.get("main", ""), layout=W.Layout(width="auto", height="30px", padding="0"),
                         tooltip="; ".join("%s: %s" % (a, k[a]) for a in ("shift1", "shift2", "alpha") if k.get(a)))
            b.style.button_color = k.get("key_color", "#2b2b2b")
            b.style.text_color = k.get("text_color", "#ffffff")
            b.style.font_weight = "bold"
            if k.get("uncertain"):
                b.layout.border = "2px dashed #f59e0b"
                b.tooltip = (b.tooltip + "; " if b.tooltip else "") + "position/legend not yet confirmed: " + (k.get("note") or "")

            def on(_b, k=k):
                fp.press(k)
                refresh()
            b.on_click(on)
            cell = W.VBox([W.HTML(top), b, W.HTML(bot)], layout=W.Layout(
                grid_row="%d / span %d" % (r + 1, k.get("rowspan", 1)),
                grid_column="%d / span %d" % (c + 1, k.get("colspan", 1)), margin="0"))
            children.append(cell)
        boxes.append(W.GridBox(children, layout=W.Layout(
            grid_template_columns="repeat(%d, 58px)" % Cn, grid_gap="2px 4px", width="max-content", margin="0 0 8px 0")))
    refresh()
    body = layout.get("body_color", "#3a3a3a")
    name = W.HTML("<div style='font:800 14px system-ui,sans-serif;letter-spacing:.1em'>%s</div>"
                  "<div style='font-size:11px;color:#555'>keys as on the %s</div>" % (calc_name(model), model))
    return W.VBox([name, lcd] + boxes + [keylog], layout=W.Layout(
        padding="10px", border="1px solid #999", width="max-content")), fp


def open_calculator(model=None):
    """Model selector + faceplate. Engines keep their state between questions."""
    import ipywidgets as W
    from IPython.display import display
    models = [m for m in ("HP-15C", "HP 35s", "TI-30XS MultiView", "TI-84 Plus CE", "Casio fx-991EX") if m in CALC_LAYOUTS]
    pick = W.Dropdown(options=[(calc_name(m), m) for m in models], value=model or globals().get("_LAST_CALC", models[0]),
                      description="Calculator")
    holder = W.Output()
    help_ = W.HTML("<small>Shifted functions: press the coloured shift key, then the key under the legend. "
                   "Grey legends that are not simulated (programming, graphing, cursor editing) say so instead of "
                   "giving a wrong answer.</small>")

    def show(_=None):
        globals()["_LAST_CALC"] = pick.value
        holder.clear_output()
        with holder:
            w, _ = calculator_widget(pick.value)
            display(w)
    pick.observe(show, names="value")
    display(W.VBox([pick, help_, holder]))
    show()


# ---------------------------------------------------------------------------
# Notebook glue: lab-report hooks, per-cell section labels, calculation cards
# and the end-of-notebook report panel.
# ---------------------------------------------------------------------------
import matplotlib.pyplot as _plt_glue

REPORT = None


def _start_report(course, session, title):
    global REPORT
    REPORT = LabReport(course, session, title)
    _orig_show = _plt_glue.show

    def _show(*a, **k):                      # every figure shown is copied into the report
        try:
            for num in _plt_glue.get_fignums():
                f = _plt_glue.figure(num)
                if not getattr(f, "_in_report", False):
                    REPORT.add_figure(f)
                    f._in_report = True
        except Exception:
            pass
        return _orig_show(*a, **k)
    _plt_glue.show = _show
    return REPORT


def _cell(label):
    """Called at the top of each hidden code cell: sets the report section."""
    if REPORT is not None:
        REPORT.set_section(label)


def card(card_id, calculator=True):
    """Interactive solver for one calculation card (+ the calculator panel)."""
    c = CARDS[card_id]
    opener = None
    if calculator:
        try:
            opener = lambda: open_calculator()      # defined by calc_ui
        except NameError:
            opener = None
    return solver_widget(c, REPORT, opener)


def report_panel():
    """Student name, reflection notes and download buttons for the lab report."""
    try:
        import ipywidgets as W
        from IPython.display import display, Markdown, clear_output
    except Exception:
        print(REPORT.counts() if REPORT else "no report")
        return
    name = W.Text(description="Your name", placeholder="as in Canvas", layout=W.Layout(width="420px"))
    notes = W.Textarea(description="Discussion", placeholder="What do your calculations show? What would you check next?",
                       layout=W.Layout(width="95%", height="120px"))
    btn = W.Button(description="Build and download report (.docx + .html)", button_style="primary", icon="download",
                   layout=W.Layout(width="360px"))
    out = W.Output()

    def go(_):
        with out:
            clear_output()
            REPORT.student = name.value.strip()
            REPORT.entries = [e for e in REPORT.entries if not (e["kind"] == "note" and e["title"] == "Discussion")]
            if notes.value.strip():
                REPORT.section = "Discussion"
                REPORT.add_note("Discussion", notes.value.strip())
            c = REPORT.counts()
            display(Markdown("Collected **%d** figures, **%d** calculations and **%d** self-check answers." % (
                c.get("figure", 0), c.get("calc", 0), c.get("answer", 0))))
            print(REPORT.download())
    btn.on_click(go)
    display(W.VBox([name, notes, btn, out]))


def _log_answers(gb):
    """Wrap the gradebook's answer() so first answers (not the readiness check) go into the report."""
    orig = gb.answer

    def _resp():
        r = getattr(gb, "resp", None)
        return r if r is not None else getattr(gb, "responses", {})

    def answer(qid, choice, *a, **k):
        first = qid not in _resp()
        r = orig(qid, choice, *a, **k)
        try:
            if first and qid in _resp() and REPORT is not None:
                it = gb.items[qid]
                if getattr(it, "section", "") != "precheck":
                    ch = _resp()[qid]
                    REPORT.add_answer(qid, getattr(it, "stem", qid), ch, ch == it.key)
        except Exception:
            pass
        return r
    gb.answer = answer

# ===== END kit =====

# ===== lab helpers =====
# ---------------------------------------------------------------------------
# PHS 312 computational-lab helpers (original code for this course).
# Report capture: every matplotlib figure shown is copied into the lab report
# by the kit's plt.show hook; RDKit drawings and tables are turned into
# matplotlib figures here so that they reach the .docx/.html report as well.
# Heavy libraries (RDKit, meeko, py3Dmol) are imported lazily inside the
# functions that need them, so this block runs without them.
# ---------------------------------------------------------------------------
import os, io, re, sys, json, math, time, shutil, platform, datetime, subprocess
import numpy as np
import matplotlib
import matplotlib.pyplot as plt

try:
    from IPython.display import display, Markdown, HTML
except Exception:                                   # plain Python (validator)
    def display(*a, **k):
        for x in a:
            print(x)

    def Markdown(s):
        return s

    def HTML(s):
        return s

LAB = {"params": {}, "checks": [], "files": []}
R_KCAL = 1.987204e-3          # gas constant, kcal mol^-1 K^-1 (CODATA 2018 R / 4184)


def _md(s):
    display(Markdown(s))


def lab_step(label):
    """Start a numbered lab step: sets the report section and prints a banner."""
    _cell(label)
    print("=" * 78)
    print(label)
    print("=" * 78)


def record_params(title, **params):
    """Store run parameters (they become a table in the report)."""
    LAB["params"].update(params)
    rows = [[k, str(v)] for k, v in params.items()]
    report_table(rows, title, columns=["Parameter", "Value"], show=True)


def checkpoint(name, value, lo=None, hi=None, unit="", why="", expect=None):
    """Compare a result with the expected range; print PASS/CHECK and log it.
    expect: optional boolean condition result used instead of a numeric range."""
    if expect is not None:
        ok = bool(expect)
        rng = "condition"
    else:
        ok = (lo is None or value >= lo) and (hi is None or value <= hi)
        rng = "%s to %s %s" % ("-inf" if lo is None else _f(lo), "+inf" if hi is None else _f(hi), unit)
    shown = _f(value) if isinstance(value, (int, float, np.floating, np.integer)) else str(value)
    status = "PASS" if ok else "CHECK"
    print("[%s] %s = %s %s   (expected %s)" % (status, name, shown, unit, rng))
    if why and not ok:
        print("       What to look at: " + why)
    LAB["checks"].append((name, shown + " " + unit, rng, status))
    if REPORT is not None:
        REPORT.add_note("Checkpoint: " + name, "%s %s (expected %s) -> %s%s" % (
            shown, unit, rng, status, ("" if ok else ". " + why)))
    return ok


def _f(x):
    try:
        x = float(x)
    except Exception:
        return str(x)
    if x == 0:
        return "0"
    if abs(x) >= 1e4 or abs(x) < 1e-3:
        return "%.3e" % x
    return "%.4g" % x


def report_table(rows, title, columns=None, show=True, max_rows=40, note=None):
    """Display a table (list of rows or a pandas DataFrame) and add it to the
    lab report as a figure (so it appears in both .docx and .html)."""
    try:
        import pandas as pd
        if isinstance(rows, pd.DataFrame):
            df = rows
        else:
            df = pd.DataFrame(rows, columns=columns)
        if show:
            display(df)
        cols = [str(c) for c in df.columns]
        cells = [[_f(v) if isinstance(v, (float, np.floating)) else str(v) for v in r] for r in df.values.tolist()]
    except ImportError:                              # pandas missing: plain lists
        cols = list(columns or [])
        cells = [[_f(v) if isinstance(v, float) else str(v) for v in r] for r in rows]
        if show:
            print(title)
            for r in [cols] + cells:
                print("  ".join(r))
    extra = ""
    if len(cells) > max_rows:
        extra = " (first %d of %d rows)" % (max_rows, len(cells))
        cells = cells[:max_rows]
    ncol = max(1, len(cols))
    widths = [max([len(c)] + [len(r[j]) for r in cells]) for j, c in enumerate(cols)] if cols else [10]
    tot = float(sum(widths)) or 1.0
    fig_w = min(11.0, max(5.0, 0.11 * tot + 1.0))
    fig_h = 0.5 + 0.28 * (len(cells) + 1)
    fig, ax = plt.subplots(figsize=(fig_w, fig_h))
    ax.axis("off")
    tb = ax.table(cellText=cells or [[""] * ncol], colLabels=cols, loc="center", cellLoc="left",
                  colWidths=[w / tot for w in widths])
    tb.auto_set_font_size(False)
    tb.set_fontsize(8)
    tb.scale(1, 1.25)
    for (r, c), cell in tb.get_celld().items():
        cell.set_edgecolor("#cbd5e1")
        if r == 0:
            cell.set_facecolor("#eef4fb")
            cell.set_text_props(weight="bold")
    ax.set_title(title + extra, fontsize=10, loc="left")
    if REPORT is not None:
        REPORT.add_figure(fig, "Table: " + title + extra)
        if note:
            REPORT.add_note(title, note)
    plt.close(fig)
    return cells


def show_pil(img, title, width=8.0):
    """Show a PIL image (e.g. an RDKit drawing) as a matplotlib figure so the
    plt.show hook copies it into the lab report."""
    arr = np.asarray(img)
    h, w = arr.shape[0], arr.shape[1]
    fig, ax = plt.subplots(figsize=(width, max(1.5, width * h / float(w))))
    ax.imshow(arr)
    ax.axis("off")
    ax.set_title(title, fontsize=10)
    plt.show()


def draw_mols(mols, legends=None, title="Structures", per_row=4, size=(280, 220)):
    """Grid drawing of RDKit molecules -> report figure (Cairo PNG -> PIL)."""
    from rdkit.Chem import Draw
    from rdkit.Chem.Draw import rdMolDraw2D
    from PIL import Image
    mols = list(mols)
    legends = list(legends or [""] * len(mols))
    try:
        img = Draw.MolsToGridImage(mols, molsPerRow=per_row, subImgSize=size, legends=legends, returnPNG=False)
        if not hasattr(img, "size"):                       # some builds return PNG bytes
            img = Image.open(io.BytesIO(img))
    except Exception as e:                                  # no Cairo: text fallback
        print("(structure drawing unavailable in this build: %s)" % e)
        from rdkit import Chem
        fig, ax = plt.subplots(figsize=(8, 0.4 * len(mols) + 0.6))
        ax.axis("off")
        for k, (m, l) in enumerate(zip(mols, legends)):
            ax.text(0, 1 - (k + 1) / (len(mols) + 1.0), "%s   %s" % (l, Chem.MolToSmiles(m)), fontsize=8, family="monospace")
        ax.set_title(title, fontsize=10)
        plt.show()
        return
    show_pil(img, title, width=min(11.0, 2.6 * min(per_row, len(mols))))


def software_versions(extra=None):
    """Collect software versions for the report (a reporting-standard item)."""
    v = {"Python": platform.python_version(), "Platform": platform.platform(terse=True)}
    for mod in ("numpy", "matplotlib", "pandas", "rdkit", "meeko", "py3Dmol", "gemmi", "scipy"):
        try:
            m = __import__(mod)
            v[mod] = getattr(m, "__version__", "installed")
        except Exception:
            pass
    if extra:
        v.update(extra)
    LAB["versions"] = v
    report_table([[k, str(x)] for k, x in v.items()], "Software versions", columns=["Software", "Version"])
    return v


def save_text(path, text):
    with open(path, "w") as fh:
        fh.write(text)
    LAB["files"].append(path)
    return path


def download_file(url, path):
    """Download once (Colab has internet); reuse the local copy afterwards."""
    if not os.path.exists(path) or os.path.getsize(path) == 0:
        import urllib.request
        urllib.request.urlretrieve(url, path)
    LAB["files"].append(path)
    return path


def boltzmann(energies_kcal, T=298.15):
    """Boltzmann populations from relative energies (kcal/mol)."""
    e = np.asarray(energies_kcal, float)
    w = np.exp(-(e - e.min()) / (R_KCAL * T))
    return w / w.sum()


def report_preview():
    """Show what the lab report has collected so far (figures, calculations, checkpoints). This is the
    object that lets a student assemble and inspect the report before the final export."""
    c = REPORT.counts() if REPORT is not None else {}
    rows = [["Figures and tables", c.get("figure", 0)], ["Calculations (cards solved)", c.get("calc", 0)],
            ["Checkpoints and notes", c.get("note", 0)], ["Self-check answers", c.get("answer", 0)]]
    report_table(rows, "Report contents collected so far", columns=["Item", "Count"])
    return c


def reporting_check(required):
    """Is everything a reviewer needs to reproduce this work recorded? Compares the parameters
    logged with record_params() (and the software versions) with the lab's required list."""
    have = set(LAB["params"])
    missing = [k for k in required if k not in have]
    rows = [[k, str(LAB["params"].get(k, "NOT RECORDED"))] for k in required]
    rows.append(["software versions", "recorded" if LAB.get("versions") else "NOT RECORDED"])
    report_table(rows, "Reporting checklist - items a reviewer needs", columns=["Item", "Value recorded"])
    ok = not missing and bool(LAB.get("versions"))
    checkpoint("Reporting items recorded", len(required) - len(missing), len(required), len(required),
               why="Run the steps that record: " + ", ".join(missing) if missing else "Run Step 1 to record software versions.")
    return ok


def lab_summary():
    """Print and log a summary of every checkpoint."""
    rows = [[n, v, r, s] for n, v, r, s in LAB["checks"]]
    if rows:
        report_table(rows, "Checkpoint summary", columns=["Checkpoint", "Your value", "Expected", "Status"])
    else:
        print("No checkpoints recorded yet - run the lab steps first.")


# ===== calculation cards =====

import math

# Coefficients of the 3-descriptor Hansch model fitted in this lab to the ten
# 3-substituted 4-anilino-6,7-dimethoxyquinazolines (ChEMBL 27 via TeachOpenCADD
# T001; substituent constants from Hansch, Leo & Taft, Chem. Rev. 1991, 91, 165).
# check_labs.py refits the model with numpy and confirms these values.
HANSCH_COEF = {"a": 2.433, "b": 2.436, "c": -2.070, "k": 7.840}


def _solve_hansch(card, d):
    S = Solution(card, d)
    V = card.V
    a, b, c, k = d["a"], d["b"], d["c"], d["k"]
    pi, sg, mr = d["pi"], d["sigma"], d["MR"]
    t1, t2, t3 = a * pi, b * sg, c * mr / 10.0
    p = t1 + t2 + t3 + k
    ic50_nM = 10 ** (-p) * 1e9
    S.general().given().unit_check("pi, sigma and MR/10 are dimensionless substituent constants (MR in cm^3/mol is "
                                   "scaled by 0.1 as in Hansch's tables), so every term is in pIC50 units; IC50 is "
                                   "converted from mol/L to nM (1 nM = 10^-9 mol/L).")
    S.step("Hydrophobic term", r"%s \times %s = %s \times %s = %s" % (V["a"].tex(), V["pi"].tex(), fmt(a), fmt(pi), fmt(t1)))
    S.step("Electronic term", r"%s \times %s = %s \times %s = %s" % (V["b"].tex(), V["sigma"].tex(), fmt(b), fmt(sg), fmt(t2)))
    S.step("Steric term", r"%s \times \frac{%s}{10} = %s \times %s = %s" % (V["c"].tex(), V["MR"].tex(), fmt(c), fmt(mr / 10.0), fmt(t3)))
    S.step("Sum with intercept", r"pIC_{50} = %s + %s + (%s) + %s = %s" % (fmt(t1), fmt(t2), fmt(t3), fmt(k), fmt(p)))
    S.step("Back to IC50", r"IC_{50} = 10^{-%s}\,\mathrm{mol/L} = %s\,\mathrm{nM}" % (fmt(p), fmt(ic50_nM)))
    S.answer(p, "", "The model predicts pIC50 = %s (IC50 about %s nM) for the 3-ethynyl analogue. Because q^2 for this "
             "model is only about 0.24 and the training set mixes assays, the prediction is a hypothesis to test, "
             "not a reliable estimate." % (fmt(p), fmt(ic50_nM)), label="Answer (predicted pIC50)")
    return S


HANSCH = Card(
    "l29_hansch_prediction", "Predicting potency with a fitted Hansch equation",
    r"pIC_{50} = {{a}}\,{{pi}} + {{b}}\,{{sigma}} + {{c}}\,\frac{{{MR}}}{10} + {{k}}",
    [Var("pi", r"\pi", "hydrophobic constant of the new 3-substituent (C≡CH)", "", "input",
         "Hansch, Leo & Taft, Chem. Rev. 1991, 91, 165", 0.40),
     Var("sigma", r"\sigma_m", "Hammett meta constant of C≡CH", "", "input", "Hansch, Leo & Taft 1991", 0.21),
     Var("MR", r"MR", "molar refractivity of C≡CH", "cm^3/mol", "input", "Hansch, Leo & Taft 1991", 9.55),
     Var("a", r"a", "fitted coefficient of pi", "", "input", "your regression in Step 5 (lab fit)", HANSCH_COEF["a"]),
     Var("b", r"b", "fitted coefficient of sigma", "", "input", "your regression in Step 5 (lab fit)", HANSCH_COEF["b"]),
     Var("c", r"c", "fitted coefficient of MR/10", "", "input", "your regression in Step 5 (lab fit)", HANSCH_COEF["c"]),
     Var("k", r"k", "fitted intercept", "", "input", "your regression in Step 5 (lab fit)", HANSCH_COEF["k"])],
    "Erlotinib carries a 3-ethynylanilino group. Before ordering the 6,7-dimethoxy analogue, the project team uses "
    "your Hansch model ([[a]], [[b]], [[c]], [[k]]) and the substituent constants [[pi]], [[sigma]] and [[MR]]. "
    "What pIC50 does the model predict?",
    _solve_hansch,
    conversions=["pIC50 = -log10(IC50 in mol/L)", "pIC50 9 = 1 nM; pIC50 6 = 1 µM"],
    result_name=r"pIC_{50,pred}", source="Hansch-Fujita analysis (Hansch & Fujita, J. Am. Chem. Soc. 1964, 86, 1616)")


def _solve_q2(card, d):
    S = Solution(card, d)
    V = card.V
    press, ss = d["PRESS"], d["SS"]
    if ss <= 0:
        raise ValueError("total sum of squares must be > 0")
    ratio = press / ss
    q2 = 1 - ratio
    S.general().given().unit_check("PRESS and SS are both sums of squared pIC50 differences, so their ratio is "
                                   "dimensionless.")
    S.step("Ratio", r"\frac{%s}{%s} = \frac{%s}{%s} = %s" % (V["PRESS"].tex(), V["SS"].tex(), fmt(press), fmt(ss), fmt(ratio)))
    S.step("Cross-validated q^2", r"q^2 = 1 - %s = %s" % (fmt(ratio), fmt(q2)))
    S.answer(q2, "", "q^2 = %s. A model is normally considered predictive only when q^2 > 0.5; a value this low means "
             "leave-one-out predictions are barely better than predicting the mean pIC50 for every compound." % fmt(q2),
             label="Answer (q^2)")
    return S


Q2 = Card(
    "l30_q2_loo", "Leave-one-out cross-validated q^2",
    r"q^2 = 1 - \frac{{{PRESS}}}{{{SS}}} = 1 - \frac{\sum (y_i - \hat{y}_{i,(-i)})^2}{\sum (y_i - \bar{y})^2}",
    [Var("PRESS", r"PRESS", "predicted residual sum of squares (each compound predicted by a model fitted without it)",
         "", "input", "LOO loop in Step 6 (pi + sigma + MR model)", 11.93),
     Var("SS", r"SS", "total sum of squares of pIC50 about its mean", "", "input", "Step 6", 15.70)],
    "After fitting the three-descriptor Hansch model, you run leave-one-out cross-validation and obtain [[PRESS]]. "
    "The total sum of squares of the ten pIC50 values is [[SS]]. What is q^2?",
    _solve_q2,
    conversions=["r^2 uses the residual sum of squares of the fitted model; q^2 uses PRESS"],
    result_name=r"q^2", source="Cramer, Bunce & Patterson, Quant. Struct.-Act. Relat. 1988, 7, 18")


# ===== helpers for this lab =====
# ---------------------------------------------------------------------------
# Data set: ten 3-substituted 4-anilino-6,7-dimethoxyquinazolines (one substituent varied,
# same scaffold = congeneric series).
#   IC50 -> pIC50: ChEMBL 27 (target CHEMBL203 = human EGFR; type IC50, relation '=', assay type B;
#   first record per molecule), as frozen in TeachOpenCADD talktorial T001, EGFR_compounds.csv
#   (CC BY 4.0; ChEMBL data CC BY-SA 3.0). Original SAR series: Bridges et al., J. Med. Chem. 1996, 39, 267.
#   NOTE: these records come from different publications and assay protocols, which is itself a
#   limitation students must report.
#   pi (hydrophobic), sigma_m (Hammett meta), MR (molar refractivity, cm3/mol) for the substituent:
#   Hansch, Leo & Taft, Chem. Rev. 1991, 91, 165 (standard substituent-constant tables).
# ---------------------------------------------------------------------------
SERIES = [
    # ChEMBL id,      substituent, pi,    sigma_m, MR,    IC50 (nM), SMILES
    ("CHEMBL94191",  "H",     0.00,  0.00,  1.03,  29.0,   "COc1cc2ncnc(Nc3ccccc3)c2cc1OC"),
    ("CHEMBL94431",  "3-F",   0.14,  0.34,  0.92,  3.8,    "COc1cc2ncnc(Nc3cccc(F)c3)c2cc1OC"),
    ("CHEMBL7917",   "3-Cl",  0.71,  0.37,  6.03,  6.0,    "COc1cc2ncnc(Nc3cccc(Cl)c3)c2cc1OC"),
    ("CHEMBL29197",  "3-Br",  0.86,  0.39,  8.88,  0.025,  "COc1cc2ncnc(Nc3cccc(Br)c3)c2cc1OC"),
    ("CHEMBL96065",  "3-I",   1.12,  0.35,  13.94, 0.89,   "COc1cc2ncnc(Nc3cccc(I)c3)c2cc1OC"),
    ("CHEMBL56802",  "3-CH3", 0.56, -0.07,  5.65,  5.0,    "COc1cc2ncnc(Nc3cccc(C)c3)c2cc1OC"),
    ("CHEMBL316064", "3-CF3", 0.88,  0.43,  5.02,  0.24,   "COc1cc2ncnc(Nc3cccc(C(F)(F)F)c3)c2cc1OC"),
    ("CHEMBL29641",  "3-SCH3", 0.61, 0.15,  13.82, 126.7,  "COc1cc2ncnc(Nc3cccc(SC)c3)c2cc1OC"),
    ("CHEMBL2333998", "3-CN", -0.57, 0.56,  6.33,  412.8,  "COc1cc2ncnc(Nc3cccc(C#N)c3)c2cc1OC"),
    ("CHEMBL2333991", "3-Ph",  1.96, 0.06,  25.36, 126.7,  "COc1cc2ncnc(Nc3cccc(-c4ccccc4)c3)c2cc1OC"),
]
# The 3-ethynyl substituent (as in erlotinib) is held back as an external test compound.
TEST_SUBSTITUENT = dict(name="3-C(triple)CH (ethynyl)", pi=0.40, sigma=0.21, MR=9.55)


def ols(X, y):
    """Ordinary least squares with an intercept column already present in X.
    Returns coefficients, predictions, r2, adjusted r2 and the standard error of the estimate."""
    beta, *_ = np.linalg.lstsq(X, y, rcond=None)
    pred = X @ beta
    resid = y - pred
    ss_tot = float(((y - y.mean()) ** 2).sum())
    ss_res = float((resid ** 2).sum())
    n, p = X.shape[0], X.shape[1] - 1
    r2 = 1 - ss_res / ss_tot
    adj = 1 - (1 - r2) * (n - 1) / (n - p - 1) if n - p - 1 > 0 else float("nan")
    se = math.sqrt(ss_res / (n - p - 1)) if n - p - 1 > 0 else float("nan")
    return beta, pred, r2, adj, se, ss_tot, ss_res


def loo_q2(X, y):
    """Leave-one-out cross-validation: PRESS and q^2. Each compound is predicted by a model
    refitted without it, which is the only honest way to use a ten-compound set."""
    press, preds = 0.0, []
    for i in range(len(y)):
        m = np.ones(len(y), bool)
        m[i] = False
        b, *_ = np.linalg.lstsq(X[m], y[m], rcond=None)
        p = float(X[i] @ b)
        preds.append(p)
        press += (y[i] - p) ** 2
    ss_tot = float(((y - y.mean()) ** 2).sum())
    return press, 1 - press / ss_tot, np.array(preds)


MODEL_CHOICES = ["pi + sigma + MR10", "pi + sigma", "pi + MR10", "sigma + MR10", "pi"]


def parse_model(spec):
    """Accept 'pi + sigma + MR10' (the Colab dropdown passes a string) or a list of column names."""
    cols = [c.strip() for c in spec.split("+")] if isinstance(spec, str) else [str(c).strip() for c in spec]
    cols = [c for c in cols if c]
    allowed = {"pi", "sigma", "MR10", "pi2", "pi_sigma", "MR2"}
    bad = [c for c in cols if c not in allowed]
    if bad or not cols:
        raise ValueError("unknown descriptor(s) %s; choose from %s" % (bad or spec, sorted(allowed)))
    return cols


def design(cols, table):
    """Build the design matrix (with intercept) for the named descriptor columns
    (a list of names, or a model string such as 'pi + sigma + MR10')."""
    cols = parse_model(cols)
    return np.column_stack([np.asarray(table[c], float) for c in cols] + [np.ones(len(table))])


def equation_text(cols, beta, r2, q2, n):
    terms = " ".join("%+.3f %s" % (b, c) for b, c in zip(beta[:-1], cols))
    return "pIC50 = %s %+.3f   (n = %d, r2 = %.3f, q2 = %.3f)" % (terms, beta[-1], n, r2, q2)

import matplotlib.pyplot as plt
plt.rcParams.update({'figure.dpi': 110})
_start_report('PHS 312', 'L29-L30 Lab', 'Lab: Building and Validating a Hansch QSAR with numpy')
print('Set-up complete: %d calculation cards loaded; figures, tables and calculations will be collected for your lab report.' % len(CARDS))

## Part 1 - Protocol (best practice, with the reason for each step)

1. **Use a congeneric series.** Every compound shares the 4-anilino-6,7-dimethoxyquinazoline scaffold and differs
   only at the aniline 3-position, so a substituent-constant model is meaningful. Mixing scaffolds needs different
   descriptors and far more compounds.
2. **Convert potency to a logarithmic, molar scale** (pIC50 = −log10 IC50 in mol/L). Free energy is proportional to
   log activity, so only the log scale can be linear in the descriptors.
3. **State the source of every number**: the ChEMBL release and assay filter for the activities, the published tables
   for π, σ and MR. Mixed assays are the biggest weakness of a data set like this one, and must be disclosed.
4. **Keep the number of descriptors small.** A common rule is at least five compounds per descriptor; with ten
   compounds that allows two, at most three, terms. More terms always raise r².
5. **Fit by least squares and inspect the residuals**, not just r². One badly fitted compound often reveals a
   different binding mode or a data error.
6. **Validate by leave-one-out cross-validation** (q²) and, where possible, an external test compound. r² measures
   fit; q² measures prediction. A model with r² = 0.9 and q² = 0.1 predicts nothing.
7. **Check the applicability domain**: a prediction is only meaningful if the new substituent's descriptors lie
   inside the range used for fitting.
8. **Report** the full equation with n, r², q², standard error and the data, following the checklist at the end.

Expected run time: a few seconds (the regressions are 10 × 4 matrices).

## Part 2 - Assemble and check the data set

*Objectives addressed: L29_L30.1*

In [ ]:
#@title Step 1 - Assemble the data set
lab_step('Step 1 - Data set')
import numpy as np, pandas as pd
rows = []
for cid, sub, pi, sg, mr, ic50, smi in SERIES:
    rows.append(dict(ChEMBL=cid, substituent=sub, pi=pi, sigma=sg, MR=mr, MR10=mr / 10.0,
                     IC50_nM=ic50, pIC50=round(9 - np.log10(ic50), 3), SMILES=smi))
data = pd.DataFrame(rows)
data["pi2"] = data.pi ** 2
report_table(data[["ChEMBL", "substituent", "pi", "sigma", "MR", "IC50_nM", "pIC50"]],
             "Step 1 - The data set (ten congeners)", note="Activities: ChEMBL 27, EGFR (CHEMBL203), IC50, "
             "relation '=', binding assays. Substituent constants: Hansch, Leo & Taft 1991.")
y = data.pIC50.values
print("pIC50 range %.2f to %.2f (%.1f log units); mean %.2f, SD %.2f"
      % (y.min(), y.max(), y.max() - y.min(), y.mean(), y.std(ddof=1)))
software_versions()
record_params("Data set", endpoint="EGFR inhibition, IC50 (nM) -> pIC50", activity_source="ChEMBL 27 via TeachOpenCADD T001 (binding assays, mixed publications)", descriptor_source="pi, sigma_m, MR: Hansch, Leo & Taft 1991", n_compounds=len(data))

In [ ]:
#@title Checkpoint 1 { display-mode: "form" }
checkpoint("Compounds in the series", len(data), 10, 10)
checkpoint("Activity range", float(y.max() - y.min()), 2.0, None, "log units",
           why="A QSAR needs at least ~2 log units of variation to model; less and you are fitting noise.")

### Step 2 - Draw the series and confirm it is congeneric
RDKit is used here only to check that every SMILES parses and that all ten molecules really share one scaffold
(the maximum common substructure should be the whole quinazoline core).

In [ ]:
#@title Step 2 - Check the series is congeneric
lab_step('Step 2 - Series check')
from rdkit import Chem
from rdkit.Chem import rdFMCS
mols = [Chem.MolFromSmiles(s) for s in data.SMILES]
mcs = rdFMCS.FindMCS(mols, timeout=30, completeRingsOnly=True)
core = Chem.MolFromSmarts(mcs.smartsString)
print("Common scaffold: %d atoms, %d bonds" % (mcs.numAtoms, mcs.numBonds))
draw_mols(mols, list(data.substituent), title="Step 2 - The congeneric series (substituent at aniline C3)", per_row=5)
draw_mols([core], ["maximum common substructure"], title="Step 2 - Shared scaffold")

In [ ]:
#@title Checkpoint 2 { display-mode: "form" }
checkpoint("Atoms in the common scaffold", mcs.numAtoms, 18, 30,
           why="If the MCS is small, one SMILES is wrong or the series is not congeneric.")

## Part 3 - Descriptors one at a time and collinearity

*Objectives addressed: L29_L30.1, L29_L30.2*

### Step 3 - One descriptor at a time
Before fitting anything multi-term, look at each descriptor alone. This is where the classical Hansch questions are
asked: is activity driven by hydrophobicity (π), by electronics (σ), by bulk (MR), or by none of them on its own?

In [ ]:
#@title Step 3 - Single-descriptor models
lab_step('Step 3 - Single descriptors')
single = []
for c, label in [("pi", "pi (hydrophobic)"), ("sigma", "sigma_m (electronic)"), ("MR10", "MR/10 (bulk)"),
                 ("pi2", "pi^2 (parabolic term)")]:
    X = design([c], data)
    b, pred, r2, adj, se, sst, ssr = ols(X, y)
    press, q2, _ = loo_q2(X, y)
    single.append([label, round(b[0], 3), round(b[1], 3), round(r2, 3), round(q2, 3), round(se, 3)])
report_table(single, "Step 3 - One-descriptor models",
             columns=["Descriptor", "Coefficient", "Intercept", "r2", "q2 (LOO)", "s"])
fig, axs = plt.subplots(1, 3, figsize=(11, 3.4))
for ax, c, lab in zip(axs, ["pi", "sigma", "MR10"], ["pi", "sigma_m", "MR/10"]):
    ax.scatter(data[c], y, color="#1d6fb8")
    for _, r in data.iterrows():
        ax.annotate(r.substituent, (r[c], r.pIC50), textcoords="offset points", xytext=(3, 3), fontsize=6)
    ax.set_xlabel(lab); ax.set_ylabel("pIC50")
fig.suptitle("Step 3 - pIC50 against each substituent constant")
plt.tight_layout()
plt.show()

In [ ]:
#@title Checkpoint 3 { display-mode: "form" }
best_single = max(single, key=lambda r: r[3])
checkpoint("Best single-descriptor r2", best_single[3], 0.0, 0.5,
           why="No single constant explains this series - that is the point of a multi-term Hansch equation.")

### Step 4 - Correlation between descriptors
If two descriptors are strongly correlated with each other, their individual coefficients become unstable and cannot
be interpreted (collinearity). Check this **before** fitting, not after.

In [ ]:
#@title Step 4 - Collinearity check
lab_step('Step 4 - Collinearity')
cols = ["pi", "sigma", "MR10"]
C = np.corrcoef(np.column_stack([data[c] for c in cols]).T)
fig, ax = plt.subplots(figsize=(4.4, 3.8))
im = ax.imshow(C, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(3), cols); ax.set_yticks(range(3), cols)
for i in range(3):
    for j in range(3):
        ax.text(j, i, "%.2f" % C[i, j], ha="center", va="center", fontsize=9)
plt.colorbar(im, ax=ax)
ax.set_title("Step 4 - Descriptor cross-correlation", fontsize=10)
plt.show()
offdiag = max(abs(C[0, 1]), abs(C[0, 2]), abs(C[1, 2]))
print("Largest absolute cross-correlation: %.2f" % offdiag)

In [ ]:
#@title Checkpoint 4 { display-mode: "form" }
checkpoint("Largest descriptor cross-correlation", float(offdiag), 0, 0.8,
           why="Above about 0.8 the coefficients of the two descriptors cannot be interpreted separately.")

## Part 4 - Fit and cross-validate the Hansch equation

*Objectives addressed: L29_L30.2, L29_L30.3*

### Step 5 - Fit the Hansch equation
$$pIC_{50} = a\,\pi + b\,\sigma_m + c\,\frac{MR}{10} + k$$
`numpy.linalg.lstsq` solves the least-squares problem directly. Read the coefficients physically: a positive
π coefficient means a more hydrophobic substituent helps, a positive σ coefficient means electron-withdrawing
substituents help, and a negative MR coefficient means bulk is penalised.

In [ ]:
#@title Step 5 - Multi-term least-squares fit
lab_step('Step 5 - Hansch fit')
MODEL_SPEC = "pi + sigma + MR10"    #@param ["pi + sigma + MR10", "pi + sigma", "pi + MR10", "sigma + MR10", "pi"]
MODEL = parse_model(MODEL_SPEC)          # -> list of column names, e.g. ["pi", "sigma", "MR10"]
X = design(MODEL, data)
beta, pred, r2, adj, se, sst, ssr = ols(X, y)
press, q2, loo_pred = loo_q2(X, y)
print(equation_text(MODEL, beta, r2, q2, len(y)))
print("adjusted r2 = %.3f   s = %.3f   PRESS = %.3f   SS_total = %.3f" % (adj, se, press, sst))
report_table([[c, round(b, 3)] for c, b in zip(MODEL + ["intercept"], beta)] +
             [["r2", round(r2, 3)], ["adjusted r2", round(adj, 3)], ["q2 (LOO)", round(q2, 3)],
              ["s (standard error)", round(se, 3)], ["PRESS", round(press, 3)], ["SS total", round(sst, 3)]],
             "Step 5 - Fitted Hansch model", columns=["Term / statistic", "Value"])
fig, axs = plt.subplots(1, 2, figsize=(10, 4.2))
axs[0].scatter(pred, y, color="#1d6fb8", label="fitted")
axs[0].scatter(loo_pred, y, facecolors="none", edgecolors="#b91c1c", label="leave-one-out")
lim = [min(y.min(), pred.min()) - 0.5, max(y.max(), pred.max()) + 0.5]
axs[0].plot(lim, lim, "k--", lw=1)
axs[0].set_xlabel("predicted pIC50"); axs[0].set_ylabel("observed pIC50"); axs[0].legend(fontsize=8)
axs[0].set_title("Observed vs predicted (r2 = %.2f, q2 = %.2f)" % (r2, q2), fontsize=10)
axs[1].bar(range(len(y)), y - pred, color="#c2410c")
axs[1].set_xticks(range(len(y)), list(data.substituent), rotation=60, fontsize=7)
axs[1].axhline(0, color="k", lw=1)
axs[1].set_ylabel("residual (observed - fitted)")
axs[1].set_title("Residuals by compound", fontsize=10)
plt.tight_layout()
plt.show()
worst = data.substituent[int(np.argmax(np.abs(y - pred)))]
print("Largest residual: %s (%.2f log units) - look at this compound before trusting the model." %
      (worst, float(np.max(np.abs(y - pred)))))
record_params("Model", model=equation_text(MODEL, beta, r2, q2, len(y)), fit_method="numpy.linalg.lstsq (ordinary least squares)", r2=round(r2, 3), q2_LOO=round(q2, 3), s=round(se, 3))

In [ ]:
#@title Checkpoint 5 { display-mode: "form" }
checkpoint("r2 of the three-term model", float(r2), 0.5, 1.0)
checkpoint("q2 (leave-one-out)", float(q2), 0.0, 1.0,
           why="q2 below 0.5 means the model is not predictive; that is the honest result for ten mixed-assay compounds.")

**1. Predicting potency with a fitted Hansch equation**

$$pIC_{50} = {\color{#7c3aed}{a}}\,{\color{#1d6fb8}{\pi}} + {\color{#b91c1c}{b}}\,{\color{#c2410c}{\sigma_m}} + {\color{#0e7490}{c}}\,\frac{{\color{#15803d}{MR}}}{10} + {\color{#a16207}{k}}$$

| Symbol | Meaning | Role in the calculation | Units | Source / value |
|---|---|---|---|---|
| ${\color{#1d6fb8}{\pi}}$ | hydrophobic constant of the new 3-substituent (C≡CH) | input | — | Hansch, Leo & Taft, Chem. Rev. 1991, 91, 165 |
| ${\color{#c2410c}{\sigma_m}}$ | Hammett meta constant of C≡CH | input | — | Hansch, Leo & Taft 1991 |
| ${\color{#15803d}{MR}}$ | molar refractivity of C≡CH | input | cm^3/mol | Hansch, Leo & Taft 1991 |
| ${\color{#7c3aed}{a}}$ | fitted coefficient of pi | input | — | your regression in Step 5 (lab fit) |
| ${\color{#b91c1c}{b}}$ | fitted coefficient of sigma | input | — | your regression in Step 5 (lab fit) |
| ${\color{#0e7490}{c}}$ | fitted coefficient of MR/10 | input | — | your regression in Step 5 (lab fit) |
| ${\color{#a16207}{k}}$ | fitted intercept | input | — | your regression in Step 5 (lab fit) |

*Source of the relationship:* Hansch-Fujita analysis (Hansch & Fujita, J. Am. Chem. Soc. 1964, 86, 1616)

**2. Practice problem (case)**

Erlotinib carries a 3-ethynylanilino group. Before ordering the 6,7-dimethoxy analogue, the project team uses your Hansch model ($\mathbf{{\color{#7c3aed}{2.433}}}$, $\mathbf{{\color{#b91c1c}{2.436}}}$, $\mathbf{{\color{#0e7490}{-2.07}}}$, $\mathbf{{\color{#a16207}{7.84}}}$) and the substituent constants $\mathbf{{\color{#1d6fb8}{0.4}}}$, $\mathbf{{\color{#c2410c}{0.21}}}$ and $\mathbf{{\color{#15803d}{9.55\,\mathrm{cm^{3}/mol}}}}$. What pIC50 does the model predict?

<details><summary><b>▸ 3. Step-by-step solution (try it first)</b></summary>

| Step | Work |
|---|---|
| 1. General equation | $pIC_{50} = {\color{#7c3aed}{a}}\,{\color{#1d6fb8}{\pi}} + {\color{#b91c1c}{b}}\,{\color{#c2410c}{\sigma_m}} + {\color{#0e7490}{c}}\,\frac{{\color{#15803d}{MR}}}{10} + {\color{#a16207}{k}}$ |
| 2. Values from the case | ${\color{#1d6fb8}{\pi}} = 0.4$; ${\color{#c2410c}{\sigma_m}} = 0.21$; ${\color{#15803d}{MR}} = 9.55\,\mathrm{cm^{3}/mol}$; ${\color{#7c3aed}{a}} = 2.433$; ${\color{#b91c1c}{b}} = 2.436$; ${\color{#0e7490}{c}} = -2.07$; ${\color{#a16207}{k}} = 7.84$ |
| 3. Unit check | Prefixes written out: $9.55\,\mathrm{cm^{3}/mol} = 9.55 \times 10^{-6}\,\mathrm{m^3/mol} = 9.55 \times 10^{-6}\,\mathrm{m^{3}/mol}$. pi, sigma and MR/10 are dimensionless substituent constants (MR in cm^3/mol is scaled by 0.1 as in Hansch's tables), so every term is in pIC50 units; IC50 is converted from mol/L to nM (1 nM = 10^-9 mol/L). |
| 4. Hydrophobic term | ${\color{#7c3aed}{a}} \times {\color{#1d6fb8}{\pi}} = 2.433 \times 0.4 = 0.9732$ |
| 5. Electronic term | ${\color{#b91c1c}{b}} \times {\color{#c2410c}{\sigma_m}} = 2.436 \times 0.21 = 0.5116$ |
| 6. Steric term | ${\color{#0e7490}{c}} \times \frac{{\color{#15803d}{MR}}}{10} = -2.07 \times 0.955 = -1.977$ |
| 7. Sum with intercept | $pIC_{50} = 0.9732 + 0.5116 + (-1.977) + 7.84 = 7.348$ |
| 8. Back to IC50 | $IC_{50} = 10^{-7.348}\,\mathrm{mol/L} = 44.88\,\mathrm{nM}$ |
| 9. Answer (predicted pIC50) | $\boxed{pIC_{50,pred} = 7.348}$ |
| 10. What it means | The model predicts pIC50 = 7.348 (IC50 about 44.88 nM) for the 3-ethynyl analogue. Because q^2 for this model is only about 0.24 and the training set mixes assays, the prediction is a hypothesis to test, not a reliable estimate. |

**Common conversions for this calculation:** pIC50 = -log10(IC50 in mol/L); pIC50 9 = 1 nM; pIC50 6 = 1 µM
</details>

In [ ]:
#@title Solve it yourself: Predicting potency with a fitted Hansch equation { display-mode: "form" }
_cell('Calculation: Predicting potency with a fitted Hansch equation')
card('l29_hansch_prediction')

**1. Leave-one-out cross-validated q^2**

$$q^2 = 1 - \frac{{\color{#1d6fb8}{PRESS}}}{{\color{#c2410c}{SS}}} = 1 - \frac{\sum (y_i - \hat{y}_{i,(-i)})^2}{\sum (y_i - \bar{y})^2}$$

| Symbol | Meaning | Role in the calculation | Units | Source / value |
|---|---|---|---|---|
| ${\color{#1d6fb8}{PRESS}}$ | predicted residual sum of squares (each compound predicted by a model fitted without it) | input | — | LOO loop in Step 6 (pi + sigma + MR model) |
| ${\color{#c2410c}{SS}}$ | total sum of squares of pIC50 about its mean | input | — | Step 6 |

*Source of the relationship:* Cramer, Bunce & Patterson, Quant. Struct.-Act. Relat. 1988, 7, 18

**2. Practice problem (case)**

After fitting the three-descriptor Hansch model, you run leave-one-out cross-validation and obtain $\mathbf{{\color{#1d6fb8}{11.93}}}$. The total sum of squares of the ten pIC50 values is $\mathbf{{\color{#c2410c}{15.7}}}$. What is q^2?

<details><summary><b>▸ 3. Step-by-step solution (try it first)</b></summary>

| Step | Work |
|---|---|
| 1. General equation | $q^2 = 1 - \frac{{\color{#1d6fb8}{PRESS}}}{{\color{#c2410c}{SS}}} = 1 - \frac{\sum (y_i - \hat{y}_{i,(-i)})^2}{\sum (y_i - \bar{y})^2}$ |
| 2. Values from the case | ${\color{#1d6fb8}{PRESS}} = 11.93$; ${\color{#c2410c}{SS}} = 15.7$ |
| 3. Unit check | No unit prefixes to expand. PRESS and SS are both sums of squared pIC50 differences, so their ratio is dimensionless. |
| 4. Ratio | $\frac{{\color{#1d6fb8}{PRESS}}}{{\color{#c2410c}{SS}}} = \frac{11.93}{15.7} = 0.7599$ |
| 5. Cross-validated q^2 | $q^2 = 1 - 0.7599 = 0.2401$ |
| 6. Answer (q^2) | $\boxed{q^2 = 0.2401}$ |
| 7. What it means | q^2 = 0.2401. A model is normally considered predictive only when q^2 > 0.5; a value this low means leave-one-out predictions are barely better than predicting the mean pIC50 for every compound. |

**Common conversions for this calculation:** r^2 uses the residual sum of squares of the fitted model; q^2 uses PRESS
</details>

In [ ]:
#@title Solve it yourself: Leave-one-out cross-validated q^2 { display-mode: "form" }
_cell('Calculation: Leave-one-out cross-validated q^2')
card('l30_q2_loo')

## Part 5 - Over-fitting and the applicability domain

*Objectives addressed: L29_L30.4*

### Step 6 - Over-fitting: watch r² rise while q² falls
Now add descriptors one at a time, up to six terms for ten compounds. r² can only increase when a term is added -
that is arithmetic, not chemistry. q² tells you when the extra terms are fitting noise.

In [ ]:
#@title Step 6 - Demonstrate over-fitting
lab_step('Step 6 - Over-fitting')
data["pi_sigma"] = data.pi * data.sigma
data["MR2"] = data.MR10 ** 2
ladder = [["pi"], ["pi", "sigma"], ["pi", "sigma", "MR10"], ["pi", "sigma", "MR10", "pi2"],
          ["pi", "sigma", "MR10", "pi2", "pi_sigma"], ["pi", "sigma", "MR10", "pi2", "pi_sigma", "MR2"]]
lr = []
for cols_ in ladder:
    Xl = design(cols_, data)
    b, p_, r2_, adj_, se_, sst_, ssr_ = ols(Xl, y)
    pr_, q2_, _ = loo_q2(Xl, y)
    lr.append([len(cols_), " + ".join(cols_), round(r2_, 3), round(adj_, 3), round(q2_, 3), round(se_, 3)])
report_table(lr, "Step 6 - Adding descriptors to ten compounds",
             columns=["Terms", "Model", "r2", "adjusted r2", "q2 (LOO)", "s"])
fig, ax = plt.subplots(figsize=(6.6, 4))
ax.plot([r[0] for r in lr], [r[2] for r in lr], "-o", label="r2 (fit)", color="#1d6fb8")
q2s = [r[4] for r in lr]
ax.plot([r[0] for r in lr], [max(q, -2.0) for q in q2s], "-o", label="q2 (prediction)", color="#b91c1c")
for r, q in zip(lr, q2s):
    if q < -2.0:
        ax.annotate("q2 = %.1f\n(off scale)" % q, (r[0], -2.0), textcoords="offset points",
                    xytext=(-12, 10), fontsize=7, color="#b91c1c")
ax.axhline(0.5, ls="--", color="#6b7280", lw=1)
ax.set_ylim(-2.2, 1.1)
ax.set_xlabel("number of descriptors (n = 10 compounds)"); ax.set_ylabel("statistic (q2 clipped at -2)")
ax.set_title("Step 6 - Over-fitting: r2 always rises, q2 does not", fontsize=10)
ax.legend()
plt.show()

In [ ]:
#@title Checkpoint 6 { display-mode: "form" }
r2_last, q2_last = lr[-1][2], lr[-1][4]
checkpoint("r2 of the six-term model", r2_last, r2, 1.0, why="r2 must be at least as high as the three-term model.")
checkpoint("q2 of the six-term model is worse than the three-term model", q2_last < q2, expect=q2_last < q2,
           why="If not, note it in your report: with ten compounds either outcome is within the noise.")

### Step 7 - Predict an external compound, and check the applicability domain
The 3-ethynyl substituent (as in erlotinib) was held out. Before using the prediction, confirm its π, σ and MR
lie inside the ranges spanned by the training set; a prediction outside the domain is extrapolation.

In [ ]:
#@title Step 7 - External prediction and applicability domain
lab_step('Step 7 - Prediction')
new = TEST_SUBSTITUENT
new_desc = {"pi": new["pi"], "sigma": new["sigma"], "MR10": new["MR"] / 10.0}
xnew = np.array([new_desc[c] for c in MODEL] + [1.0])   # same columns, same order as the fitted model
pred_new = float(xnew @ beta)
inside = {k: (float(data[c].min()) <= v <= float(data[c].max()))
          for k, c, v in [("pi", "pi", new["pi"]), ("sigma", "sigma", new["sigma"]), ("MR/10", "MR10", new["MR"] / 10.0)]}
report_table([[k, "yes" if v else "NO - extrapolation"] for k, v in inside.items()],
             "Step 7 - Applicability domain for %s" % new["name"], columns=["Descriptor", "Inside training range?"])
print("Predicted pIC50 for %s: %.2f  (IC50 about %.1f nM)" % (new["name"], pred_new, 10 ** (9 - pred_new)))
fig, ax = plt.subplots(figsize=(6.6, 4))
ax.scatter(data.pi, y, color="#1d6fb8", label="training set")
ax.scatter([new["pi"]], [pred_new], marker="*", s=220, color="#15803d", label="prediction (%s)" % new["name"])
for _, r in data.iterrows():
    ax.annotate(r.substituent, (r.pi, r.pIC50), textcoords="offset points", xytext=(3, 3), fontsize=6)
ax.set_xlabel("pi of the 3-substituent"); ax.set_ylabel("pIC50")
ax.set_title("Step 7 - Where the prediction sits relative to the training set", fontsize=10)
ax.legend(fontsize=8)
plt.show()
record_params("Applicability domain", applicability_domain="training ranges: pi %.2f to %.2f, sigma %.2f to %.2f, MR/10 %.2f to %.2f" % (data.pi.min(), data.pi.max(), data.sigma.min(), data.sigma.max(), data.MR10.min(), data.MR10.max()))

In [ ]:
#@title Checkpoint 7 { display-mode: "form" }
checkpoint("Prediction inside the applicability domain", all(inside.values()), expect=all(inside.values()),
           why="If a descriptor is outside the training range, report the prediction as an extrapolation.")
checkpoint("Predicted pIC50", pred_new, 4, 12)

## Part 6 - Whole-molecule descriptors for comparison

*Objectives addressed: L29_L30.3, L29_L30.4*

### Step 8 - The modern alternative: whole-molecule RDKit descriptors
Hansch descriptors describe one substituent. Modern QSAR usually uses whole-molecule descriptors. With ten
compounds this is a cautionary exercise: the computed descriptors are highly correlated with each other and with
molecular size, so the fit looks good and the cross-validation does not.

In [ ]:
#@title Step 8 - RDKit descriptor model for comparison
lab_step('Step 8 - RDKit descriptors')
from rdkit.Chem import Crippen, rdMolDescriptors, Descriptors
desc = []
for m in mols:
    desc.append(dict(cLogP=Crippen.MolLogP(m), TPSA=rdMolDescriptors.CalcTPSA(m), MW=Descriptors.MolWt(m),
                     MolMR=Crippen.MolMR(m), HA=float(m.GetNumHeavyAtoms())))
D = pd.DataFrame(desc)
report_table(pd.concat([data[["substituent", "pIC50"]], D.round(2)], axis=1),
             "Step 8 - Whole-molecule RDKit descriptors")
for cols_ in (["cLogP"], ["cLogP", "TPSA"], ["cLogP", "TPSA", "MolMR"]):
    Xd = np.column_stack([D[c].values for c in cols_] + [np.ones(len(D))])
    b, p_, r2_, adj_, se_, sst_, ssr_ = ols(Xd, y)
    pr_, q2_, _ = loo_q2(Xd, y)
    print("%-28s r2 = %.3f  adj r2 = %.3f  q2 = %.3f" % (" + ".join(cols_), r2_, adj_, q2_))
print("\nCompare with the substituent-constant model: r2 = %.3f, q2 = %.3f" % (r2, q2))

## Part 7 - Reporting check

*Objectives addressed: L29_L30.5*

In [ ]:
#@title Assemble the lab report
report_preview()

In [ ]:
#@title Checkpoint 8 - Does the record meet the OECD reporting principles? { display-mode: "form" }
reporting_check(["endpoint", "activity_source", "descriptor_source", "n_compounds", "model", "fit_method",
                 "r2", "q2_LOO", "s", "applicability_domain"])

## Part 8 - Reporting standards for a QSAR

Follow the five OECD principles for the validation of (Q)SAR models, which reviewers and regulators expect:
1. **A defined endpoint**: EGFR inhibition, IC50 in nM converted to pIC50; assay type, species and the exact ChEMBL
   release; a warning when records come from different publications and protocols (as here).
2. **An unambiguous algorithm**: the full equation with every coefficient, the intercept, and the software and
   version used to fit it (numpy least squares; printed in your report).
3. **A defined applicability domain**: the range of each descriptor in the training set, and a statement for every
   prediction about whether it is inside that range.
4. **Appropriate measures of goodness-of-fit, robustness and predictivity**: n, r², adjusted r², standard error s,
   leave-one-out q² (and PRESS), ideally y-randomisation and an external test set. Never report r² alone.
5. **A mechanistic interpretation, where possible**: what each descriptor means physically, and whether the sign of
   its coefficient makes chemical sense for this binding site.

Also state: the compounds-per-descriptor ratio (here 10/3), the descriptor source (published tables vs computed),
how collinearity was checked, which compounds had the largest residuals and why they might, and that correlation
over ten congeners cannot establish a mechanism.

## Part 9 - Lab report (automatic)

Every figure you displayed, every table, every checkpoint and every calculation you solved with **Solve step by step**
was collected as you worked. Add your name and a short discussion (what your results show, what you would check
next, and the limitations listed under *Reporting standards*), then build the report: a **Word (.docx)** file and an
**HTML** copy. A file is written only when you press the button.

In [ ]:
#@title Build and download my lab report { display-mode: "form" }
lab_summary()
report_panel()

## Part 10 - Data and sources

* **Activities**: ChEMBL 27 (doi:10.6019/CHEMBL.database.27; Mendez et al., *Nucleic Acids Res.* 2019, 47, D930;
  CC BY-SA 3.0), human EGFR (CHEMBL203), IC50, relation '=', binding assays, as frozen in TeachOpenCADD talktorial
  T001 (`EGFR_compounds.csv`, commit c140d6c; Sydow et al., *J. Chem. Inf. Model.* 2019, 59, 1; CC BY 4.0).
* **Original series and SAR**: Bridges et al., *J. Med. Chem.* 1996, 39, 267 (4-anilinoquinazolines as EGFR inhibitors).
* **Substituent constants** π, σ_m, MR: Hansch, Leo & Taft, *Chem. Rev.* 1991, 91, 165.
* **Method**: Hansch & Fujita, *J. Am. Chem. Soc.* 1964, 86, 1616; q² and PRESS: Cramer, Bunce & Patterson,
  *Quant. Struct.-Act. Relat.* 1988, 7, 18; Golbraikh & Tropsha, *J. Mol. Graph. Model.* 2002, 20, 269 (validation);
  OECD, *Guidance Document on the Validation of (Q)SAR Models*, 2007.
* **Software**: numpy (BSD-3), pandas (BSD-3), RDKit (BSD-3) `rdFMCS`, `Crippen`, `rdMolDescriptors`.
* Protocol layout modelled on the UCL Open Docking Lab Handbook (doi:10.1021/acs.jchemed.1c00289; structure only).
* Full provenance: `labs/SOURCES.md`.